# Interventions on the dissolution of the register state (FLUX.1-dev)

This code runs paired interventions on the FLUX.1-dev image stream to test
what ends the high-norm register state at the end of the band (blocks 18
to 39). The register token is the image token aligned with the register
direction v\*, which sits mostly on channel 154. Channel 1446 is the
catching-up channel. Every condition is compared with the baseline run of
the same scenario.

1. Channel operators. From the catch-up onset to the last block we zero the
   catching-up channel (`zero_ch2`) or stop it from growing (`nogrow_ch2`).
   `growth_x2` does the opposite and scales the block's write to that
   channel by 2. `zero_ctrl` zeroes a matched control channel.
2. Token operators. `dir_refresh` reprojects the register token onto v\* at
   every block of the late range and keeps its norm. The controls are
   `norm_refresh` (norm floor, direction kept) and `sham` (the register is
   written back unchanged).
3. If neither family keeps the register alive longer, the takeover table,
   the x_perp trajectory and the attention probes show where the state went.

With `baseline` these are the eight default conditions. More are defined in
I4 and can be switched on in I1a or I1b, for example `nogrow_others`,
`freeze_ch2`, other growth factors (`growth_x<g>`, `_early` to start at the
band onset), `dir_refresh_random`, and `zero_ch3` and `zero_both` for
channel 85.

At every block and step we record each token's norm (the persistence
endpoint), the register's split x = alpha v\* + x_perp and its values on the
tracked channels. At the probe blocks we also record, per head, the register
key's rank among image keys, its cosine with the mean image query and the
share of attention it receives.

v\* is fitted from the full tensors of the channel recurrence run at
(l\*, t\*). It is frozen before any intervention run, together with the
channels, the onset block, the refresh range and the probe blocks.

## How to run

1. You need gated access to FLUX.1-dev, a Hugging Face read token in the
   Colab secret `HF_TOKEN`, and an A100 GPU runtime.
2. This experiment reuses the outputs of the earlier channel recurrence run.
   The Drive folder `MyDrive/flux_ma_channels` must already contain the
   prompt list `meta/prompts.json`, the pilot decision
   `meta/pilot_decision.json`, the prompt-embedding cache, the five pilot npz
   files, and the full tensors of the main run (`main/*__Xfull_*.npy`, used
   to fit v\*). The code stops with a message if any of these is missing.
3. Run the cells from top to bottom. Each of the following has to succeed
   before the next one runs. They are the pipeline smoke test (S3c), the
   intervention decision (I2), the operator unit tests (I3, on CPU), the v\*
   fit (I2b), the intervention smoke test (I5), the capture (I6) and the
   audit (I7). The analysis is in I8 to I10.
4. `scenario_scope` in I1b defaults to `"full"`, which is 100 scenarios (20
   prompts times 5 seeds, the same grid as the recurrence run). Set it to
   `"pilot"` for a five-scenario dry run. The capture can be resumed. After a
   disconnect, run from the top through I6 and it picks up after the last
   finished run.
5. You can add a condition later and only the new runs are computed, since
   the condition list is not part of the run hash or the decision. Changing
   a hashed config field, the decision or v\* makes the finished runs invalid
   (`_HASH_EXCLUDE` in I1b lists the fields that are not hashed).

On an A100 40 GB, 8 conditions times 100 scenarios is 800 generations of 28
steps. At about 20 to 25 seconds each that is roughly 5 hours. With the
register-vector capture on it needs about 25 GiB on Drive, or about 10 GiB
with `store_reg_vec=False`.

## S0. Environment

Installs the libraries with version bounds (Colab's preinstalled torch is left
alone), mounts Drive, checks free space and picks the precision mode from the
GPU. On a GPU with at least 35 GiB (A100) the model runs in bf16. Smaller GPUs
fall back to quantized weights, which is recorded in every output file as a
possible confound. The cell also checks access to the gated FLUX.1-dev
weights before any compute is spent.

In [ ]:
# =====================================================================
# S0. Environment setup
# =====================================================================
import importlib, importlib.util, subprocess, sys, os

def _pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[setup][warn] pip install {' '.join(pkgs)} failed:\n{r.stderr[-800:]}")

# Leave torch alone. Colab's preinstalled build matches its GPUs.
_pip("diffusers>=0.32.0,<0.40", "transformers>=4.44.0", "accelerate>=0.33.0",
     "safetensors", "sentencepiece", "protobuf", "huggingface_hub>=0.24.0",
     "pyarrow", "pandas")
# Quantization back ends, only needed for the fallback on GPUs below 35 GiB.
_pip("torchao", "bitsandbytes")

import gc, json, math, time, glob, re, shutil, hashlib, getpass, platform, datetime, csv
import numpy as np
import torch

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# ---------------------------------------------------------------- Drive
BASE = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = "/content/drive/MyDrive/flux_ma_channels"
    except Exception as e:
        print(f"[setup][warn] Drive mount failed ({e}); falling back to local disk "
              f"-- outputs will NOT persist across sessions!")
if BASE is None:
    BASE = os.path.abspath("./flux_ma_channels")

PATHS = {
    "base":    BASE,
    "meta":    os.path.join(BASE, "meta"),
    "embeds":  os.path.join(BASE, "prompt_embeds"),
    "pilot":   os.path.join(BASE, "pilot"),
    "main":    os.path.join(BASE, "main"),
    "smoke":   os.path.join(BASE, "smoke"),
    "figures": os.path.join(BASE, "figures"),
    "analysis": os.path.join(BASE, "analysis"),
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)
RUN_LOG_PATH   = os.path.join(PATHS["meta"], "run_log.csv")
ERROR_LOG_PATH = os.path.join(PATHS["meta"], "errors.log")
print(f"[setup] output root: {BASE}")

# ------------------------------------------------------- Drive preflight
try:
    du = shutil.disk_usage(BASE)
    free_gib = du.free / 2**30
    print(f"[setup] free space at output root: {free_gib:.1f} GiB "
          f"(the intervention experiment needs ~15 GiB)")
    if free_gib < 16:
        print("[setup][WARN] <16 GiB free -- clear Drive space before the capture loop.")
except Exception as e:
    print(f"[setup][warn] could not check disk usage: {e}")

# ---------------------------------------------------------- GPU detect
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU visible. In Colab: Runtime -> Change runtime type -> "
        "GPU (A100 preferred, L4 fallback), then restart and re-run this cell.")
GPU_PROPS  = torch.cuda.get_device_properties(0)
GPU_NAME   = GPU_PROPS.name
TOTAL_GIB  = GPU_PROPS.total_memory / 2**30
CC         = (GPU_PROPS.major, GPU_PROPS.minor)
DTYPE      = torch.bfloat16 if CC >= (8, 0) else torch.float16
DEVICE     = "cuda"

# bf16 on GPUs with at least 35 GiB (A100), quantized weights otherwise.
if TOTAL_GIB >= 35.0:
    PRECISION_MODE = "bf16_full"
elif CC >= (8, 9):                     # L4 and other Ada GPUs, fp8 weight-only
    PRECISION_MODE = "quant_fp8"
else:                                  # T4, V100, A10 and so on, NF4 as a last resort
    PRECISION_MODE = "quant_nf4"

print(f"[setup] GPU: {GPU_NAME} | {TOTAL_GIB:.1f} GiB | compute capability {CC[0]}.{CC[1]}")
print(f"[setup] precision mode: {PRECISION_MODE} | activation dtype at hooks: fp32 "
      f"(model dtype {DTYPE})")
if PRECISION_MODE != "bf16_full":
    print("[setup][SCIENCE CAVEAT] Quantized fallback active: weight-only quantization "
          "perturbs the very activations we measure. Channel *identities* are expected "
          "to survive; magnitudes will shift. This flag is recorded in every output "
          "file and in the run manifest. If an A100 becomes available later, re-run "
          "the 5-scenario pilot in bf16 and compare top-10 sets (see manifest notes).")
    if PRECISION_MODE == "quant_nf4" and TOTAL_GIB < 20:
        print("[setup][WARN] Small GPU (likely T4): expect ~10 min/image at 1024^2. "
              "The main run will need multiple resumed sessions.")

# ------------------------------------------------------ Hugging Face auth
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        if HF_TOKEN:
            print("[setup] HF token loaded from Colab secret 'HF_TOKEN'.")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or None
if not HF_TOKEN:
    print("No HF token found (Colab secret 'HF_TOKEN' or env var). Paste one now.")
    print("The token account must have ACCEPTED the license of "
          "black-forest-labs/FLUX.1-dev (gated model).")
    tok = getpass.getpass("HF_TOKEN (input hidden): ").strip()
    HF_TOKEN = tok or None

MODEL_ID = "black-forest-labs/FLUX.1-dev"
from huggingface_hub import model_info
try:
    _ = model_info(MODEL_ID, token=HF_TOKEN)
    print(f"[setup] Hugging Face access to {MODEL_ID}: OK")
except Exception as e:
    raise RuntimeError(
        f"Cannot access {MODEL_ID}. FLUX.1-dev is a *gated* model:\n"
        f"  1) visit https://huggingface.co/{MODEL_ID} and accept the license,\n"
        f"  2) create a READ token at https://huggingface.co/settings/tokens,\n"
        f"  3) store it as a Colab secret named HF_TOKEN (key icon in left sidebar),\n"
        f"  4) re-run this cell.\nUnderlying error: {e}") from e

# ------------------------------------------------- library version record
def _ver(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "n/a"
LIB_VERSIONS = {k: _ver(k) for k in
                ("torch", "diffusers", "transformers", "accelerate",
                 "numpy", "huggingface_hub", "torchao", "bitsandbytes")}
LIB_VERSIONS["python"] = platform.python_version()
print("[setup] versions:", json.dumps(LIB_VERSIONS))


## S1. Experiment config

All experiment parameters live in one frozen dataclass, and its hash is
stored in every output file. The rules for picking the analysis block and
step are part of this config and are fixed before the pilot. They are the
search band (blocks 18 to 39), the tie priority (block 22, then 18), the
analysis step (half way through sampling, or three quarters as the
fallback), the statistic-agreement thresholds and the anchor channels 154 and
1446. The pilot only supplies the data these rules are applied to.

In [ ]:
# =====================================================================
# S1. Experiment config
#
# The config hash is stored in every output file.
# =====================================================================
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class ExperimentConfig:
    # -- model and sampling -----------------------------------------------
    model_id: str = "black-forest-labs/FLUX.1-dev"
    height: int = 1024                    # -> 64x64 = 4096 image tokens
    width: int = 1024
    num_steps: int = 28                   # common FLUX.1-dev setting
    guidance_scale: float = 3.5           # FLUX.1-dev embedded-guidance default
    max_seq_len: int = 512                # T5 text tokens, the image tokens come after these
    # -- scenario grid -----------------------------------------------------
    seeds: tuple = (0, 42, 100, 1234, 2024)
    n_prompts: int = 20
    prompt_source_seed: int = 1234        # RNG seed for DiffusionDB stratified sampling
    # -- architecture expectations (FLUX.1-dev) ----------------------------
    d_model: int = 3072
    n_blocks_expected: int = 57           # 19 dual-stream (MMDiT) + 38 single-stream
    n_dual_expected: int = 19
    # -- analysis block and step selection (rules fixed before the pilot) ---
    band: tuple = (18, 39)                # high-norm band, the search range for l*
    block_tie_priority: tuple = (22, 18)  # ties go to block 22 (Gan et al.), then band onset 18
    block_tie_window_log10: float = 0.1   # blocks within 0.1 log10 of best count as ties
    t_star_default: int = 14              # 0-indexed, half way through the 28 steps
    t_star_fallback: int = 21             # 0-indexed, three quarters of the way
    tstar_mag_frac_min: float = 0.6       # top-1 magnitude at t* >= 60% of the final step's, pilot median
    tstar_stab_jaccard_min: float = 0.6   # median top-10 Jaccard, t* vs final step
    # -- statistic-agreement gate (P4, checked on the pilot before Phase B) ---
    primary_stat_default: str = "abs_signed_mean"   # |mean| (Turri et al., Eq. 1)
    stat_gate_jaccard_min: float = 0.8    # keep |mean| if median J(top10 |mean|, top10 mean|x|) >= this
    sign_consistency_min: float = 0.9     # top-10 channels by mean|x| below this count as mixed-sign
    # -- external anchor check (P5) -----------------------------------------
    dg_anchor_channels: tuple = (154, 1446)  # MA channels reported for FLUX by Gan et al.
    allow_missing_dg_anchor: bool = False    # only set True after debugging by hand
    # -- logging ------------------------------------------------------------
    secondary_blocks: tuple = (6, 11, 18, 22, 30, 45)
    # 6 early control, 11 and 22 used in prior work, 18 band onset,
    # 30 mid-band, 45 post-band control
    k_list: tuple = (1, 5, 10)
    top_save: int = 20
    save_full_tensor: bool = True         # fp16 [4096,3072] at (l*, t*) per scenario (~24 MB)
    # -- pilot & smoke ---------------------------------------------------------
    pilot_pairs: tuple = (("p00", 42), ("p01", 0), ("p02", 1234),
                          ("p03", 100), ("p04", 2024))
    smoke_steps: int = 4
    recurring_set_threshold: float = 0.9  # G_k = {d : f_k(d) >= this}

CFG = ExperimentConfig()
N_IMG = (CFG.height // 8 // 2) * (CFG.width // 8 // 2)   # VAE /8, patchify 2x2
assert N_IMG == 4096, f"unexpected token count {N_IMG}; resolution changed?"

CONFIG_HASH = hashlib.sha256(
    json.dumps(asdict(CFG), sort_keys=True, default=str).encode()).hexdigest()[:12]

print(f"[config] {CFG.n_prompts} prompts x {len(CFG.seeds)} seeds = "
      f"{CFG.n_prompts * len(CFG.seeds)} scenarios | {CFG.num_steps} steps @ "
      f"{CFG.height}x{CFG.width} ({N_IMG} image tokens, D={CFG.d_model})")
print(f"[config] hash: {CONFIG_HASH}")
for k, v in asdict(CFG).items():
    print(f"    {k:26s} = {v}")


### LIB A. Core utilities

Small helpers used throughout. They cover Jaccard, top-k and rank functions,
the primary score and sign consistency, atomic writes (write a temp file,
then rename), scenario paths, the completeness check that lets a run resume,
and the run-log columns.


In [ ]:
# =====================================================================
# LIB A. Core utilities
#
# numpy and the standard library only.
# =====================================================================
import os, json, csv, hashlib, datetime
import numpy as np

STAT_NAMES = ("signed_mean", "mean_abs", "max_abs", "p99_abs", "frac_pos")

# ------------------------------------------------------------ set math
def jaccard(a, b):
    """Jaccard overlap of two iterables of channel ids."""
    A, B = set(int(x) for x in a), set(int(x) for x in b)
    if not A and not B:
        return 1.0
    return len(A & B) / len(A | B)

def topk_ids(score_vec, k):
    """Indices of the k largest scores, largest first. Ties go to the lower id."""
    s = np.asarray(score_vec, dtype=np.float64)
    # stable sort, so tied ids stay in ascending order
    order = np.argsort(-s, kind="stable")
    return order[:k].astype(np.int64)

def rank_of(score_vec, channel):
    """Competition rank (1 = largest) of `channel` within score_vec."""
    s = np.asarray(score_vec, dtype=np.float64)
    return 1 + int((s > s[int(channel)]).sum())

def primary_score(stats_at_t, primary_stat):
    """Primary score [D] (float64) at one block and step.

    stats_at_t maps a stat name to its [D] vector. primary_stat is
    'abs_signed_mean' or 'mean_abs'."""
    if primary_stat == "abs_signed_mean":
        return np.abs(np.asarray(stats_at_t["signed_mean"], dtype=np.float64))
    if primary_stat == "mean_abs":
        return np.asarray(stats_at_t["mean_abs"], dtype=np.float64)
    raise ValueError(f"unknown primary_stat {primary_stat!r}")

def sign_consistency(frac_pos):
    """1.0 if the channel keeps one sign on every token, 0.5 if fully mixed."""
    f = np.asarray(frac_pos, dtype=np.float64)
    return np.maximum(f, 1.0 - f)

# ------------------------------------------------------------ atomic IO
def save_json_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)

def save_npz_atomic(path, **arrays):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.savez_compressed(f, **arrays)
    os.replace(tmp, path)

def save_npy_atomic(path, arr):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)

def save_png_atomic(pil_image, path):
    tmp = path + ".tmp.png"
    pil_image.save(tmp, format="PNG")
    os.replace(tmp, path)

def meta_to_npz_field(meta_dict):
    return np.array(json.dumps(meta_dict, default=str))

def meta_from_npz(z):
    return json.loads(str(z["meta_json"].item()))

# --------------------------------------------------- scenario bookkeeping
def scenario_id(pid, seed):
    return f"{pid}_s{seed}"

def scenario_paths(out_dir, sid, save_tensor, lstar=None, tstar=None):
    p = {"npz": os.path.join(out_dir, f"{sid}.npz"),
         "png": os.path.join(out_dir, f"{sid}.png"),
         "tensor": None}
    if save_tensor:
        assert lstar is not None and tstar is not None, \
            "tensor capture requested but (l*, t*) not set"
        p["tensor"] = os.path.join(out_dir, f"{sid}__Xfull_l{lstar:02d}_t{tstar:02d}.npy")
    return p

def is_scenario_complete(paths, need_tensor, expect=None):
    """True if the npz and png (and the tensor, when needed) exist and the npz
    opens with the required keys. If `expect` holds meta fields such as lstar,
    tstar or primary_stat, the stored meta has to match them too, so a resume
    doesn't keep results made under an older pilot decision."""
    if not (os.path.exists(paths["npz"]) and os.path.exists(paths["png"])):
        return False
    if need_tensor and not (paths["tensor"] and os.path.exists(paths["tensor"])):
        return False
    try:
        with np.load(paths["npz"], allow_pickle=False) as z:
            if not (("meta_json" in z.files) and ("sigmas" in z.files)):
                return False
            if expect:
                m = meta_from_npz(z)
                for k, v in expect.items():
                    if m.get(k) != v:
                        return False
            return True
    except Exception:
        return False

# ---------------------------------------------------------- run log CSV
RUN_LOG_FIELDS = [
    "timestamp", "tag", "sid", "prompt_id", "seed", "status", "wall_s",
    "peak_vram_gib", "fires_ok", "nan_flag", "hook_errors", "tensor_captured",
    "top1_channel", "top1_score", "top1_over_median", "top1_sign_consistency",
    "jacc_primary_vs_meanabs_top10", "precision_mode",
]

def append_run_log(csv_path, row_dict):
    new = not os.path.exists(csv_path)
    row = {k: row_dict.get(k, "") for k in RUN_LOG_FIELDS}
    row["timestamp"] = row.get("timestamp") or datetime.datetime.now().isoformat(timespec="seconds")
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RUN_LOG_FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)

def append_error_log(path, sid, err_text):
    with open(path, "a") as f:
        f.write(f"[{datetime.datetime.now().isoformat(timespec='seconds')}] {sid}: {err_text}\n\n")

def sha256_of_text(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

print("[lib-core] loaded:", ", ".join(
    ["jaccard", "topk_ids", "rank_of", "primary_score", "sign_consistency",
     "atomic IO", "scenario paths", "run log"]))


### LIB F. Figure style

`save_fig` doesn't add a title. The figure-finishing routine of the
intervention cells, `save_paper_fig`, writes a titled and an untitled copy of
every figure, and caption drafts go to `.caption.txt` files next to them for
use in LaTeX. Categorical colors use the colorblind-safe Okabe-Ito palette.
Magnitude heatmaps use sequential colormaps with a log norm, and the diverging
colormap is only used for signed data. Each figure is saved as a vector PDF
and a 600 dpi PNG, with TrueType (Type 42) fonts sized for a single column
(3.35 in) or the full page width (7 in).

In [ ]:
# =====================================================================
# LIB F. Figure style
#
# save_fig doesn't add a title. save_paper_fig, defined with the
# intervention config, writes a titled and an untitled copy of each figure.
# Caption drafts go to <stem>.caption.txt for LaTeX.
# Colors follow the Okabe-Ito palette. Magnitude heatmaps use sequential
# colormaps with a log norm, and signed data uses a dark-centered diverging
# colormap. Each figure is saved as a vector PDF plus a 600 dpi PNG, with
# TrueType (Type 42) fonts sized for a column or the full page.
#
# For color coding, each channel is put in a category by comparing a
# per-channel magnitude statistic with the median channel. This follows the
# massive-activations habit of calling a value an outlier when it is orders
# of magnitude above the typical one.
#   massive   statistic >= MASSIVE_MULT x median   (default 100x)
#   high      statistic >= HIGH_MULT x median      (default 10x)
#   typical   everything else
# Captions that use these colors state the multipliers.
# =====================================================================
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm

FIG_COL, FIG_PAGE = 3.35, 7.00      # inches, single column and full width
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00",
             "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
_BASE_PT = 8.0

def use_paper_style():
    mpl.rcParams.update({
        "figure.constrained_layout.use": True,
        "figure.dpi": 110,
        "pdf.fonttype": 42, "ps.fonttype": 42,
        "font.size": _BASE_PT,
        "axes.titlesize": _BASE_PT, "axes.labelsize": _BASE_PT,
        "xtick.labelsize": _BASE_PT - 1, "ytick.labelsize": _BASE_PT - 1,
        "legend.fontsize": _BASE_PT - 1, "legend.frameon": False,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "xtick.major.size": 2.5, "ytick.major.size": 2.5,
        "lines.linewidth": 1.0,
        "axes.prop_cycle": mpl.cycler(color=OKABE_ITO),
    })

use_paper_style()

MODEL_LABEL = {
    "black-forest-labs/FLUX.1-dev": "FLUX.1-dev",
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS": "PixArt-Sigma",
    "stabilityai/stable-diffusion-3-medium-diffusers": "SD3-Medium",
}.get(CFG.model_id, CFG.model_id)

BLOCK_AXIS_LABEL = {
    "FLUX.1-dev": "block index (0 to 18 dual, 19 to 56 single)",
}.get(MODEL_LABEL, "block index")

PRIMARY_TEX = {"mean_abs": r"mean $|x|$",
               "abs_signed_mean": r"$|\mathrm{mean}\,x|$"}
PRIMARY_WORDS = {"mean_abs": "the per-channel mean absolute activation",
                 "abs_signed_mean": "the absolute per-channel signed mean"}

# ---------------- channel categorization (high / massive color coding)
HIGH_MULT, MASSIVE_MULT = 10.0, 100.0
# profile figures on a white background use warm accent colors
CAT_COLORS = {0: "#6A51A3", 1: "#E69F00", 2: "#D55E00"}
# 3D bar landscape uses a blue floor and purples for the outlier tiers
BAR3D_COLORS = {0: "#6BAED6", 1: "#9E8FD0", 2: "#5C2D91"}
CAT_NAMES = {0: "typical",
             1: f"high (>= {HIGH_MULT:g}x median)",
             2: f"massive (>= {MASSIVE_MULT:g}x median)"}
CAT_CAPTION = (f"Channels are categorized relative to the median channel of "
               f"the same statistic: massive is at least {MASSIVE_MULT:g} "
               f"times the median, high is at least {HIGH_MULT:g} times the "
               f"median.")

def classify_channels(vec, high_mult=None, massive_mult=None):
    """[D] magnitude statistic -> (categories [D] int8, median).

    2 is massive, 1 high, 0 typical. Thresholds are multiples of the median
    channel and default to HIGH_MULT and MASSIVE_MULT."""
    v = np.asarray(vec, np.float64)
    med = max(float(np.median(v)), 1e-12)
    hm = HIGH_MULT if high_mult is None else float(high_mult)
    mm = MASSIVE_MULT if massive_mult is None else float(massive_mult)
    cat = np.zeros(v.size, np.int8)
    cat[v >= hm * med] = 1
    cat[v >= mm * med] = 2
    return cat, med

def _make_signed_cmap():
    """Black-centered diverging colormap. The positive half is magma, as in
    the magnitude heatmaps, and the negative half is a matching ramp from
    black through blue and cyan to pale cyan, so all landscapes look alike."""
    from matplotlib.colors import ListedColormap, LinearSegmentedColormap
    pos = plt.get_cmap("magma")(np.linspace(0.0, 1.0, 256))
    neg_ramp = LinearSegmentedColormap.from_list(
        "negside", ["#000004", "#1D1147", "#1F4287", "#2D7DD2",
                    "#40C9E0", "#B5F2F5"])
    neg = neg_ramp(np.linspace(0.0, 1.0, 256))[::-1]
    return ListedColormap(np.vstack([neg, pos]), name="signed_magma")

SIGNED_CMAP = _make_signed_cmap()

def fig_stem(name):
    return os.path.join(PATHS["figures"], name)

def save_fig(fig, stem, caption=None, show=True, tight=True):
    """Write <stem>.pdf, <stem>.png and, with a caption, <stem>.caption.txt
    atomically.

    Use tight=False for 3D axes. The tight bounding box under-measures 3D
    axis labels and crops them, and their placement drifts a little with
    the raster dpi, so 3D figures skip the tight box and set their margins
    with ax.set_position instead."""
    for ext, dpi in (("pdf", None), ("png", 600)):
        tmp = f"{stem}.tmp.{ext}"
        fig.savefig(tmp, bbox_inches="tight" if tight else None, dpi=dpi)
        os.replace(tmp, f"{stem}.{ext}")
    if caption:
        tmp = f"{stem}.caption.txt.tmp"
        with open(tmp, "w") as f:
            f.write(caption.strip() + "\n")
        os.replace(tmp, f"{stem}.caption.txt")
    print(f"[fig] saved {os.path.basename(stem)}"
          + (" (.pdf/.png/.caption.txt)" if caption else " (.pdf/.png)"))
    if show:
        plt.show()

# Color floor percentile for the magnitude heatmaps. At 1.0 the whole line
# texture of the landscape stays visible. Raise it toward 90 to black out
# the weaker lines and show only the strongest channels.
HEAT_FLOOR_PCTL = 1.0

def _cluster_sides(xs_sorted, span, sep_frac=0.04):
    """Pick a label side for each sorted point. Points closer than sep_frac
    of the axis span form a cluster. A lone point is labeled centered
    above, a pair gets left and right, and a larger cluster gets left, then
    center for the middle points, then right."""
    sep = sep_frac * float(span)
    sides, cluster = [], [0]

    def _flush(c):
        if len(c) == 1:
            sides.append("center")
        elif len(c) == 2:
            sides.extend(["left", "right"])
        else:
            sides.append("left")
            sides.extend(["center"] * (len(c) - 2))
            sides.append("right")

    for i in range(1, len(xs_sorted)):
        if xs_sorted[i] - xs_sorted[i - 1] <= sep:
            cluster.append(i)
        else:
            _flush(cluster)
            cluster = [i]
    _flush(cluster)
    return sides

def annotate_points(ax, xs, ys, labels):
    """Channel-id labels at their points. A lone channel is labeled directly
    above its point. Two neighboring channels are labeled to the left and
    right of their points, so the pair stays readable without extra marks."""
    xs = np.asarray(xs, dtype=np.float64)
    ys = np.asarray(ys, dtype=np.float64)
    order = np.argsort(xs)
    xmin, xmax = ax.get_xlim()
    sides = _cluster_sides([float(xs[k]) for k in order], xmax - xmin)
    off = {"left": (-5, 1), "center": (0, 5), "right": (5, 1)}
    ha = {"left": "right", "center": "center", "right": "left"}
    for k, side in zip(order, sides):
        ax.annotate(str(labels[k]), (xs[k], ys[k]),
                    xytext=off[side], textcoords="offset points",
                    ha=ha[side], va="bottom", fontsize=_BASE_PT - 1.5)

def label_channels_top(ax, positions, labels, color="0.15"):
    """Channel-id labels in one row above the axes. Labels that would overlap
    are pushed sideways just enough to clear each other, using widths
    estimated from the font size and axes width. A thin leader line ties
    each label to its exact channel position."""
    if len(positions) == 0:
        return
    xs = np.asarray(positions, dtype=np.float64)
    order = np.argsort(xs)
    xs = xs[order]
    labs = [str(int(np.asarray(labels)[k])) for k in order]
    xmin, xmax = sorted(ax.get_xlim())
    fs = _BASE_PT - 3
    ax_pt = max(ax.figure.get_figwidth() * ax.get_position().width * 72.0,
                1.0)
    upp = (xmax - xmin) / ax_pt                    # data units per point
    wid = [(0.62 * fs * len(t) + 3.0) * upp for t in labs]
    pos = list(xs)
    for _ in range(200):                           # 1D dodge until no labels overlap
        moved = False
        for i in range(1, len(pos)):
            need = 0.5 * (wid[i - 1] + wid[i])
            if pos[i] - pos[i - 1] < need:
                mid = 0.5 * (pos[i] + pos[i - 1])
                pos[i - 1] = mid - 0.5 * need
                pos[i] = mid + 0.5 * need
                moved = True
        for i in range(len(pos)):                  # stay inside the axes
            pos[i] = min(max(pos[i], xmin + 0.5 * wid[i]),
                         xmax - 0.5 * wid[i])
        if not moved:
            break
    for x0, xl, t in zip(xs, pos, labs):
        ax.annotate(t, xy=(x0, 1.0), xycoords=("data", "axes fraction"),
                    xytext=((xl - x0) / upp, 8.0),
                    textcoords="offset points",
                    ha="center", va="bottom", fontsize=fs, color=color,
                    annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", lw=0.5, color=color,
                                    shrinkA=1.0, shrinkB=0.0))

def legend_above(ax, ncol=2, title=None):
    """Legend in a strip above the axes, never inside the data area."""
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.02), ncol=ncol,
                  borderaxespad=0.0, frameon=False, columnspacing=1.0,
                  handletextpad=0.4, title=title,
                  title_fontsize=_BASE_PT - 1)

def category_profile(ax, vec, cats, top_ids=None, label_top=True):
    """Log-scale per-channel profile with category color coding."""
    D = len(vec)
    ax.semilogy(np.arange(D), vec, lw=0.4, color="0.75", zorder=1)
    for c in (1, 2):
        idx = np.where(cats == c)[0]
        if len(idx):
            ax.scatter(idx, np.asarray(vec)[idx], s=10 if c == 2 else 5,
                       color=CAT_COLORS[c], zorder=3, label=CAT_NAMES[c],
                       linewidths=0)
    ax.set_xlim(0, D)
    if top_ids is not None and label_top:
        annotate_points(ax, list(top_ids),
                        [float(np.asarray(vec)[c]) for c in top_ids],
                        [int(c) for c in top_ids])

print(f"[lib-fig] paper style active | model label: {MODEL_LABEL} | "
      f"signed cmap: {SIGNED_CMAP.name}")


## S2. Prompt manifest and scenario grid
Prompts p00 to p02 are three fixed bridge prompts, used as written. Prompts
p03 to p19 come from the DiffusionDB metadata table (text and NSFW scores
only, no images). We keep ASCII prompts of 5 to 25 words with both NSFW
scores below 0.2, drop anything on a small blocklist or with more than four
comma-separated parts, and remove duplicates after normalizing case and
punctuation. The survivors are sampled round-robin over eight content
buckets so the set isn't dominated by one kind of scene.

If the download fails, a curated fallback list is used instead. In both cases
the list is written once to `prompts.json` with a sha-256 digest, and later
sessions reload that file instead of sampling again.

In [ ]:
# =====================================================================
# S2. Prompt manifest and scenario grid
# =====================================================================
# p00 to p02 are the three fixed bridge prompts. p03 to p19 are filtered
# from poloclub/diffusiondb metadata.parquet and sampled over 8 content
# buckets, with a curated fallback list if that fails. The list is written
# to prompts.json on the first build and reloaded, not resampled, after that.
import random

TABLE6_PROMPTS = [
    "a cat holding hello world sign",
    "an astronaut in the moon holding the USA flag",
    "a protestor in a city square holding a 'change' sign",
]

FALLBACK_PROMPTS = [
    "a close-up portrait of an elderly fisherman with a weathered face, dramatic lighting",
    "a red fox standing in a snowy birch forest at dawn",
    "a wide mountain valley with a winding river under storm clouds",
    "a cozy cluttered artist studio with paint tubes and canvases by a window",
    "a vintage brass pocket watch on a wooden desk, macro photograph",
    "a rusty pickup truck parked beside a desert gas station at sunset",
    "a neon sign that says 'open all night' above a diner entrance",
    "an isometric illustration of a tiny floating island village, pastel colors",
    "a chef plating dessert in a busy restaurant kitchen",
    "a lighthouse on a cliff during a thunderstorm, long exposure",
    "a black cat sleeping on a stack of old books",
    "an astronaut greenhouse on mars with rows of green plants",
    "a watercolor painting of a rainy tokyo street at night",
    "a medieval knight's helmet on a stone pedestal in a museum",
    "a hot air balloon festival over lavender fields",
    "a robot barista serving coffee in a futuristic cafe",
    "an oil painting of a stormy sea with a small sailboat",
]

BUCKET_KEYWORDS = {
    "portrait":  ("portrait", "face", " man ", " woman ", " girl ", " boy ", "person"),
    "animal":    ("cat", "dog", "fox", "bird", "horse", "wolf", "tiger", "lion", "deer", "animal"),
    "landscape": ("mountain", "forest", "valley", "landscape", "river", "desert",
                  "beach", "sunset", "field", "waterfall"),
    "indoor":    ("room", "kitchen", "interior", "office", "studio", "library", "bedroom"),
    "object":    ("watch", "bottle", "chair", "table", "lamp", "book", "sword",
                  "cup", "clock", "vase"),
    "vehicle":   ("car", "truck", "ship", "train", "plane", "motorcycle", "boat", "bus"),
    "text_in_image": ("sign", " text", "poster", "billboard", "label", "graffiti"),
    "stylized":  ("watercolor", "oil painting", "isometric", "pixel art",
                  "illustration", "anime", "sketch", "low poly"),
}

_BLOCKLIST_RE = re.compile(
    r"\b(nsfw|nude|naked|topless|porn|sexy|erotic|hentai|lingerie|gore|blood|"
    r"beheading|corpse|mutilat\w*|loli|shota)\b", re.IGNORECASE)

def _normalize_prompt(p):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(p).lower())).strip()

def _passes_filters(p, p_nsfw, i_nsfw):
    if not isinstance(p, str):
        return False
    p = p.strip()
    if not (p and p.isascii()):
        return False
    n_words = len(p.split())
    if not (5 <= n_words <= 25):
        return False
    try:
        if float(p_nsfw) >= 0.2 or float(i_nsfw) >= 0.2:
            return False
    except (TypeError, ValueError):
        return False
    if _BLOCKLIST_RE.search(p):
        return False
    if p.count(",") + 1 > 4:            # at most 4 comma-separated segments
        return False
    return True

def _bucket_of(p):
    q = f" {p.lower()} "
    for b, kws in BUCKET_KEYWORDS.items():
        if any(kw in q for kw in kws):
            return b
    return "other"

def select_diffusiondb_prompts(n_needed, rng_seed, exclude_norms):
    """Download the DiffusionDB metadata (prompt text and NSFW scores, no
    images), filter and dedupe it, then pick n_needed prompts round-robin
    over the content buckets."""
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as pq
    print("[prompts] downloading DiffusionDB metadata.parquet (~0.5 GB, one time) ...")
    fp = hf_hub_download("poloclub/diffusiondb", "metadata.parquet", repo_type="dataset")
    tbl = pq.read_table(fp, columns=["prompt", "prompt_nsfw", "image_nsfw"])
    df = tbl.to_pandas()
    print(f"[prompts] {len(df):,} raw rows")
    df = df[df["prompt"].map(lambda p: _passes_filters(p, 0.0, 0.0))]  # cheap text filters first
    mask = [( _passes_filters(p, pn, im)) for p, pn, im in
            zip(df["prompt"], df["prompt_nsfw"], df["image_nsfw"])]
    df = df[np.asarray(mask, dtype=bool)]
    print(f"[prompts] {len(df):,} rows after filters")
    seen, rows = set(exclude_norms), []
    for p in df["prompt"]:
        k = _normalize_prompt(p)
        if k and k not in seen:
            seen.add(k)
            rows.append(p.strip())
    buckets = {b: [] for b in list(BUCKET_KEYWORDS) + ["other"]}
    for p in rows:
        buckets[_bucket_of(p)].append(p)
    rng = random.Random(rng_seed)
    for b in buckets:
        rng.shuffle(buckets[b])
    order = list(BUCKET_KEYWORDS) + ["other"]
    picked, provenance = [], []
    while len(picked) < n_needed:
        progressed = False
        for b in order:
            if buckets[b] and len(picked) < n_needed:
                p = buckets[b].pop()
                picked.append(p)
                provenance.append(b)
                progressed = True
        if not progressed:
            raise RuntimeError("DiffusionDB pool exhausted before filling the manifest")
    return picked, provenance

def build_or_load_prompts(cfg, meta_dir):
    """Return ({pid: text}, manifest dict). The manifest is fixed after the
    first build."""
    path = os.path.join(meta_dir, "prompts.json")
    if os.path.exists(path):
        with open(path) as f:
            man = json.load(f)
        prompts = {e["pid"]: e["text"] for e in man["prompts"]}
        assert len(prompts) == cfg.n_prompts, \
            f"frozen prompts.json has {len(prompts)} prompts, config expects {cfg.n_prompts}"
        got = sha256_of_text("\n".join(prompts[f"p{i:02d}"] for i in range(cfg.n_prompts)))
        assert got == man["sha256"], "prompts.json content/sha mismatch -- file corrupted?"
        print(f"[prompts] loaded FROZEN manifest ({man['source']}, sha {man['sha256']})")
        return prompts, man
    n_extra = cfg.n_prompts - len(TABLE6_PROMPTS)
    exclude = {_normalize_prompt(p) for p in TABLE6_PROMPTS}
    try:
        extra, prov = select_diffusiondb_prompts(n_extra, cfg.prompt_source_seed, exclude)
        source = "table6+diffusiondb"
    except Exception as e:
        print(f"[prompts][warn] DiffusionDB path failed ({type(e).__name__}: {e})")
        print("[prompts] using curated fallback list (source recorded in manifest).")
        extra, prov = FALLBACK_PROMPTS[:n_extra], ["curated"] * n_extra
        source = "table6+curated_fallback"
    texts = TABLE6_PROMPTS + extra
    assert len(texts) == cfg.n_prompts, \
        f"prompt build produced {len(texts)} prompts, config expects {cfg.n_prompts}"
    provs = ["table6_bridge"] * len(TABLE6_PROMPTS) + prov
    entries = [{"pid": f"p{i:02d}", "text": t, "bucket": b}
               for i, (t, b) in enumerate(zip(texts, provs))]
    man = {"source": source, "rng_seed": cfg.prompt_source_seed,
           "built": datetime.datetime.now().isoformat(timespec="seconds"),
           "sha256": sha256_of_text("\n".join(texts)), "prompts": entries}
    save_json_atomic(man, path)
    print(f"[prompts] built & FROZE manifest -> {path} (source={source})")
    return {e["pid"]: e["text"] for e in entries}, man

PROMPTS, PROMPT_MANIFEST = build_or_load_prompts(CFG, PATHS["meta"])

# ---------------- scenario grid, prompt-major (all seeds of p00, then p01, ...)
SCENARIOS = [(scenario_id(pid, s), pid, s, PROMPTS[pid])
             for pid in sorted(PROMPTS) for s in CFG.seeds]
import pandas as pd
scen_df = pd.DataFrame(SCENARIOS, columns=["sid", "prompt_id", "seed", "prompt"])
scen_df.to_csv(os.path.join(PATHS["meta"], "scenarios.csv"), index=False)
print(f"[prompts] {len(SCENARIOS)} scenarios written to scenarios.csv")
for pid in sorted(PROMPTS):
    print(f"    {pid}: {PROMPTS[pid]}")


## S3. Model loading and smoke test
Loading happens in two phases. S3a loads only the text encoders (T5-XXL and
CLIP), encodes all 20 prompts once, saves the fp32 embeddings to Drive and
unloads the encoders. That frees about 10 GiB of VRAM, gives every seed of a
prompt exactly the same conditioning, and means a resumed session never
loads T5 at all. S3b then loads the generation pipeline without text
encoders.

S3c is a 4-step smoke run with hooks on all 57 blocks. It checks the block
count, that the image stream is found in both dual and single blocks, that
every hook fired the right number of times, that the statistics are finite,
and that the saved tensor and token norms load back from disk. Nothing else
should be run until it passes.

In [ ]:
# =====================================================================
# S3a. Prompt embedding cache
# =====================================================================
# Encode all 20 prompts once with T5-XXL and CLIP, save the fp32 embeddings
# to Drive and delete the text encoders. This frees about 10 GiB of VRAM,
# keeps the conditioning identical across seeds, and lets a resumed session
# skip T5.

def build_prompt_embed_cache(cfg, prompts, embed_dir, token, enc_dtype):
    embed_path = lambda pid: os.path.join(embed_dir, f"{pid}.npz")

    def _valid(pid):
        p = embed_path(pid)
        if not os.path.exists(p):
            return False
        try:
            with np.load(p, allow_pickle=False) as z:
                m = meta_from_npz(z)
                return (m["text_sha"] == sha256_of_text(prompts[pid])
                        and z["prompt_embeds"].shape[1] == cfg.max_seq_len)
        except Exception:
            return False

    missing = [pid for pid in sorted(prompts) if not _valid(pid)]
    if not missing:
        print(f"[embeds] all {len(prompts)} prompt embeddings cached & verified -- "
              f"text encoders will NOT be loaded this session.")
        return
    print(f"[embeds] encoding {len(missing)} prompt(s): {missing}")
    from diffusers import FluxPipeline
    pipe_text = FluxPipeline.from_pretrained(
        cfg.model_id, transformer=None, vae=None,
        torch_dtype=enc_dtype, token=token)
    pipe_text.to("cuda")
    with torch.inference_mode():
        for pid in missing:
            text = prompts[pid]
            pe, pp, _text_ids = pipe_text.encode_prompt(
                prompt=text, prompt_2=text, device="cuda",
                max_sequence_length=cfg.max_seq_len)
            assert pe.ndim == 3 and pe.shape[0] == 1 and pe.shape[1] == cfg.max_seq_len, \
                f"unexpected prompt_embeds shape {tuple(pe.shape)}"
            assert pp.ndim == 2 and pp.shape[0] == 1, \
                f"unexpected pooled shape {tuple(pp.shape)}"
            save_npz_atomic(
                embed_path(pid),
                prompt_embeds=pe.float().cpu().numpy(),
                pooled=pp.float().cpu().numpy(),
                meta_json=meta_to_npz_field({
                    "pid": pid, "text": text, "text_sha": sha256_of_text(text),
                    "enc_dtype": str(enc_dtype), "max_seq_len": cfg.max_seq_len,
                    "model_id": cfg.model_id}))
            print(f"    {pid}: encoded & saved  (T5 tokens={pe.shape[1]})")
    del pipe_text
    gc.collect(); torch.cuda.empty_cache()
    print("[embeds] text encoders unloaded; VRAM freed for generation.")

def load_embed_cache(prompts, embed_dir):
    """Return {pid: (prompt_embeds [1,512,4096], pooled [1,768])}, fp32 on CPU."""
    out = {}
    for pid in sorted(prompts):
        with np.load(os.path.join(embed_dir, f"{pid}.npz"), allow_pickle=False) as z:
            m = meta_from_npz(z)
            assert m["text_sha"] == sha256_of_text(prompts[pid]), \
                f"{pid}: cached embedding text mismatch -- delete {embed_dir} and re-run"
            out[pid] = (torch.from_numpy(z["prompt_embeds"].copy()),
                        torch.from_numpy(z["pooled"].copy()))
    return out

build_prompt_embed_cache(CFG, PROMPTS, PATHS["embeds"], HF_TOKEN, DTYPE)
EMBEDS = load_embed_cache(PROMPTS, PATHS["embeds"])
N_TXT = int(next(iter(EMBEDS.values()))[0].shape[1])
assert N_TXT == CFG.max_seq_len
print(f"[embeds] loaded {len(EMBEDS)} embeddings into RAM "
      f"(n_txt={N_TXT}, ~{len(EMBEDS)*8.4:.0f} MB)")


In [ ]:
# =====================================================================
# S3b. Generation pipeline
# =====================================================================
# The text encoders stay unloaded because the embeddings are cached.
#   bf16_full      everything on the GPU (A100 40/80 GB), the main path
#   bf16_offload   used automatically if bf16_full runs out of memory
#   quant_fp8      torchao float8 weight-only transformer (L4, Ada)
#   quant_nf4      bitsandbytes NF4 transformer, last resort (T4 etc.)
# The quantized modes change the activations we measure, so every saved
# file records `precision_mode`.

def load_generation_pipeline(cfg, mode, dtype, token):
    from diffusers import FluxPipeline
    no_text = dict(text_encoder=None, tokenizer=None,
                   text_encoder_2=None, tokenizer_2=None)

    def _bf16():
        pipe = FluxPipeline.from_pretrained(cfg.model_id, torch_dtype=dtype,
                                            token=token, **no_text)
        try:
            pipe.to("cuda")
            return pipe, "bf16_full"
        except torch.cuda.OutOfMemoryError:
            print("[pipe][warn] full-GPU load OOM'd -> enabling model CPU offload")
            gc.collect(); torch.cuda.empty_cache()
            pipe.enable_model_cpu_offload()
            return pipe, "bf16_offload"

    def _fp8():
        from diffusers import FluxTransformer2DModel, TorchAoConfig
        last_err = None
        for qname in ("float8wo", "float8_weight_only"):
            try:
                qc = TorchAoConfig(qname)
                tr = FluxTransformer2DModel.from_pretrained(
                    cfg.model_id, subfolder="transformer",
                    quantization_config=qc, torch_dtype=dtype, token=token)
                pipe = FluxPipeline.from_pretrained(
                    cfg.model_id, transformer=tr, torch_dtype=dtype,
                    token=token, **no_text)
                pipe.to("cuda")
                return pipe, "quant_fp8"
            except Exception as e:
                last_err = e
        raise RuntimeError(f"torchao fp8 path failed: {last_err}")

    def _nf4():
        from diffusers import FluxTransformer2DModel, BitsAndBytesConfig
        qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=dtype)
        tr = FluxTransformer2DModel.from_pretrained(
            cfg.model_id, subfolder="transformer",
            quantization_config=qc, torch_dtype=dtype, token=token)
        pipe = FluxPipeline.from_pretrained(
            cfg.model_id, transformer=tr, torch_dtype=dtype,
            token=token, **no_text)
        try:                       # bnb weights are already on the GPU, move the rest
            pipe.to("cuda")
        except Exception:
            pipe.vae.to("cuda")
        return pipe, "quant_nf4"

    if mode == "bf16_full":
        return _bf16()
    if mode == "quant_fp8":
        try:
            return _fp8()
        except Exception as e:
            print(f"[pipe][warn] fp8 unavailable ({e}); cascading to NF4")
            gc.collect(); torch.cuda.empty_cache()
            return _nf4()
    if mode == "quant_nf4":
        return _nf4()
    raise ValueError(f"unknown precision mode {mode!r}")

print(f"[pipe] loading FLUX.1-dev generation pipeline (mode={PRECISION_MODE}, "
      f"dtype={DTYPE}) -- first load downloads ~24 GB of transformer/VAE weights ...")
_t0 = time.time()
pipe, PRECISION_MODE = load_generation_pipeline(CFG, PRECISION_MODE, DTYPE, HF_TOKEN)
pipe.set_progress_bar_config(disable=True)
try:
    _pdev = next(pipe.transformer.parameters()).device
    _pdt = next(pipe.transformer.parameters()).dtype
except StopIteration:
    _pdev = _pdt = "n/a"
print(f"[pipe] ready in {time.time()-_t0:.0f}s | effective mode={PRECISION_MODE} | "
      f"transformer on {_pdev} ({_pdt}) | "
      f"VRAM allocated {torch.cuda.memory_allocated()/2**30:.1f} GiB")


### LIB B and LIB C. Measurement core
Forward hooks sit on the block outputs. The image stream is found by its
shape, not by its position in the output tuple. Dual blocks return two
streams and we take the one with sequence length 4096. Single blocks may
return the joined text-then-image sequence, and then the first 512 text
tokens are sliced off. Statistics are computed in fp32 on the GPU, so only
five D-dimensional vectors per block and step are copied to the CPU.

There is exactly one transformer forward per denoising step, so each
block's hook counter doubles as the step index. The counters are checked
when the recorder is finalized.

`run_scenario` is the one function used by the smoke test, the pilot and
the main run, so every phase measures the same way. It saves the image, then
the tensor, then the npz. The npz is written last and marks a scenario as
complete when resuming.

In [ ]:
# =====================================================================
# LIB B. Activation recorder
# =====================================================================
# We measure the residual-stream hidden state at each block output. Only
# the image stream is used, batch 1, cast to fp32 on the GPU before any
# reduction.
#
# The image stream is picked by shape, not by tuple position.
#   dual (MMDiT) blocks      separate streams, take the one with seq_len n_img
#   single blocks            joined [text | image] sequence, take [:, n_txt:, :]
#   tuple from single block  (newer diffusers) same rule as dual blocks
# Any other output raises inside the hook. Hook errors are logged and never
# stop the generation.
import math
import numpy as np
import torch

STAT_NAMES = ("signed_mean", "mean_abs", "max_abs", "p99_abs", "frac_pos")

def enumerate_flux_blocks(transformer):
    """List of (combined_index, module, kind). Dual blocks are 0..18, single
    blocks 19..56."""
    blocks = []
    for m in transformer.transformer_blocks:
        blocks.append((len(blocks), m, "dual"))
    for m in transformer.single_transformer_blocks:
        blocks.append((len(blocks), m, "single"))
    return blocks

def compute_channel_stats(X):
    """Per-channel statistics over tokens. X is a float32 [N_tokens, D]
    tensor on any device. Returns a dict of [D] float32 tensors.

    signed_mean    mean_n X[n,d], its absolute value is the default ranking statistic
    mean_abs       mean_n |X[n,d]|, used for the landscape
    max_abs        max_n |X[n,d]|, sensitive to single-token spikes
    p99_abs        99th percentile of |X[.,d]|, less spike-sensitive
    frac_pos       fraction of tokens with X[n,d] > 0, shows sign structure
    """
    N = X.shape[0]
    absX = X.abs()
    k99 = min(N, max(1, int(math.ceil(0.99 * N))))
    return {
        "signed_mean": X.mean(dim=0),
        "mean_abs":    absX.mean(dim=0),
        "max_abs":     absX.amax(dim=0),
        "p99_abs":     absX.kthvalue(k99, dim=0).values,
        "frac_pos":    (X > 0).to(torch.float32).mean(dim=0),
    }

class ActivationRecorder:
    def __init__(self, transformer, n_steps, n_img, n_txt, d_model, stat_blocks,
                 lstar=None, tstar=None, capture_full=False,
                 capture_token_norms=False):
        self.all_blocks = enumerate_flux_blocks(transformer)
        self.n_steps, self.n_img, self.n_txt, self.d_model = \
            int(n_steps), int(n_img), int(n_txt), int(d_model)
        self.stat_blocks = sorted({int(b) for b in stat_blocks})
        bad = [b for b in self.stat_blocks if not (0 <= b < len(self.all_blocks))]
        if bad:
            raise ValueError(f"stat_blocks out of range: {bad} "
                             f"(model has {len(self.all_blocks)} blocks)")
        self.lstar = None if lstar is None else int(lstar)
        self.tstar = None if tstar is None else int(tstar)
        self.capture_full = bool(capture_full)
        self.capture_token_norms = bool(capture_token_norms)
        if self.capture_full or self.capture_token_norms:
            assert self.lstar in self.stat_blocks, \
                "capture requested but lstar not among stat_blocks"
        self._handles = []
        self.reset()

    def reset(self):
        self.stats = {b: {s: np.zeros((self.n_steps, self.d_model), np.float32)
                          for s in STAT_NAMES} for b in self.stat_blocks}
        self.counters = {b: 0 for b in self.stat_blocks}
        self.overflow = {b: 0 for b in self.stat_blocks}
        self.token_norms = (np.zeros((self.n_steps, self.n_img), np.float16)
                            if (self.capture_token_norms and self.lstar is not None)
                            else None)
        self.full_tensor = None
        self.full_clamped = 0
        self.errors = []

    # ------------------------------------------------ stream extraction
    def _extract_image_stream(self, output):
        if isinstance(output, (tuple, list)):
            cands = [o for o in output if torch.is_tensor(o) and o.dim() == 3]
            for o in cands:
                if o.shape[1] == self.n_img:
                    return o
            raise RuntimeError(
                "no image stream (seq_len=%d) in tuple output; got shapes %s"
                % (self.n_img, [tuple(o.shape) for o in cands]))
        if torch.is_tensor(output) and output.dim() == 3:
            L = output.shape[1]
            if L == self.n_img:
                return output
            if L == self.n_img + self.n_txt:      # FLUX joins as [text | image]
                return output[:, self.n_txt:, :]
            raise RuntimeError(
                f"unexpected sequence length {L} "
                f"(expected {self.n_img} or {self.n_txt + self.n_img})")
        raise RuntimeError(f"unexpected hook output type {type(output)}")

    # ------------------------------------------------------------ hooks
    def _make_hook(self, bidx):
        def hook(module, args, output):
            step = self.counters[bidx]
            self.counters[bidx] = step + 1
            if step >= self.n_steps:
                self.overflow[bidx] += 1
                return
            try:
                img = self._extract_image_stream(output)
                if img.shape[0] != 1:
                    raise RuntimeError(f"expected batch 1, got shape {tuple(img.shape)}")
                if img.shape[2] != self.d_model:
                    raise RuntimeError(f"hidden size {img.shape[2]} != d_model {self.d_model}")
                X = img[0].to(torch.float32)
                st = compute_channel_stats(X)
                packed = torch.stack([st[s] for s in STAT_NAMES], dim=0) \
                              .to("cpu", torch.float32).numpy()   # one device-to-host copy
                for si, s in enumerate(STAT_NAMES):
                    self.stats[bidx][s][step] = packed[si]
                if self.token_norms is not None and bidx == self.lstar:
                    self.token_norms[step] = \
                        X.norm(dim=1).to("cpu", torch.float16).numpy()
                if (self.capture_full and bidx == self.lstar
                        and self.tstar is not None and step == self.tstar):
                    n_over = int((X.abs() > 65504.0).sum().item())
                    if n_over:                      # would overflow fp16, so clamp and count
                        self.full_clamped = n_over
                    self.full_tensor = X.clamp(-65504.0, 65504.0) \
                                        .to("cpu", torch.float16).numpy()
            except Exception as e:   # log it, don't crash the generation
                self.errors.append(
                    f"block {bidx} step {step}: {type(e).__name__}: {e}")
        return hook

    def attach(self):
        assert not self._handles, "recorder already attached"
        hooked = set(self.stat_blocks)
        for i, mod, _kind in self.all_blocks:
            if i in hooked:
                self._handles.append(mod.register_forward_hook(self._make_hook(i)))
        return self

    def detach(self):
        for h in self._handles:
            h.remove()
        self._handles = []

    def finalize(self, expected_steps=None):
        exp = int(self.n_steps if expected_steps is None else expected_steps)
        fires_ok = (all(self.counters[b] == exp for b in self.stat_blocks)
                    and all(v == 0 for v in self.overflow.values()))
        nan_flag = False
        for b in self.stat_blocks:
            for s in STAT_NAMES:
                if not np.isfinite(self.stats[b][s][:exp]).all():
                    nan_flag = True
        return {"fires_ok": bool(fires_ok),
                "nan_flag": bool(nan_flag),
                "hook_errors": len(self.errors),
                "errors": list(self.errors),
                "counters": {int(b): int(c) for b, c in self.counters.items()},
                "tensor_captured": self.full_tensor is not None,
                "full_tensor_clamped": int(self.full_clamped)}

print(f"[lib-recorder] loaded (stats: {', '.join(STAT_NAMES)})")


In [ ]:
# =====================================================================
# LIB C. Scenario runner
# =====================================================================
# run_scenario does one generation with the recorder attached and saves the
# results atomically. The smoke test, the pilot (phase A) and the main run
# (phase B) all call it. Files are saved as png, then tensor, then npz. The
# npz comes last and marks the scenario as done, so a scenario that was cut
# off halfway is simply run again.
import gc, time, datetime
import numpy as np
import torch

def run_scenario(pipe, cfg, sid, pid, prompt_text, seed, embeds, out_dir,
                 stat_blocks, n_steps, n_img, pipe_dtype, device="cuda",
                 lstar=None, tstar=None, primary_stat=None,
                 capture_full=False, capture_token_norms=False,
                 tag="main", extra_meta=None, force=False):
    extra_meta = dict(extra_meta or {})
    need_tensor = bool(capture_full and cfg.save_full_tensor)
    paths = scenario_paths(out_dir, sid, need_tensor, lstar, tstar)
    expect = None
    if lstar is not None and tstar is not None and primary_stat is not None:
        expect = {"lstar": int(lstar), "tstar": int(tstar),
                  "primary_stat": primary_stat}
    if not force and is_scenario_complete(paths, need_tensor=need_tensor,
                                          expect=expect):
        return {"tag": tag, "sid": sid, "prompt_id": pid, "seed": seed,
                "status": "skipped",
                "precision_mode": extra_meta.get("precision_mode", "")}

    pe, pp = embeds[pid]
    n_txt = int(pe.shape[1])
    rec = ActivationRecorder(
        pipe.transformer, n_steps=n_steps, n_img=n_img, n_txt=n_txt,
        d_model=cfg.d_model, stat_blocks=stat_blocks,
        lstar=lstar, tstar=tstar,
        capture_full=need_tensor, capture_token_norms=capture_token_norms)

    torch.manual_seed(seed)                         # global RNG as well
    gen = torch.Generator("cpu").manual_seed(seed)  # CPU generator gives the same noise on any device
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    rec.attach()
    try:
        with torch.inference_mode():
            out = pipe(prompt_embeds=pe.to(device, pipe_dtype),
                       pooled_prompt_embeds=pp.to(device, pipe_dtype),
                       height=cfg.height, width=cfg.width,
                       num_inference_steps=n_steps,
                       guidance_scale=cfg.guidance_scale,
                       generator=gen, output_type="pil")
    finally:
        rec.detach()
    wall = time.time() - t0
    peak = (torch.cuda.max_memory_allocated() / 2**30
            if torch.cuda.is_available() else 0.0)
    image = out.images[0]
    fin = rec.finalize(expected_steps=n_steps)
    if fin["errors"]:
        append_error_log(ERROR_LOG_PATH, sid,
                         "hook errors:\n  " + "\n  ".join(fin["errors"][:20]))

    sig = (pipe.scheduler.sigmas.detach().float().cpu().numpy()
           if hasattr(pipe.scheduler, "sigmas") else np.zeros(0, np.float32))
    tsv = (pipe.scheduler.timesteps.detach().float().cpu().numpy()
           if hasattr(pipe.scheduler, "timesteps") else np.zeros(0, np.float32))

    arrays = {}
    for b in rec.stat_blocks:
        for s in STAT_NAMES:
            arrays[f"b{b:02d}__{s}"] = rec.stats[b][s]
    if rec.token_norms is not None:
        arrays["token_norms_lstar"] = rec.token_norms

    diag = {"tag": tag, "sid": sid, "prompt_id": pid, "seed": seed,
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "fires_ok": fin["fires_ok"], "nan_flag": fin["nan_flag"],
            "hook_errors": fin["hook_errors"],
            "tensor_captured": fin["tensor_captured"],
            "precision_mode": extra_meta.get("precision_mode", "")}

    # ---- per-scenario top-k under the primary statistic and under mean|x|, and their overlap
    if lstar is not None and tstar is not None and primary_stat is not None:
        stats_t = {s: rec.stats[lstar][s][tstar] for s in STAT_NAMES}
        sp = primary_score(stats_t, primary_stat)
        sa = np.asarray(stats_t["mean_abs"], dtype=np.float64)
        ids_p = topk_ids(sp, cfg.top_save)
        ids_a = topk_ids(sa, cfg.top_save)
        arrays["top20_ids"] = ids_p.astype(np.int32)
        arrays["top20_scores"] = sp[ids_p].astype(np.float32)
        arrays["top20_ids_mean_abs"] = ids_a.astype(np.int32)
        arrays["top20_scores_mean_abs"] = sa[ids_a].astype(np.float32)
        cons = sign_consistency(stats_t["frac_pos"])
        top1 = int(ids_p[0])
        med = max(float(np.median(sp)), 1e-12)
        diag.update(
            top1_channel=top1,
            top1_score=round(float(sp[top1]), 4),
            top1_over_median=round(float(sp[top1] / med), 1),
            top1_sign_consistency=round(float(cons[top1]), 4),
            jacc_primary_vs_meanabs_top10=round(jaccard(ids_p[:10], ids_a[:10]), 3))

    meta = {"sid": sid, "prompt_id": pid, "seed": int(seed), "prompt": prompt_text,
            "tag": tag, "model_id": cfg.model_id,
            "height": cfg.height, "width": cfg.width, "num_steps": int(n_steps),
            "guidance_scale": cfg.guidance_scale,
            "n_img": int(n_img), "n_txt": int(n_txt), "d_model": cfg.d_model,
            "stat_blocks": [int(b) for b in rec.stat_blocks],
            "lstar": lstar, "tstar": tstar, "primary_stat": primary_stat,
            "counters": fin["counters"], "fires_ok": fin["fires_ok"],
            "nan_flag": fin["nan_flag"], "hook_errors": fin["hook_errors"],
            "hook_error_msgs": fin["errors"][:10],
            "tensor_captured": fin["tensor_captured"],
            "full_tensor_clamped": fin["full_tensor_clamped"],
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            **extra_meta}
    arrays["sigmas"] = sig.astype(np.float32)
    arrays["timesteps"] = tsv.astype(np.float32)
    arrays["meta_json"] = meta_to_npz_field(meta)

    save_png_atomic(image, paths["png"])
    if need_tensor and rec.full_tensor is not None:
        save_npy_atomic(paths["tensor"], rec.full_tensor)
    save_npz_atomic(paths["npz"], **arrays)   # written last, marks completion

    ok = (fin["fires_ok"] and not fin["nan_flag"] and fin["hook_errors"] == 0
          and (fin["tensor_captured"] or not need_tensor))
    diag["status"] = "ok" if ok else "warn"
    return diag

print("[lib-run] run_scenario ready")


In [ ]:
# =====================================================================
# S3c. Smoke test
# =====================================================================
# One 4-step generation with hooks on all 57 blocks, about 15 to 60 s. It
# asserts that there are 19 dual and 38 single blocks, that the image stream
# (4096 tokens, D=3072) is found in both block types, that each hook fired
# exactly `smoke_steps` times without errors, that the stats are finite, and
# that the full tensor and token norms load back from Drive.
# It also checks, as a warning only, that the top mean|x| channel carries
# over the boundary from the last dual block (18) to the first single
# block (19).
print("=" * 70)
print("SMOKE TEST")
print("=" * 70)

_blocks = enumerate_flux_blocks(pipe.transformer)
_n_dual = sum(1 for _, _, k in _blocks if k == "dual")
assert len(_blocks) == CFG.n_blocks_expected and _n_dual == CFG.n_dual_expected, (
    f"block census mismatch: found {len(_blocks)} total / {_n_dual} dual, "
    f"expected {CFG.n_blocks_expected}/{CFG.n_dual_expected}. diffusers "
    f"{LIB_VERSIONS.get('diffusers')} may have restructured FLUX -- inspect "
    f"pipe.transformer before proceeding.")
print(f"[smoke] block census OK: {_n_dual} dual + {len(_blocks)-_n_dual} single")

SMOKE_SID = "smoke__p00_s0"
SMOKE_LSTAR, SMOKE_TSTAR = 18, CFG.smoke_steps - 1
for _f in glob.glob(os.path.join(PATHS["smoke"], "smoke__*")):
    os.remove(_f)                       # always a fresh smoke run

_diag = run_scenario(
    pipe, CFG, SMOKE_SID, "p00", PROMPTS["p00"], 0, EMBEDS, PATHS["smoke"],
    stat_blocks=list(range(len(_blocks))), n_steps=CFG.smoke_steps, n_img=N_IMG,
    pipe_dtype=DTYPE, device=DEVICE, lstar=SMOKE_LSTAR, tstar=SMOKE_TSTAR,
    primary_stat=CFG.primary_stat_default,
    capture_full=True, capture_token_norms=True, tag="smoke",
    extra_meta={"precision_mode": PRECISION_MODE, "config_hash": CONFIG_HASH},
    force=True)
print(f"[smoke] generation done in {_diag['wall_s']}s "
      f"(peak {_diag['peak_vram_gib']} GiB) status={_diag['status']}")
assert _diag["status"] == "ok", f"smoke diagnostics not clean: {_diag}"

_paths = scenario_paths(PATHS["smoke"], SMOKE_SID, True, SMOKE_LSTAR, SMOKE_TSTAR)
with np.load(_paths["npz"], allow_pickle=False) as _z:
    _meta = meta_from_npz(_z)
    assert _meta["n_txt"] == CFG.max_seq_len, f"n_txt={_meta['n_txt']}"
    assert _meta["n_img"] == N_IMG, f"n_img={_meta['n_img']}"
    assert all(int(c) == CFG.smoke_steps for c in _meta["counters"].values()), \
        f"hook fire counts wrong: {_meta['counters']}"
    _stat_keys = [k for k in _z.files if k.startswith("b") and "__" in k]
    assert len(_stat_keys) == len(_blocks) * len(STAT_NAMES), \
        f"{len(_stat_keys)} stat arrays, expected {len(_blocks)*len(STAT_NAMES)}"
    _tn = _z["token_norms_lstar"]
    assert _tn.shape == (CFG.smoke_steps, N_IMG) and _tn.dtype == np.float16
    assert _z["sigmas"].shape[0] in (0, CFG.smoke_steps + 1)
    for _k in _stat_keys:
        assert np.isfinite(_z[_k]).all(), f"non-finite values in {_k}"
    _v18 = _z["b18__mean_abs"][SMOKE_TSTAR].copy()
    _v19 = _z["b19__mean_abs"][SMOKE_TSTAR].copy()

_X = np.load(_paths["tensor"])
assert _X.shape == (N_IMG, CFG.d_model) and _X.dtype == np.float16, \
    f"tensor round-trip: shape {_X.shape} dtype {_X.dtype}"
_Xf = _X.astype(np.float32)
assert np.isfinite(_Xf).all() and float(np.abs(_Xf).max()) > 0.0
del _X, _Xf

_t18, _t19 = int(np.argmax(_v18)), int(np.argmax(_v19))
_top10_19 = set(topk_ids(_v19, 10).tolist())
if _t18 == _t19:
    print(f"[smoke] residual-stream continuity OK: top mean|x| channel {_t18} "
          f"identical across dual->single boundary (blocks 18->19)")
elif _t18 in _top10_19:
    print(f"[smoke][note] block-18 top-1 (ch {_t18}) is in block-19 top-10 "
          f"(top-1 there: ch {_t19}) -- acceptable")
else:
    print(f"[smoke][WARN] block-18 top-1 (ch {_t18}) absent from block-19 top-10 "
          f"({_t19} leads there). Usually the residual stream carries the MA "
          f"channel across the boundary; if the pilot's DG-anchor check (P5) "
          f"also fails, suspect the [text|image] concat order in single blocks.")

import matplotlib.pyplot as plt
from PIL import Image
plt.figure(figsize=(4, 4))
plt.imshow(Image.open(_paths["png"])); plt.axis("off")
plt.title(f"smoke image ({CFG.smoke_steps} steps -- low quality is expected)")
plt.show()
print("SMOKE TEST PASSED -- safe to start Phase A (pilot).")


## I1. Intervention configuration
All parameters of the intervention experiment live in one frozen dataclass,
and its hash is written into every output. The condition list and the
scenario scope are left out of the hash so runs can be added later.

The next cell picks the channels, the onset block, the refresh range and the
control channel from the pilot data using rules fixed in advance. Probe
blocks and steps are set in the model-parameter cell. This cell also checks
that the channel recurrence outputs we build on are on Drive and builds the
scenario list.


In [ ]:
# =====================================================================
# I1a. Model-specific intervention parameters (FLUX.1-dev)
#
# Everything the shared intervention cells need to know about this model.
# The expected channel ids come from the channel recurrence run and its
# pilot. Channel 154 is dominant. Channel 1446 is the largest other channel
# at the band end and overtakes 154 near block 35. Channel 85 is the next
# largest at the band end (the additional grower). The decision cell
# recomputes these from the pilot data and stops on a mismatch unless the
# override flag is set. Probe blocks, probe steps and the head count are
# also fixed here, before any intervention run.
# =====================================================================
MODEL_FAMILY = "flux"        # branches the pipeline call and batch handling
EXPECTED_CH1 = 154           # dominant MA channel, f1 = 1.00 in the recurrence run
EXPECTED_CH2 = 1446          # catching-up channel, overtakes near block 35
EXPECTED_CH3 = 85            # additional grower, about 5 to 3432 over the band in the pilot
TRACK_CH3 = True             # cheap to track in every run, its own conditions are optional
EXTRA_CONDITIONS = ()        # e.g. ("zero_ch3", "zero_both") to probe the additional grower
REG_EXPECT_TOKEN = None      # FLUX register position varies across prompts and seeds
LAST_BLOCK = CFG.n_blocks_expected - 1   # 56
# Attention probe blocks are the analysis block, mid-band, the block where 1446
# overtakes 154 in the pilot, the band end, and two blocks after the band.
PROBE_BLOCKS = (22, 30, 35, 39, 45, 50)
PROBE_ROLES = ("analysis_block", "mid_band", "overtake", "band_end", "post_band", "late")
# The roles above give each probe block a model-independent label.
PROBE_STEP_ROLES = ("early", "tstar")   # matches PROBE_STEPS after resolving and sorting
PROBE_STEPS = ("tstar", 10)  # t* and one earlier step (0-indexed, 28 steps)
N_HEADS_EXPECTED = 24
print(f"[intv-params] family={MODEL_FAMILY} expected ch1={EXPECTED_CH1} "
      f"ch2={EXPECTED_CH2} ch3={EXPECTED_CH3} (tracked={TRACK_CH3}) | extra conditions "
      f"{list(EXTRA_CONDITIONS)} | blocks 0..{LAST_BLOCK} | band {CFG.band} | "
      f"probe blocks {PROBE_BLOCKS} steps {PROBE_STEPS}")


In [ ]:
# =====================================================================
# I1b. Intervention config
#
# The rules live here and are hashed into the outputs. The pilot only
# supplies the evidence they are applied to (next cell).
#
# Condition names, parsed by make_conditions in I4:
#   baseline                     no intervention
#   zero_ch2 / nogrow_ch2        channel operators on the catching-up channel
#   freeze_ch2                   the block's write to ch2 removed (optional)
#   zero_ctrl                    same operator on the matched control channel
#   nogrow_others                every channel except ch1 and ch2 clamped (optional)
#   zero_ch3 / zero_both         probes of the additional grower (optional)
#   growth_x<g>[_early]          the block's write to ch2 scaled by gamma g
#                                (0 = freeze, 1 = baseline, 2 = doubled growth).
#                                "_early" starts at the band onset, not L_grow
#   dir_refresh                  register token reprojected onto v* at each block
#                                of the refresh range, norm unchanged
#   dir_refresh_random           same operator on a random ordinary token
#                                (position control)
#   norm_refresh                 register norm floored at its baseline peak,
#                                direction unchanged (magnitude control)
#   sham                         register token written back unchanged
# =====================================================================
from dataclasses import dataclass, asdict, field

@dataclass(frozen=True)
class InterventionConfig:
    # -- catch-up onset rule (applied to pilot medians at t*) -------------
    growth_onset_ratio: float = 0.2       # ch2/ch1 ratio that defines "catching up"
    growth_onset_consecutive: int = 2     # ratio must hold for this many consecutive blocks
    catch_ratio: float = 0.8              # reported only, where ch2 reaches 80% of ch1
    # -- intervention extent ----------------------------------------------
    intervene_end: str = "last"           # "last" runs through the final block, "band_end" stops at the band end
    intervene_steps: str = "all"          # every denoising step
    refresh_start: str = "l_grow"         # first block of the token operators ("l_grow", "band_end" or an int)
    # -- conditions ---------------------------------------------------------
    # Run in this order, baseline first. EXTRA_CONDITIONS comes from the
    # model-parameter cell.
    conditions: tuple = ("baseline", "zero_ch2", "nogrow_ch2", "zero_ctrl",
                         "growth_x2", "dir_refresh", "norm_refresh", "sham") + EXTRA_CONDITIONS
    # -- scenario scope -----------------------------------------------------
    scenario_scope: str = "full"          # "full" (100 = 20 prompts x 5 seeds) | "prompts20" (20 x seed0) | "pilot" (5)
    # -- register tracking --------------------------------------------------
    reg_ref_block: str = "lstar"          # register = baseline argmax-norm token at this block
    reg_tau_primary: float = 4.0          # primary persistence threshold (x median token norm)
    reg_tau_list: tuple = (2.0, 4.0, 8.0) # robustness thresholds
    norm_ref_rule: str = "baseline_peak"  # per step, max over blocks l*..band end of the baseline's largest token norm
    # -- v* (register direction) ---------------------------------------------
    vstar_source: str = "v4_main_tensors" # fit from the recurrence run's full tensors at (l*, t*)
    vstar_norm_mult: float = 3.0          # high-norm token has norm >= mult x median
    vstar_max_per_scenario: int = 8       # at most the eight largest per scenario
    vstar_min_scenarios: int = 20         # refuse to fit from fewer scenarios
    # -- geometry and attention readouts ---------------------------------------
    geom_cos_thr: float = 0.9             # a token counts as aligned if cos(x, v*) >= thr (secondary endpoint)
    geom_cos_list: tuple = (0.8, 0.9, 0.95)
    store_reg_vec: bool = True            # register vector at every (block, step) in fp16, blocks x steps x D x 2 bytes per run
    probe_chunk: int = 256                # query chunk for the attention probe softmax
    random_token_seed: int = 7            # seed offset for the position-control token
    # -- control-channel rule ----------------------------------------------
    ctrl_exclude_mult: float = 10.0       # ctrl must be below this multiple of the median channel
    # -- expected channel ids -------------------------------------------------
    expected_ch1: int = EXPECTED_CH1
    expected_ch2: int = EXPECTED_CH2
    allow_ch_mismatch: bool = False       # set True only after checking the printed tables
    force_redecide: bool = False          # rewrite intervention_decision_v3.json (invalidates finished runs)
    force_refit_vstar: bool = False       # rewrite vstar_v3.npz (invalidates finished runs)

INTV = InterventionConfig()
assert INTV.conditions[0] == "baseline", "baseline must be the first condition"
assert len(set(INTV.conditions)) == len(INTV.conditions), "duplicate condition names"
# The run hash covers every field that changes what one run computes or
# records. The condition list and scenario scope are left out, so adding a
# condition or widening the scope later only adds runs. Each run stores its
# own condition definition, and the audit re-checks it.
_HASH_EXCLUDE = {"conditions", "scenario_scope", "allow_ch_mismatch",
                 "force_redecide", "force_refit_vstar"}
INTV_HASH = hashlib.sha256(json.dumps(
    {k: v for k, v in asdict(INTV).items() if k not in _HASH_EXCLUDE},
    sort_keys=True, default=str).encode()).hexdigest()[:12]

# ------------------------------------------------- output tree (under BASE)
INT_PATHS = {
    "runs":     os.path.join(BASE, "intervention_v3"),
    "smoke":    os.path.join(BASE, "intervention_v3_smoke"),
    "figs":     os.path.join(BASE, "figures_intervention_v3"),
    "analysis": os.path.join(BASE, "analysis_intervention_v3"),
}
for _p in INT_PATHS.values():
    os.makedirs(_p, exist_ok=True)
INT_DEC_PATH = os.path.join(PATHS["meta"], "intervention_decision_v3.json")
VSTAR_PATH = os.path.join(PATHS["meta"], "vstar_v3.npz")

def int_fig_stem(name):
    return os.path.join(INT_PATHS["figs"], name)

# ------------------------------------------------- inputs from the channel recurrence run
_dec_path = os.path.join(PATHS["meta"], "pilot_decision.json")
if not os.path.exists(_dec_path):
    raise RuntimeError(
        "pilot_decision.json missing. This notebook builds on the v4 "
        "recurrence run: complete its Phase A (S4) and decision cell (S5) "
        "on this Drive first.")
with open(_dec_path) as _f:
    PILOT_DECISION = json.load(_f)
L_STAR = int(PILOT_DECISION["l_star"])
T_STAR = int(PILOT_DECISION["t_star"])
PRIMARY_STAT = PILOT_DECISION["primary_stat"]
PILOT_SIDS = [f"pilot__{scenario_id(p, s)}" for p, s in CFG.pilot_pairs]
for _sid in PILOT_SIDS:
    if not os.path.exists(os.path.join(PATHS["pilot"], f"{_sid}.npz")):
        raise RuntimeError(f"pilot capture missing: {_sid}.npz. Run the v4 "
                           f"notebook's Phase A on this Drive first.")

REG_REF_BLOCK = L_STAR if INTV.reg_ref_block == "lstar" else int(INTV.reg_ref_block)
N_TXT_IN_SEQ = int(CFG.max_seq_len) if MODEL_FAMILY == "flux" else int(CFG.n_txt_in_seq)
EXP_BATCH = 1 if MODEL_FAMILY == "flux" else (
    2 if (CFG.guidance_mode == "cfg_batch2" and CFG.guidance_scale > 1.0) else 1)
SELECT_ROW = 0 if MODEL_FAMILY == "flux" else int(CFG.cond_row)
PROBE_STEPS_RESOLVED = tuple(sorted({T_STAR if s == "tstar" else int(s) for s in PROBE_STEPS}))
assert all(0 <= s < CFG.num_steps for s in PROBE_STEPS_RESOLVED), PROBE_STEPS_RESOLVED
assert all(0 <= b <= LAST_BLOCK for b in PROBE_BLOCKS), PROBE_BLOCKS
assert all(b >= REG_REF_BLOCK for b in PROBE_BLOCKS), \
    "probe blocks must not precede the register reference block (baseline identity is unknown before it)"

# The attention probe wraps torch's scaled_dot_product_attention, so ask
# diffusers for the native backend where that API exists.
if hasattr(pipe.transformer, "set_attention_backend"):
    try:
        pipe.transformer.set_attention_backend("native")
        print("[intv-config] attention backend set to native (probe-compatible)")
    except Exception as _e:
        print(f"[intv-config][warn] could not set attention backend: {_e}")

# ------------------------------------------------- scenario scope
DEEP_PAIRS = {(p, s) for p, s in CFG.pilot_pairs}
if INTV.scenario_scope == "pilot":
    INTV_SCENARIOS = [(scenario_id(p, s), p, s, PROMPTS[p]) for p, s in CFG.pilot_pairs]
elif INTV.scenario_scope == "prompts20":
    _s0 = CFG.seeds[0]
    INTV_SCENARIOS = [(scenario_id(p, _s0), p, _s0, PROMPTS[p]) for p in sorted(PROMPTS)]
elif INTV.scenario_scope == "full":
    INTV_SCENARIOS = list(SCENARIOS)
else:
    raise ValueError(f"unknown scenario_scope {INTV.scenario_scope!r}")
N_COND = len(INTV.conditions)
_n_deep = sum(1 for _, p, s, _t in INTV_SCENARIOS if (p, s) in DEEP_PAIRS)

_k_tr_est = 3 + (1 if TRACK_CH3 else 0)
_L, _T, _D = CFG.n_blocks_expected, CFG.num_steps, CFG.d_model
_light_mb = (_L * _T * N_IMG * 2                                   # tn_all fp16
             + _L * _T * (5 * _k_tr_est + 30 + 2 * (_k_tr_est + 1) + 12 + 2) * 4
             + (_L * _T * _D * 2 if INTV.store_reg_vec else 0)      # reg_vec fp16
             + _L * _D * 2                                          # amax_vec at t*
             + len(PROBE_BLOCKS) * len(PROBE_STEPS_RESOLVED) * N_HEADS_EXPECTED * (N_IMG * 2 + 7 * 4)
             ) / 2**20
_deep_mb = _light_mb + (_L * _T * _D * 5 * 4 + _k_tr_est * _L * _T * N_IMG * 2) / 2**20
print(f"[intv-config] hash {INTV_HASH} | conditions {list(INTV.conditions)}")
print(f"[intv-config] scope={INTV.scenario_scope}: {len(INTV_SCENARIOS)} scenarios "
      f"x {N_COND} conditions = {len(INTV_SCENARIOS) * N_COND} generations "
      f"({_n_deep * N_COND} deep)")
print(f"[intv-config] pilot decision: l*={L_STAR} t*={T_STAR} primary={PRIMARY_STAT} "
      f"| register reference block {REG_REF_BLOCK} | probe steps {PROBE_STEPS_RESOLVED}")
print(f"[intv-config] per-run storage: light ~{_light_mb:.0f} MB, deep ~{_deep_mb:.0f} MB "
      f"-> total ~{((len(INTV_SCENARIOS) * N_COND - _n_deep * N_COND) * _light_mb + _n_deep * N_COND * _deep_mb) / 1024:.1f} GiB")
for _k, _v in asdict(INTV).items():
    print(f"    {_k:26s} = {_v}")

# ------------------------------------------------- comparability manifest
# Model facts that differ between architectures (block count, steps,
# guidance, probe positions), with blocks and steps also given as relative
# depth and relative denoising time, plus the rules in plain words. Saved with the analysis
# outputs and copied into the final manifest.
assert len(PROBE_ROLES) == len(PROBE_BLOCKS), "one role per probe block"
_nb, _nt = int(CFG.n_blocks_expected), int(CFG.num_steps)
COMPARABILITY = {
    "model_family": MODEL_FAMILY, "model_id": CFG.model_id,
    "n_blocks": _nb, "n_steps": _nt, "d_model": int(CFG.d_model), "n_heads": int(N_HEADS_EXPECTED),
    "n_img_tokens": int(N_IMG), "text_tokens_in_sequence": int(N_TXT_IN_SEQ),
    "guidance": ("embedded guidance, batch 1" if MODEL_FAMILY == "flux"
                 else f"classifier-free guidance, batch {EXP_BATCH}, conditional row {SELECT_ROW} measured, operators applied to all rows"),
    "precision_mode": PRECISION_MODE,
    "analysis_block_lstar": L_STAR, "lstar_rel": round(L_STAR / (_nb - 1), 3),
    "analysis_step_tstar": T_STAR, "tstar_rel": round(T_STAR / (_nt - 1), 3),
    "band": [int(CFG.band[0]), int(CFG.band[1])],
    "band_rel": [round(int(CFG.band[0]) / (_nb - 1), 3), round(int(CFG.band[1]) / (_nb - 1), 3)],
    "probe_blocks": [int(b) for b in PROBE_BLOCKS], "probe_roles": list(PROBE_ROLES),
    "probe_blocks_rel": [round(int(b) / (_nb - 1), 3) for b in PROBE_BLOCKS],
    "probe_steps": [int(t) for t in PROBE_STEPS_RESOLVED],
    "probe_steps_rel": [round(int(t) / (_nt - 1), 3) for t in PROBE_STEPS_RESOLVED],
    "register_identity": "baseline argmax-norm image token at l*, per denoising step"
                         + (f"; expected at token {REG_EXPECT_TOKEN}" if REG_EXPECT_TOKEN is not None else "; position varies"),
    "rules_identical_across_models": {
        "ch1": "largest channel by mean |x| at l* (pilot median, t*)",
        "ch2": "largest non-ch1 channel at the band end",
        "onset": f"first band block with ch2/ch1 >= {INTV.growth_onset_ratio:g} for {INTV.growth_onset_consecutive} consecutive blocks",
        "control": f"largest channel at the onset block below {INTV.ctrl_exclude_mult:g}x the median channel, not tracked",
        "channel_ops_range": f"onset .. {INTV.intervene_end} block, every step",
        "token_ops_range": f"{INTV.refresh_start} .. {INTV.intervene_end} block, every step",
        "vstar": f"top right-singular vector of unit-normalized tokens with norm >= {INTV.vstar_norm_mult:g}x median (<= {INTV.vstar_max_per_scenario} per scenario) at (l*, t*)",
        "primary_endpoint": f"persistence depth, tau = {INTV.reg_tau_primary:g}, step t*",
        "alignment_endpoint": f"deepest block with cos(register, v*) >= {INTV.geom_cos_thr:g}",
        "norm_reference": INTV.norm_ref_rule,
        "conditions": list(INTV.conditions),
        "statistics": "paired per scenario; sign test, Wilcoxon, prompt-level sign test, bootstrap CI, Holm; equivalence margin 1 block",
    },
    "blocks_per_relative_unit": _nb - 1,
    "note": ("Depth shifts are reported in blocks and in relative depth (blocks / (n_blocks - 1)); "
             "one PixArt block spans about twice the relative depth of one FLUX block, and the "
             "one-block equivalence margin is the endpoint resolution, not a matched effect size."),
}
save_json_atomic(COMPARABILITY, os.path.join(INT_PATHS["analysis"], "comparability.json"))
print("[comparability] " + " | ".join(
    f"{k}={COMPARABILITY[k]}" for k in ("n_blocks", "n_steps", "lstar_rel", "tstar_rel", "band_rel",
                                        "probe_blocks_rel", "probe_steps_rel", "guidance")))

# ------------------------------------------------- figure finishing
# save_paper_fig finishes every intervention figure. It can add a legend
# strip above the panels, then puts the title above all drawn content with
# a fixed gap so it can be cropped off. Each figure is saved twice, as
# <stem> with the title and <stem>_notitle without, each as PDF and 600 dpi
# PNG, plus a caption draft. The title is wrapped to the content width,
# which is measured from the drawn extents.
import textwrap as _textwrap

TITLE_PT = _BASE_PT + 0.5          # points
TITLE_GAP_PT = 8.0                 # gap between the top of the content and the title
LEGEND_PT = _BASE_PT - 1.5

def _artist_boxes(fig):
    """Display-coordinate bounding boxes of everything drawn in the figure.
    The pieces are gathered one by one because axes.get_tightbbox misses a
    y-label that is taller than its axes."""
    renderer = fig.canvas.get_renderer()
    boxes = []
    def _add(art, bb=None):
        try:
            if bb is None:
                if art is None or not art.get_visible():
                    return
                if hasattr(art, "get_text") and not str(art.get_text()).strip():
                    return                       # empty texts (unlabelled minor ticks) can sit anywhere
                bb = art.get_window_extent(renderer)
            if bb is not None and bb.width > 0 and bb.height > 0 and np.isfinite([bb.x0, bb.x1, bb.y0, bb.y1]).all():
                boxes.append(bb)
        except Exception:
            pass
    for ax in fig.axes:
        if not ax.get_visible():
            continue
        _add(ax)
        for axis in (ax.xaxis, ax.yaxis):
            try:
                _add(None, axis.get_tightbbox(renderer))   # visible ticks, tick labels and the axis label
            except Exception:
                pass
        for art in (ax.title, getattr(ax, "_left_title", None), getattr(ax, "_right_title", None),
                    ax.xaxis.label, ax.yaxis.label, ax.get_legend()):
            _add(art)
        for t in list(ax.texts):
            _add(t)
    for leg in fig.legends:
        _add(leg)
    for t in fig.texts:
        _add(t)
    return boxes

def _content_extent_display(fig):
    boxes = _artist_boxes(fig)
    if not boxes:
        return fig.bbox.x0, fig.bbox.x1, fig.bbox.y1
    return (min(b.x0 for b in boxes), max(b.x1 for b in boxes), max(b.y1 for b in boxes))

def add_fig_legend(fig, ax, ncol=4, title=None):
    """Legend strip above the panels, built from one axes' handles."""
    h, l = ax.get_legend_handles_labels()
    if not h:
        return None
    return fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, 1.0),
                      ncol=ncol, frameon=False, fontsize=LEGEND_PT,
                      columnspacing=1.0, handletextpad=0.4, title=title,
                      title_fontsize=LEGEND_PT)

def add_fig_title(fig, title, gap_pt=TITLE_GAP_PT, fontsize=TITLE_PT):
    """Title above all content, legend and panel titles included. It is
    centred on the content, wrapped to be no wider than it, and placed a
    fixed gap above it so it can be cropped off."""
    fig.canvas.draw()
    x0, x1, top = _content_extent_display(fig)
    inv = fig.transFigure.inverted()
    cx = inv.transform((0.5 * (x0 + x1), 0.0))[0]
    top_fig = inv.transform((0.0, top))[1]
    gap_fig = (gap_pt / 72.0) / fig.get_figheight()
    max_chars = max(24, int((x1 - x0) / fig.dpi * 72.0 / (0.55 * fontsize)))
    txt = None
    for _ in range(6):                       # shrink the wrap width until the title fits
        if txt is not None:
            txt.remove()
        text = "\n".join(_textwrap.wrap(title, width=max_chars))
        txt = fig.text(cx, top_fig + gap_fig, text, ha="center", va="bottom",
                       fontsize=fontsize, fontweight="semibold", linespacing=1.15)
        fig.canvas.draw()
        bb = txt.get_window_extent(fig.canvas.get_renderer())
        if bb.width <= (x1 - x0) * 1.005 or max_chars <= 24:
            break
        max_chars = max(24, int(max_chars * min(0.92, (x1 - x0) / bb.width)))
    return txt

def save_paper_fig(fig, stem, title, caption=None, legend_ax=None, ncol=4,
                   legend_title=None, show=True):
    """Finish and save one figure with its legend strip, once without and
    once with the title. The caption draft is written once, for <stem>."""
    if legend_ax is not None:
        add_fig_legend(fig, legend_ax, ncol=ncol, title=legend_title)
    save_fig(fig, stem + "_notitle", caption=None, show=False)
    add_fig_title(fig, title)
    save_fig(fig, stem, caption=caption, show=show)

def panel_grid(n, max_cols=4, panel_w=None, panel_h=1.9, sharey=True):
    """Page-wide grid for n panels, at most max_cols per row, unused cells
    hidden. Returns (fig, list of n axes)."""
    ncols = min(max_cols, n)
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(FIG_PAGE, panel_h * nrows),
                             sharex=True, sharey=sharey, squeeze=False)
    flat = list(axes.flat)
    for ax in flat[n:]:
        ax.set_visible(False)
    return fig, flat[:n]

def wrap_label(text, width=22):
    """Wrap a panel title or tick label onto a few short lines."""
    return "\n".join(_textwrap.wrap(str(text), width=width))

def grid_labels(axs, max_cols, xlabel, ylabel, fontsize=None):
    """Put the x-label on the lowest visible panel of each column and the
    y-label on the first column only. For panel_grid layouts with shared
    axes."""
    fs = LEGEND_PT if fontsize is None else fontsize
    n = len(axs); ncols = min(max_cols, n)
    for col in range(ncols):
        col_axes = [axs[i] for i in range(col, n, ncols)]
        col_axes[-1].set_xlabel(xlabel, fontsize=fs)
        col_axes[-1].tick_params(labelbottom=True)
        for ax in col_axes[:-1]:
            ax.tick_params(labelbottom=False)
    for i in range(0, n, ncols):
        axs[i].set_ylabel(ylabel, fontsize=fs)

def log_floor(values, frac=1e-3, at_least=1e-2):
    """Floor for log-axis curves, a fixed fraction of the smallest positive
    finite value. Exact zeros from zeroed channels then sit on the axis floor
    instead of stretching the axis down to the numerical floor."""
    v = np.asarray(values, np.float64)
    v = v[np.isfinite(v) & (v > 0)]
    return max(at_least, float(v.min()) * frac) if v.size else at_least

print(f"[fig-finish] titled + title-free saves, title gap {TITLE_GAP_PT:g} pt, title {TITLE_PT:g} pt")


## I2. Catch-up decision from the pilot
Rules fixed in advance are applied to the pilot capture (five scenarios, all
blocks, analysis step t\*), and the result is frozen to
`meta/intervention_decision_v3.json`.

The dominant channel ch1 is the largest channel at the analysis block. The
catching-up channel ch2 is the largest remaining channel at the band end.
The catch-up onset is the first band block where ch2 reaches a set fraction
of ch1 for a set number of consecutive blocks. The control channel is the
largest channel at the onset block that is neither tracked nor an outlier.
By default the token operators start at the same onset block, so every
condition acts on the same range, and run through the last block. The cell
stops if the computed channels differ from the expected ones in the
model-parameter cell, which come from the channel recurrence run, unless the
override flag is set.


In [ ]:
# =====================================================================
# I2. Catch-up decision from the pilot capture, frozen to Drive
#
# ch2, the catching-up channel, is the channel with the largest median
# mean|x| at the band end once the dominant channel is left out. This rule
# encodes the effect under test (a second channel at the dominant channel's
# order of magnitude when the register dissolves), and it gives the same
# channels as the channel recurrence run. The ranking by growth over the
# band is saved next to it as evidence. With TRACK_CH3 set, ch3 is the next
# channel at the band end (the additional grower). It is tracked in every
# run, and its own conditions are switched on through EXTRA_CONDITIONS.
#
# The refresh range of the token operators (INTV.refresh_start) is decided
# here too. The condition list is not part of the decision hash, and each
# run records its own condition definition.
# =====================================================================
_zs = {s: np.load(os.path.join(PATHS["pilot"], f"{s}.npz"), allow_pickle=False)
       for s in PILOT_SIDS}
_nb = CFG.n_blocks_expected
MED_PROF = np.zeros((_nb, CFG.d_model), np.float64)   # mean|x| at t*, median over pilots
for _b in range(_nb):
    MED_PROF[_b] = np.median(np.stack(
        [_z[f"b{_b:02d}__mean_abs"][T_STAR].astype(np.float64)
         for _z in _zs.values()]), axis=0)
for _z in _zs.values():
    _z.close()

_lo, _hi = int(CFG.band[0]), int(CFG.band[1])

# ---- ch1 is the dominant channel at the analysis block
CH1 = int(np.argmax(MED_PROF[L_STAR]))

# ---- ch2 (and ch3 if tracked) by band-end rank, ch1 left out
_end_vals = MED_PROF[_hi].copy()
_end_vals[CH1] = -np.inf
_end_order = np.argsort(-_end_vals)
CH2 = int(_end_order[0])
HAS_CH3 = bool(TRACK_CH3)
CH3 = int(_end_order[1]) if HAS_CH3 else None

# ---- both rankings go into the decision file as evidence
_growth = MED_PROF[_hi] - MED_PROF[_lo]
_growth[CH1] = -np.inf
_delta_order = np.argsort(-_growth)

def _grower_row(c):
    v0, v1 = float(MED_PROF[_lo][c]), float(MED_PROF[_hi][c])
    return {"channel": int(c), "at_onset": round(v0, 3), "at_end": round(v1, 3),
            "delta": round(v1 - v0, 3),
            "ratio_end_over_onset": round(v1 / max(v0, 1e-9), 2)}

GROWERS_BY_END = [_grower_row(c) for c in _end_order[:10]]
GROWERS_BY_DELTA = [_grower_row(c) for c in _delta_order[:10]]

print(f"[intv-decision] band-end ranking, ch1={CH1} excluded "
      f"(mean|x| at t*={T_STAR}, block {_lo} -> {_hi}):")
for _r in GROWERS_BY_END[:5]:
    print(f"    ch {_r['channel']:4d}: {_r['at_onset']:10.3f} -> {_r['at_end']:10.3f} "
          f"(delta {_r['delta']:+.3f}, x{_r['ratio_end_over_onset']:g})")
print(f"[intv-decision] growth-delta ranking over the band (evidence, ch1 excluded):")
for _r in GROWERS_BY_DELTA[:5]:
    print(f"    ch {_r['channel']:4d}: {_r['at_onset']:10.3f} -> {_r['at_end']:10.3f} "
          f"(delta {_r['delta']:+.3f}, x{_r['ratio_end_over_onset']:g})")
for _nm, _c in (("ch2", CH2),) + ((("ch3", CH3),) if HAS_CH3 else ()):
    _rr = MED_PROF[_hi][_c] / max(MED_PROF[_lo][_c], 1e-9)
    if _rr < 2.0:
        print(f"[intv-decision][WARN] {_nm}={_c} grew only x{_rr:.2f} over the "
              f"band; the catch-up framing may not apply -- inspect the tables.")

# ---- check the channels against the expected ids
_mismatch = []
if CH1 != int(INTV.expected_ch1):
    _mismatch.append(f"ch1 computed {CH1}, expected {INTV.expected_ch1}")
if CH2 != int(INTV.expected_ch2):
    _mismatch.append(f"ch2 computed {CH2}, expected {INTV.expected_ch2}")
if HAS_CH3 and EXPECTED_CH3 is not None and CH3 != int(EXPECTED_CH3):
    _mismatch.append(f"ch3 computed {CH3}, expected {EXPECTED_CH3}")
if _mismatch and not INTV.allow_ch_mismatch:
    raise RuntimeError(
        "channel-identity tripwire: " + "; ".join(_mismatch) + ". Inspect the "
        "printed ranking tables and the v4 D1/D2 outputs; if the computed "
        "identities are right, set allow_ch_mismatch=True and re-run.")
elif _mismatch:
    print(f"[intv-decision][OVERRIDE] {_mismatch} -- proceeding with computed identities.")

# ---- catch-up onset from the ch2/ch1 ratio (only full windows inside the band)
R_CURVE = MED_PROF[:, CH2] / np.maximum(MED_PROF[:, CH1], 1e-12)
L_GROW = None
_need = int(INTV.growth_onset_consecutive)
for _l in range(_lo, _hi + 2 - _need):
    if (R_CURVE[_l:_l + _need] >= INTV.growth_onset_ratio).all():
        L_GROW = int(_l)
        break
if L_GROW is None:
    _tab = {int(l): round(float(R_CURVE[l]), 4) for l in range(_lo, _hi + 1)}
    raise RuntimeError(
        f"no catch-up onset found: ch2/ch1 never reaches "
        f"{INTV.growth_onset_ratio} for {_need} consecutive band blocks "
        f"(full windows inside the band only). Ratio table: {_tab}")
L_CATCH = next((int(l) for l in range(_lo, _hi + 1)
                if R_CURVE[l] >= INTV.catch_ratio), None)
L_END = LAST_BLOCK if INTV.intervene_end == "last" else _hi

# ---- refresh range for the token operators
if INTV.refresh_start == "l_grow":
    L_REFRESH = L_GROW
elif INTV.refresh_start == "band_end":
    L_REFRESH = _hi
else:
    L_REFRESH = int(INTV.refresh_start)
assert _lo <= L_REFRESH <= L_END, \
    f"refresh start {L_REFRESH} must lie in the band onset .. last block range [{_lo}, {L_END}]"
if L_REFRESH < REG_REF_BLOCK:
    print(f"[intv-decision][note] token operators start at block {L_REFRESH}, before the register "
          f"reference block {REG_REF_BLOCK}; they act on the baseline's register token identity "
          f"(defined at block {REG_REF_BLOCK}), which already exists from the band onset {_lo}.")

# ---- control channel is the largest non-outlier channel at the onset block
_v = MED_PROF[L_GROW].copy()
_med_scalar = max(float(np.median(_v)), 1e-12)
_eligible = _v < INTV.ctrl_exclude_mult * _med_scalar
_excl_ids = [CH1, CH2] + ([CH3] if HAS_CH3 else [])
_eligible[_excl_ids] = False
if not _eligible.any():
    raise RuntimeError("no eligible control channel below the outlier threshold")
_v_masked = np.where(_eligible, _v, -np.inf)
CH_CTRL = int(np.argmax(_v_masked))

_dec_core = {
    "ch1": CH1, "ch2": CH2, "ch_ctrl": CH_CTRL,
    "l_grow": L_GROW, "l_catch": L_CATCH, "l_end": L_END, "l_refresh": L_REFRESH,
    "band": [_lo, _hi], "l_star": L_STAR, "t_star": T_STAR,
    "reg_ref_block": int(REG_REF_BLOCK),
    "probe_blocks": [int(b) for b in PROBE_BLOCKS],
    "probe_steps": [int(s) for s in PROBE_STEPS_RESOLVED],
    "ratio_at_onset": round(float(R_CURVE[L_GROW]), 4),
    "ch1_at_onset": round(float(MED_PROF[L_GROW, CH1]), 4),
    "ch2_at_onset": round(float(MED_PROF[L_GROW, CH2]), 4),
    "ctrl_at_onset": round(float(MED_PROF[L_GROW, CH_CTRL]), 4),
    "ctrl_over_ch2_at_onset": round(float(MED_PROF[L_GROW, CH_CTRL]
                                          / max(MED_PROF[L_GROW, CH2], 1e-12)), 4),
}
if HAS_CH3:
    _dec_core["ch3"] = CH3
    _dec_core["ch3_at_onset"] = round(float(MED_PROF[L_GROW, CH3]), 4)
INT_DEC_SHA = sha256_of_text(json.dumps(_dec_core, sort_keys=True))

# ---- write the decision file, or check the recomputation against it
if os.path.exists(INT_DEC_PATH) and not INTV.force_redecide:
    with open(INT_DEC_PATH) as _f:
        _prev = json.load(_f)
    if _prev.get("dec_sha") != INT_DEC_SHA:
        raise RuntimeError(
            f"{os.path.basename(INT_DEC_PATH)} exists but disagrees with the "
            f"recomputed decision (stored {_prev.get('dec_sha')}, recomputed "
            f"{INT_DEC_SHA}). Either the config or the pilot data changed. "
            f"Set force_redecide=True to overwrite; finished runs are then "
            f"invalidated and redone.")
    INT_DEC = _prev
    print(f"[intv-decision] decision file verified against recomputation (sha {INT_DEC_SHA})")
else:
    INT_DEC = dict(_dec_core)
    INT_DEC.update({
        "dec_sha": INT_DEC_SHA,
        "rules": {k: getattr(INTV, k) for k in
                  ("growth_onset_ratio", "growth_onset_consecutive", "catch_ratio",
                   "intervene_end", "refresh_start", "ctrl_exclude_mult")},
        "ch2_rule": ("largest median mean|x| at the band end, ch1 excluded "
                     "(catching-up channel; band-end rank rule)"),
        "ch3_rule": ("next-largest median mean|x| at the band end (additional "
                     "grower, tracked)" if HAS_CH3 else "not configured"),
        "onset_rule": (f"first band block where ch2/ch1 >= "
                       f"{INTV.growth_onset_ratio:g} for "
                       f"{int(INTV.growth_onset_consecutive)} consecutive blocks "
                       f"(full windows inside the band only)"),
        "refresh_rule": f"token operators act on blocks {L_REFRESH}..{L_END} ({INTV.refresh_start})",
        "growers_by_end": GROWERS_BY_END,
        "growers_by_delta": GROWERS_BY_DELTA,
        "r_curve": [round(float(r), 5) for r in R_CURVE],
        "ch1_curve": [round(float(x), 4) for x in MED_PROF[:, CH1]],
        "ch2_curve": [round(float(x), 4) for x in MED_PROF[:, CH2]],
        "ctrl_curve": [round(float(x), 4) for x in MED_PROF[:, CH_CTRL]],
        "pilot_sids": PILOT_SIDS, "pilot_config_hash": PILOT_DECISION["config_hash"],
        "precision_mode": PRECISION_MODE, "config_hash": CONFIG_HASH,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds")})
    if HAS_CH3:
        INT_DEC["ch3_curve"] = [round(float(x), 4) for x in MED_PROF[:, CH3]]
    save_json_atomic(INT_DEC, INT_DEC_PATH)
    print(f"[intv-decision] decision frozen -> {INT_DEC_PATH} (sha {INT_DEC_SHA})")

# ---- decision figure with channel trajectories and the ratio rule
fig, axes = plt.subplots(1, 2, figsize=(FIG_PAGE, 2.5))
_x = np.arange(_nb)
for _ax in axes:
    _ax.axvspan(_lo, _hi, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band" if _ax is axes[0] else None)
    _ax.axvline(L_GROW, color="0.25", ls=":", lw=0.9, label=f"catch-up onset (block {L_GROW})" if _ax is axes[0] else None)
    _ax.axvline(L_REFRESH, color="0.25", ls="-.", lw=0.7, label=f"token-operator onset (block {L_REFRESH})" if _ax is axes[0] else None)
axes[0].semilogy(_x, MED_PROF[:, CH1], color=OKABE_ITO[0], lw=1.2, label=f"dominant ch. {CH1}")
axes[0].semilogy(_x, MED_PROF[:, CH2], color=OKABE_ITO[3], lw=1.2, label=f"catching-up ch. {CH2}")
if HAS_CH3:
    axes[0].semilogy(_x, MED_PROF[:, CH3], color=OKABE_ITO[4], lw=1.1, label=f"second grower ch. {CH3}")
axes[0].semilogy(_x, MED_PROF[:, CH_CTRL], color=OKABE_ITO[2], lw=1.0, ls="--", label=f"control ch. {CH_CTRL}")
axes[0].semilogy(_x, np.median(MED_PROF, axis=1), color="0.6", lw=0.8, label="median channel")
axes[0].set_xlabel("transformer block")
axes[0].set_ylabel("mean |activation| over image tokens")
axes[0].set_title("(a) channel magnitude across depth", fontsize=_BASE_PT, pad=3)
axes[1].plot(_x, R_CURVE, color=OKABE_ITO[3], lw=1.2, label=f"ch. {CH2} / ch. {CH1}")
if HAS_CH3:
    axes[1].plot(_x, MED_PROF[:, CH3] / np.maximum(MED_PROF[:, CH1], 1e-12),
                 color=OKABE_ITO[4], lw=0.9, label=f"ch. {CH3} / ch. {CH1}")
axes[1].axhline(INTV.growth_onset_ratio, color="0.4", ls="--", lw=0.8, label=f"onset threshold ({INTV.growth_onset_ratio:g})")
axes[1].axhline(INTV.catch_ratio, color="0.4", ls=":", lw=0.8, label=f"catch threshold ({INTV.catch_ratio:g})")
axes[1].set_xlabel("transformer block")
axes[1].set_ylabel(f"ratio to dominant ch. {CH1}")
axes[1].set_ylim(0, max(1.6, float(R_CURVE[_lo:].max()) * 1.05))   # y-range from the band on, the ratio means little before it
axes[1].set_title("(b) catch-up ratio", fontsize=_BASE_PT, pad=3)
axes[1].legend(loc="best", fontsize=LEGEND_PT, frameon=True, framealpha=0.9, edgecolor="none")
save_paper_fig(fig, int_fig_stem("I2_catchup_decision"), legend_ax=axes[0], ncol=4,
    title=(f"Channel catch-up in {MODEL_LABEL}: the catching-up channel {CH2} approaches the "
           f"dominant register channel {CH1} across depth (median of 5 pilot generations, step {T_STAR})"),
    caption=(
    f"Catch-up decision for {MODEL_LABEL}. (a) Per-block mean absolute activation over image tokens of "
    f"the dominant channel {CH1}, the catching-up channel {CH2} (largest non-dominant channel at the band end"
    + (f"), the second grower {CH3} (next-largest at the band end" if HAS_CH3 else "")
    + f"), the control channel {CH_CTRL}, and the median channel, at step {T_STAR} (median over the five "
    f"pilot scenarios; the shaded region is the high-norm band, the dotted line the catch-up onset block "
    f"{L_GROW}, the dash-dotted line the token-operator onset block {L_REFRESH}). (b) The ratio of each "
    f"tracked growing channel to channel {CH1}; dashed and dotted lines mark the onset threshold "
    f"{INTV.growth_onset_ratio:g} and the catch threshold {INTV.catch_ratio:g}. Channel interventions "
    f"act on blocks {L_GROW} to {L_END}; token interventions on blocks {L_REFRESH} to {L_END}."))

print()
print("=" * 70)
print("INTERVENTION DECISION SUMMARY (v3)")
print("=" * 70)
print(f"  ch1 (dominant)        = {CH1}")
print(f"  ch2 (catching-up)     = {CH2}   ratio at onset {INT_DEC['ratio_at_onset']}")
if HAS_CH3:
    print(f"  ch3 (extra grower)    = {CH3}   (mean|x| {INT_DEC['ch3_at_onset']} at onset; tracked only"
          + (", probed by " + ", ".join(EXTRA_CONDITIONS) if EXTRA_CONDITIONS else "") + ")")
print(f"  control channel       = {CH_CTRL}   (mean|x| {INT_DEC['ctrl_at_onset']}, "
      f"{INT_DEC['ctrl_over_ch2_at_onset']} of ch2 at onset)")
print(f"  catch-up onset L_grow = {L_GROW}   (80% catch block: {L_CATCH})")
print(f"  channel interventions = blocks {L_GROW}..{L_END}, all steps")
print(f"  token interventions   = blocks {L_REFRESH}..{L_END}, all steps")
print(f"  attention probes      = blocks {list(PROBE_BLOCKS)} at steps {list(PROBE_STEPS_RESOLVED)}")
print(f"  register reference    = baseline argmax-norm token at block {REG_REF_BLOCK}, per step")
print("=" * 70)


## I3. Intervention operators, readouts and unit tests
The channel operators edit the image-stream block output. `zero`, `freeze`
and `nogrow` do what their names say, and `growth` scales the block's write
to the channel by gamma. Since only the per-block write is scaled, the
effect doesn't compound across blocks. The token operators edit one token.
`dir_refresh` points it along v\* and keeps its norm, `norm_refresh` raises
its norm to a floor and keeps its direction, and `sham` writes it back
unchanged. Each operator re-checks its result in the model dtype.

The cell also defines the v\* fit, the geometry readout (alpha, ||x_perp||,
cosine and cosine statistics over all tokens) and the attention probe. The
probe grabs query and key at the scaled-dot-product-attention call inside a
marked block. Using image queries and image keys only, it computes each
key's rank and cosine against the mean image query and the attention the
key receives relative to a uniform share.

The unit tests run on small synthetic CPU tensors, including both attention
layouts, a planted v\* and a brute-force attention reference. They have to
pass before any model compute.


In [ ]:
# =====================================================================
# I3. Intervention operators and the attention probe
#
# Pure torch. Unit-tested on synthetic CPU tensors before any model compute.
#
# Channel operators edit chosen channels of every image token:
#   zero      y[.., c] = 0
#   freeze    y[.., c] = x_in[.., c]                 (block's write removed)
#   nogrow    |y[.., c]| <= |x_in[.., c]|            (can shrink, not grow)
#   growth    y[.., c] = x_in + gamma (x_out - x_in)
# growth scales only the block's own write to the channel. gamma = 0 is
# freeze, gamma = 1 is the identity, and gamma > 1 amplifies growth without
# compounding across blocks.
#
# Token operators edit one image token (all channels) in every batch row:
#   dir_refresh   y_r = ||x_r|| * v*                 (direction reset, norm kept)
#   norm_refresh  y_r = x_r * max(1, ref / ||x_r||)  (norm floored, direction kept)
#   sham          y_r = x_r                          (control for the hook itself)
#
# Every operator returns the modified clone and a violation count. The
# count re-checks the operator's own result in the model dtype, with a
# tolerance for bf16 rounding. It has to be zero and is recorded per
# (block, step) for the audit.
# =====================================================================
import math
import numpy as np
import torch

CHANNEL_MODES = ("zero", "freeze", "nogrow", "growth")
TOKEN_MODES = ("dir_refresh", "norm_refresh", "sham")

def resolve_channel_indices(spec, channels, exclude, d_model):
    """spec "list" gives `channels`, "all_except" gives every channel not in
    `exclude`, "none" gives an empty set. Returns a sorted int64 array."""
    if spec == "none":
        return np.zeros(0, np.int64)
    if spec == "list":
        idx = np.asarray(sorted(int(c) for c in channels), np.int64)
    elif spec == "all_except":
        mask = np.ones(int(d_model), bool)
        mask[[int(c) for c in exclude]] = False
        idx = np.where(mask)[0].astype(np.int64)
    else:
        raise ValueError(f"unknown channel spec {spec!r}")
    assert len(idx) and idx.min() >= 0 and idx.max() < d_model
    return idx

def _rel_tol(t, rel=1e-2, abs_=1e-3):
    """Tolerance for checks on model-dtype results (bf16 keeps about 3
    significant digits)."""
    return t.abs() * rel + abs_

def apply_channel_op(out_region, in_region, idx, mode, gamma=None):
    """out_region and in_region are [B, N, D]. in_region can be None for
    "zero". idx is a LongTensor of channel indices on the same device.
    Returns (modified clone, 0-dim violation count)."""
    y = out_region.clone()
    if mode == "zero":
        y[..., idx] = 0
        viol = (y[..., idx] != 0).sum()
    elif mode == "freeze":
        if in_region is None:
            raise RuntimeError("freeze needs the block-input image stream")
        y[..., idx] = in_region[..., idx]
        viol = (y[..., idx] != in_region[..., idx]).sum()
    elif mode == "nogrow":
        if in_region is None:
            raise RuntimeError("nogrow needs the block-input image stream")
        lim = in_region[..., idx].abs()
        y[..., idx] = torch.clamp(out_region[..., idx], min=-lim, max=lim)
        viol = (y[..., idx].abs() > lim * 1.001 + 1e-6).sum()
    elif mode == "growth":
        if in_region is None:
            raise RuntimeError("growth needs the block-input image stream")
        if gamma is None:
            raise RuntimeError("growth needs gamma")
        xi = in_region[..., idx].to(torch.float32)
        xo = out_region[..., idx].to(torch.float32)
        target = xi + float(gamma) * (xo - xi)
        y[..., idx] = target.to(y.dtype)
        viol = ((y[..., idx].to(torch.float32) - target).abs()
                > _rel_tol(target)).sum()
    else:
        raise ValueError(f"unknown channel mode {mode!r}")
    return y, viol

def apply_token_op(out_region, tok, mode, vstar=None, norm_ref=None):
    """out_region is [B, N, D] and tok an int token index. dir_refresh needs
    vstar, a [D] float32 unit vector. norm_refresh needs the float norm_ref.
    Acts on every batch row. Returns (modified clone, 0-dim violation
    count)."""
    y = out_region.clone()
    x = out_region[:, tok, :].to(torch.float32)                 # [B, D]
    n = x.norm(dim=-1, keepdim=True)                            # [B, 1]
    if mode == "dir_refresh":
        if vstar is None:
            raise RuntimeError("dir_refresh needs v*")
        v = vstar.to(torch.float32)
        z = n * v[None, :]
        y[:, tok, :] = z.to(y.dtype)
        zz = y[:, tok, :].to(torch.float32)
        cos = (zz @ v) / (zz.norm(dim=-1) * v.norm() + 1e-12)
        viol = ((cos < 1.0 - 1e-4).sum()
                + ((zz.norm(dim=-1, keepdim=True) - n).abs() > _rel_tol(n)).sum())
    elif mode == "norm_refresh":
        if norm_ref is None:
            raise RuntimeError("norm_refresh needs a reference norm")
        ref = float(norm_ref)
        scale = torch.clamp(ref / torch.clamp(n, min=1e-12), min=1.0)
        z = x * scale
        y[:, tok, :] = z.to(y.dtype)
        zz = y[:, tok, :].to(torch.float32)
        cos = (zz * x).sum(-1) / (zz.norm(dim=-1) * x.norm(dim=-1) + 1e-12)
        target_n = torch.maximum(n, torch.full_like(n, ref))
        viol = ((cos < 1.0 - 1e-4).sum()
                + ((zz.norm(dim=-1, keepdim=True) - target_n).abs()
                   > _rel_tol(target_n)).sum())
    elif mode == "sham":
        y[:, tok, :] = out_region[:, tok, :]
        viol = (y != out_region).sum()
    else:
        raise ValueError(f"unknown token mode {mode!r}")
    return y, viol

def modify_block_output(output, n_img, n_txt, fn):
    """Apply fn ([B, n_img, D] -> [B, n_img, D]) to the image part of a block
    output and rebuild the output. Returns (new_output, found). The image
    part is found by sequence length only. It can be a plain tensor of
    length n_img, a plain tensor of length n_txt + n_img with text first,
    or the length-n_img tensor inside a tuple or list (dual-stream blocks
    return a (text, image) pair). None entries and the text stream pass
    through untouched."""
    if torch.is_tensor(output) and output.dim() == 3:
        L = output.shape[1]
        if L == n_img:
            return fn(output), True
        if n_txt and L == n_img + n_txt:
            y = output.clone()
            y[:, n_txt:, :] = fn(output[:, n_txt:, :])
            return y, True
        return output, False
    if isinstance(output, (tuple, list)):
        out = list(output)
        for j, o in enumerate(out):
            if (o is not None and torch.is_tensor(o) and o.dim() == 3
                    and o.shape[1] == n_img):
                out[j] = fn(o)
                return (tuple(out) if isinstance(output, tuple) else out), True
        return output, False
    return output, False

def find_input_image_region(args, kwargs, n_img, n_txt):
    """Return the block-input image stream [B, n_img, D], or None. Looks in
    kwargs["hidden_states"] first, then in the positional args, since some
    pipelines pass hidden_states by keyword and others positionally. A
    concatenated input is sliced to its image part."""
    cands = []
    hs = kwargs.get("hidden_states", None) if kwargs else None
    if torch.is_tensor(hs) and hs.dim() == 3:
        cands.append(hs)
    cands += [a for a in (args or ()) if torch.is_tensor(a) and a.dim() == 3]
    for t in cands:
        if t.shape[1] == n_img:
            return t
        if n_txt and t.shape[1] == n_img + n_txt:
            return t[:, n_txt:, :]
    return None

# ---------------------------------------------------------------- v* fit
def fit_vstar(vectors, ch1=None):
    """vectors is an [M, D] array of register-token vectors at any scale.
    Returns (v, info) with v a [D] float32 unit vector. Each
    row is unit-normalized and v is the top right-singular vector of the
    stack without centering, flipped so the mean projection is positive."""
    U = np.asarray(vectors, np.float64)
    assert U.ndim == 2 and U.shape[0] >= 2, f"need >= 2 rows, got {U.shape}"
    nrm = np.linalg.norm(U, axis=1, keepdims=True)
    keep = nrm[:, 0] > 0
    U = U[keep] / nrm[keep]
    _, s, Vt = np.linalg.svd(U, full_matrices=False)
    v = Vt[0]
    if float((U @ v).mean()) < 0:
        v = -v
    info = {"n_rows": int(U.shape[0]),
            "top_singular_energy": float(s[0] ** 2 / max((s ** 2).sum(), 1e-12)),
            "mean_cos_rows": float((U @ v).mean()),
            "min_cos_rows": float((U @ v).min())}
    if ch1 is not None:
        info["ch1"] = int(ch1)
        info["energy_on_ch1"] = float(v[int(ch1)] ** 2)
        info["abs_cos_with_e_ch1"] = float(abs(v[int(ch1)]))
    return v.astype(np.float32), info

# ------------------------------------------------------- geometry readout
GEOM_FIELDS = ("alpha_reg", "cos_reg", "xperp_reg", "norm_reg",
               "alpha_amax", "cos_amax", "norm_amax",
               "med_cos_all", "p99_cos_all", "max_cos_all", "argmax_cos_tok",
               "n_cos_ge_thr")

def geometry_readout(X, vstar, reg_tok, amax_tok, thr=0.9):
    """X is [N, D] float32 for one batch row, vstar a [D] float32 unit vector
    on the same device, reg_tok an int or None, amax_tok an int. Returns a
    float32 tensor [len(GEOM_FIELDS)] on X.device, with NaN in the register
    fields when reg_tok is None."""
    proj = X @ vstar                                   # [N]  alpha per token
    norms = X.norm(dim=1)                              # [N]
    cos = proj / torch.clamp(norms, min=1e-12)         # [N]
    k99 = max(1, int(math.ceil(0.99 * X.shape[0])))
    nan = torch.tensor(float("nan"), device=X.device)
    if reg_tok is None:
        a_r, c_r, xp_r, n_r = nan, nan, nan, nan
    else:
        a_r = proj[reg_tok]
        n_r = norms[reg_tok]
        c_r = cos[reg_tok]
        # compute ||x_perp|| directly, sqrt(n^2 - alpha^2) cancels badly
        # for a token almost collinear with v*
        xp_r = (X[reg_tok] - a_r * vstar).norm()
    out = torch.stack([
        a_r, c_r, xp_r, n_r,
        proj[amax_tok], cos[amax_tok], norms[amax_tok],
        cos.median(), cos.kthvalue(k99).values, cos.max(),
        cos.argmax().to(torch.float32),
        (cos >= thr).sum().to(torch.float32)])
    return out.to(torch.float32)

# --------------------------------------------------------- attention probe
PROBE_HEAD_FIELDS = ("reg_key_rank", "reg_key_cos", "med_key_cos",
                     "reg_sink_rel", "top_sink_idx", "top_sink_rel",
                     "reg_is_top_sink")

class AttentionProbe:
    """Grabs query and key at the scaled_dot_product_attention call of the
    block currently marked active and computes per-head arrays over image
    queries and image keys only. Attention is renormalized
    over image keys, averaged over image queries and multiplied by N, so
    1 means a uniform share.
      s        [H, N]  dot product of each image key with the mean image query
      cos_k    [H, N]  cosine of each image key with the mean image query
      rel      [H, N]  attention received by each key, relative to uniform
    The block-output hook reads the register's values from these arrays
    once the register index is known.

    scaled_dot_product_attention gets [B, H, S, Dh]. diffusers permutes to
    this layout for every backend that ends in the native kernel. S is
    n_img for image-only self-attention or n_txt + n_img for joint
    attention with text first. Any other length, such as cross-attention
    over text keys, is ignored."""

    def __init__(self, n_img, n_txt, select_row=0, chunk=256):
        self.n_img, self.n_txt = int(n_img), int(n_txt)
        self.select_row, self.chunk = int(select_row), int(chunk)
        self.active = None          # block index currently being probed
        self.stash = None
        self.fires = 0
        self.ignored = 0
        self.errors = []
        self._orig = None

    # ---- patching
    def patch(self):
        import torch.nn.functional as F
        assert self._orig is None, "probe already patched"
        self._orig = F.scaled_dot_product_attention
        probe = self

        def wrapped(*args, **kwargs):
            out = probe._orig(*args, **kwargs)
            if probe.active is not None:
                try:
                    q = kwargs["query"] if "query" in kwargs else args[0]
                    k = kwargs["key"] if "key" in kwargs else args[1]
                    probe.record(q, k, kwargs.get("scale", None))
                except Exception as e:
                    probe.errors.append(f"probe block {probe.active}: "
                                        f"{type(e).__name__}: {e}")
            return out
        F.scaled_dot_product_attention = wrapped
        return self

    def unpatch(self):
        import torch.nn.functional as F
        if self._orig is not None:
            F.scaled_dot_product_attention = self._orig
            self._orig = None

    # ---- computation
    @torch.no_grad()
    def record(self, q, k, scale=None):
        if not (torch.is_tensor(q) and torch.is_tensor(k) and q.dim() == 4 and k.dim() == 4):
            self.ignored += 1
            return
        S = k.shape[2]
        if S == self.n_img:
            lo = 0
        elif self.n_txt and S == self.n_img + self.n_txt and q.shape[2] == S:
            lo = self.n_txt
        else:
            self.ignored += 1                # cross-attention or some other call
            return
        row = self.select_row if q.shape[0] > self.select_row else 0
        qi = q[row, :, lo:, :].to(torch.float32)        # [H, N, Dh]
        ki = k[row, :, lo:, :].to(torch.float32)        # [H, N, Dh]
        H, N, Dh = qi.shape
        sc = float(scale) if scale is not None else Dh ** -0.5
        qbar = qi.mean(dim=1)                           # [H, Dh]
        s = torch.einsum("hnd,hd->hn", ki, qbar)        # [H, N]
        cos_k = s / (ki.norm(dim=-1) * qbar.norm(dim=-1, keepdim=True) + 1e-12)
        rel = torch.zeros(H, N, dtype=torch.float32, device=qi.device)
        for a in range(0, N, self.chunk):
            logits = torch.einsum("hcd,hnd->hcn", qi[:, a:a + self.chunk, :], ki) * sc
            rel += torch.softmax(logits, dim=-1).sum(dim=1)   # sum over queries
        # rel = sum_q a[q, key] = N * mean_q a[q, key], so 1 is a uniform share
        self.stash = {"s": s, "cos_k": cos_k, "rel": rel, "H": H, "N": N}
        self.fires += 1

    def consume(self, reg_tok):
        """Per-head register values from the stash, then clears it. Returns
        (head_fields [H, len(PROBE_HEAD_FIELDS)], rel [H, N]), both float32,
        or (None, None) if nothing was captured."""
        st = self.stash
        self.stash = None
        if st is None:
            return None, None
        s, cos_k, rel = st["s"], st["cos_k"], st["rel"]
        H, N = st["H"], st["N"]
        top_rel, top_idx = rel.max(dim=1)
        if reg_tok is None:
            nanh = torch.full((H,), float("nan"), device=rel.device)
            fields = torch.stack([nanh, nanh, cos_k.median(dim=1).values, nanh,
                                  top_idx.to(torch.float32), top_rel, nanh], dim=1)
            return fields.to(torch.float32), rel
        r = int(reg_tok)
        rank = 1 + (s > s[:, r:r + 1]).sum(dim=1).to(torch.float32)
        fields = torch.stack([rank, cos_k[:, r], cos_k.median(dim=1).values,
                              rel[:, r], top_idx.to(torch.float32), top_rel,
                              (top_idx == r).to(torch.float32)], dim=1)
        return fields.to(torch.float32), rel

# ------------------------------------------------------------ unit tests
def _run_operator_unit_tests():
    g = torch.Generator().manual_seed(0)
    B, N, NT, D = 2, 12, 5, 8
    xin = torch.randn(B, N, D, generator=g)
    xout = xin + 0.5 * torch.randn(B, N, D, generator=g)
    idx = torch.tensor([3, 5])
    keep = [c for c in range(D) if c not in (3, 5)]

    # zero / freeze / nogrow
    y, v = apply_channel_op(xout, xin, idx, "zero")
    assert v.item() == 0 and (y[..., idx] == 0).all()
    assert torch.equal(y[..., keep], xout[..., keep])
    assert not (xout[..., idx] == 0).all(), "operator must not modify its input"
    y, v = apply_channel_op(xout, xin, idx, "freeze")
    assert v.item() == 0 and torch.equal(y[..., idx], xin[..., idx])
    y, v = apply_channel_op(xout, xin, idx, "nogrow")
    assert v.item() == 0
    assert (y[..., idx].abs() <= xin[..., idx].abs() + 1e-6).all()
    small = xout[..., idx].abs() <= xin[..., idx].abs()
    assert torch.equal(y[..., idx][small], xout[..., idx][small])
    assert torch.equal(y[..., keep], xout[..., keep])

    # growth with gamma 1 is the identity, 0 is freeze, 2 doubles the update.
    # Other channels stay untouched and the bf16 result passes its own check.
    y, v = apply_channel_op(xout, xin, idx, "growth", gamma=1.0)
    assert v.item() == 0 and torch.allclose(y, xout)
    y, v = apply_channel_op(xout, xin, idx, "growth", gamma=0.0)
    assert v.item() == 0 and torch.allclose(y[..., idx], xin[..., idx])
    y, v = apply_channel_op(xout, xin, idx, "growth", gamma=2.0)
    assert v.item() == 0
    assert torch.allclose(y[..., idx], xin[..., idx] + 2.0 * (xout[..., idx] - xin[..., idx]))
    assert torch.equal(y[..., keep], xout[..., keep])
    yb, vb = apply_channel_op(xout.to(torch.bfloat16), xin.to(torch.bfloat16), idx,
                              "growth", gamma=2.0)
    assert vb.item() == 0 and yb.dtype == torch.bfloat16
    # growth scales the block's update, not the state, so applying it twice
    # changes the result. The hook must fire exactly once per block, which
    # the smoke test and the per-block fire counts check.
    y2, _ = apply_channel_op(y, xin, idx, "growth", gamma=2.0)
    assert not torch.allclose(y2[..., idx], y[..., idx]), "sanity: re-application changes"

    # token operators
    vs = torch.zeros(D); vs[3] = 0.995; vs[1] = math.sqrt(1 - 0.995 ** 2)
    tok = 7
    y, v = apply_token_op(xout, tok, "dir_refresh", vstar=vs)
    assert v.item() == 0
    zz = y[:, tok, :]
    assert torch.allclose(zz.norm(dim=-1), xout[:, tok, :].norm(dim=-1), rtol=1e-5)
    assert ((zz @ vs) / zz.norm(dim=-1) > 1 - 1e-6).all()
    others = [t for t in range(N) if t != tok]
    assert torch.equal(y[:, others, :], xout[:, others, :])
    yb, vb = apply_token_op(xout.to(torch.bfloat16), tok, "dir_refresh", vstar=vs)
    assert vb.item() == 0
    ref = float(xout[:, tok, :].norm(dim=-1).max()) * 3.0
    y, v = apply_token_op(xout, tok, "norm_refresh", norm_ref=ref)
    assert v.item() == 0
    assert torch.allclose(y[:, tok, :].norm(dim=-1), torch.full((B,), ref), rtol=1e-5)
    c = (y[:, tok, :] * xout[:, tok, :]).sum(-1) / (y[:, tok, :].norm(dim=-1) * xout[:, tok, :].norm(dim=-1))
    assert (c > 1 - 1e-6).all()
    y, v = apply_token_op(xout, tok, "norm_refresh", norm_ref=1e-3)   # floor below the norm, token untouched
    assert v.item() == 0 and torch.equal(y, xout)
    y, v = apply_token_op(xout, tok, "sham")
    assert v.item() == 0 and torch.equal(y, xout) and y.data_ptr() != xout.data_ptr()

    # structure helpers
    enc = torch.randn(B, NT, D, generator=g)
    new, found = modify_block_output((enc, xout), N, NT,
                                     lambda img: apply_channel_op(img, xin, idx, "zero")[0])
    assert found and isinstance(new, tuple) and new[0] is enc and (new[1][..., idx] == 0).all()
    cat = torch.cat([enc, xout], dim=1)
    new, found = modify_block_output(cat, N, NT,
                                     lambda img: apply_channel_op(img, xin, idx, "zero")[0])
    assert found and torch.equal(new[:, :NT], enc) and (new[:, NT:][..., idx] == 0).all()
    _, found = modify_block_output(torch.randn(B, N + 1, D, generator=g), N, NT, lambda img: img)
    assert not found
    comp = resolve_channel_indices("all_except", [], [3, 5], D)
    assert len(comp) == D - 2 and 3 not in comp and 5 not in comp
    assert find_input_image_region((), {"hidden_states": xin}, N, NT) is xin
    assert find_input_image_region((xin,), {}, N, NT) is xin
    got = find_input_image_region((torch.cat([enc, xin], 1),), {}, N, NT)
    assert got.shape == (B, N, D) and torch.equal(got, xin)

    # v* fit recovers a planted direction from noisy scaled copies
    rng = np.random.default_rng(0)
    vt = np.zeros(D); vt[3] = 1.0; vt[1] = 0.1; vt /= np.linalg.norm(vt)
    M = 200
    rows = (rng.uniform(50, 500, (M, 1)) * vt[None, :]
            + rng.normal(0, 3.0, (M, D)))
    v, info = fit_vstar(rows, ch1=3)
    assert abs(float(v @ vt)) > 0.999, info
    assert float(v @ vt) > 0, "orientation must make mean projection positive"
    assert info["energy_on_ch1"] > 0.95

    # geometry readout with the register token on v* and random other tokens
    X = torch.randn(N, D, generator=g)
    X[tok] = 100.0 * torch.tensor(vt, dtype=torch.float32)
    gm = geometry_readout(X, torch.tensor(vt, dtype=torch.float32), tok, int(X.norm(dim=1).argmax()))
    gd = dict(zip(GEOM_FIELDS, gm.tolist()))
    assert abs(gd["cos_reg"] - 1.0) < 1e-5 and abs(gd["alpha_reg"] - 100.0) < 1e-3
    assert gd["xperp_reg"] < 1e-2 and abs(gd["norm_reg"] - 100.0) < 1e-3
    assert gd["argmax_cos_tok"] == tok and gd["n_cos_ge_thr"] >= 1
    gm2 = geometry_readout(X, torch.tensor(vt, dtype=torch.float32), None, tok)
    assert math.isnan(gm2[0].item()) and not math.isnan(gm2[4].item())

    # attention probe against a brute-force reference, both layouts
    import torch.nn.functional as F
    H, Dh = 3, 4
    for (n_txt, S) in ((0, N), (NT, NT + N)):
        pr = AttentionProbe(n_img=N, n_txt=n_txt, select_row=1, chunk=5)
        q = torch.randn(B, H, S, Dh, generator=g)
        k = torch.randn(B, H, S, Dh, generator=g)
        vv = torch.randn(B, H, S, Dh, generator=g)
        # make token `tok` a strong key for every head
        k[:, :, n_txt + tok, :] = q[:, :, n_txt:, :].mean(dim=2) * 40.0
        pr.patch()
        try:
            pr.active = 0
            out = F.scaled_dot_product_attention(q, k, vv)
            pr.active = None
            _ = F.scaled_dot_product_attention(q, k[:, :, :3, :], vv[:, :, :3, :])   # cross-attention-like, ignored
        finally:
            pr.unpatch()
        assert F.scaled_dot_product_attention is pr._orig or pr._orig is None
        assert pr.fires == 1 and pr.ignored == 0 and not pr.errors, (pr.fires, pr.ignored, pr.errors)
        ref_out = torch.nn.functional.scaled_dot_product_attention(q, k, vv)
        assert torch.allclose(out, ref_out)
        fields, rel = pr.consume(tok)
        assert fields.shape == (H, len(PROBE_HEAD_FIELDS)) and rel.shape == (H, N)
        # brute-force reference on row 1, image queries x image keys
        qi = q[1, :, n_txt:, :]; ki = k[1, :, n_txt:, :]
        A = torch.softmax(torch.einsum("hqd,hkd->hqk", qi, ki) * Dh ** -0.5, dim=-1)
        rel_ref = A.sum(dim=1)
        assert torch.allclose(rel, rel_ref, atol=1e-5)
        assert torch.allclose(rel.sum(dim=1), torch.full((H,), float(N)), atol=1e-4)
        fd = {f: fields[:, i] for i, f in enumerate(PROBE_HEAD_FIELDS)}
        assert (fd["reg_key_rank"] == 1).all() and (fd["reg_is_top_sink"] == 1).all()
        assert torch.allclose(fd["reg_sink_rel"], rel_ref[:, tok], atol=1e-5)
        assert (fd["reg_key_cos"] > fd["med_key_cos"]).all()
        # explicit scale kwarg is honored
        pr.patch()
        try:
            pr.active = 0
            _ = F.scaled_dot_product_attention(q, k, vv, scale=0.1)
        finally:
            pr.unpatch()
        _, rel2 = pr.consume(tok)
        A2 = torch.softmax(torch.einsum("hqd,hkd->hqk", qi, ki) * 0.1, dim=-1)
        assert torch.allclose(rel2, A2.sum(dim=1), atol=1e-5)
        # keyword call form, the diffusers native backend passes query= and key=
        pr.patch()
        try:
            pr.active = 0
            _ = F.scaled_dot_product_attention(query=q, key=k, value=vv)
        finally:
            pr.unpatch()
        assert pr.stash is not None and pr.fires == 3
        pr.consume(None)
    print("[intv-operators] all unit tests PASSED (zero / freeze / nogrow / growth; "
          "dir_refresh / norm_refresh / sham; v* fit; geometry readout; attention probe)")

_run_operator_unit_tests()


## I2b. Register direction v\*
v\* is fitted from the full tensors that the
channel recurrence main run stored at (l\*, t\*). We take the high-norm
tokens (norm at least three times the median, at most eight per scenario),
unit-normalize and stack them, take the top right-singular vector without
centering, and flip it so the mean projection is positive.

The result goes to `meta/vstar_v3.npz` together with its hash, its energy
per channel, the top singular energy and a split-half stability cosine. We
expect most of the energy on the dominant channel. After the capture, I8b
refits v\* per block from the baseline register vectors to see whether the
register direction itself rotates with depth.

In [ ]:
# =====================================================================
# I2b. Register direction v*, fitted from the recurrence main run and
# frozen to Drive
#
# Every high-norm register vector is unit-normalized, the
# vectors are stacked as rows, and v* is the top right-singular vector
# without centering, flipped so the mean projection is positive. The input
# is the fp16 full tensor [N_IMG, D] that the recurrence main run stored at
# (l*, t*) for each scenario. High-norm tokens have norm >= vstar_norm_mult
# x the median token norm, with at most vstar_max_per_scenario per scenario.
#
# This fit uses one block. After the capture, I8b refits v* from the
# baseline register vectors pooled over the register zone and reports the
# cosine between the two fits.
# =====================================================================
def _v4_tensor_path(sid):
    return scenario_paths(PATHS["main"], sid, True, L_STAR, T_STAR)["tensor"]

def _load_vstar_file(path):
    with np.load(path, allow_pickle=False) as z:
        v = z["vstar"].astype(np.float32)
        info = json.loads(str(z["info_json"].item()))
    return v, info

if os.path.exists(VSTAR_PATH) and not INTV.force_refit_vstar:
    VSTAR, VSTAR_INFO = _load_vstar_file(VSTAR_PATH)
    assert VSTAR.shape == (CFG.d_model,), VSTAR.shape
    assert VSTAR_INFO.get("ch1") == CH1 and VSTAR_INFO.get("l_star") == L_STAR \
        and VSTAR_INFO.get("t_star") == T_STAR, \
        "stored v* was fitted under a different decision; set force_refit_vstar=True"
    print(f"[vstar] loaded {VSTAR_PATH} (sha {VSTAR_INFO['vstar_sha']}, "
          f"{VSTAR_INFO['n_rows']} rows from {VSTAR_INFO['n_scenarios']} scenarios)")
else:
    _rows, _used, _missing = [], [], []
    _per_scn = []
    for _sid, _pid, _seed, _text in SCENARIOS:
        _tp = _v4_tensor_path(_sid)
        if not os.path.exists(_tp):
            _missing.append(_sid)
            continue
        X = np.load(_tp).astype(np.float32)                # [N_IMG, D]
        if X.shape != (N_IMG, CFG.d_model):
            _missing.append(_sid)
            continue
        nrm = np.linalg.norm(X, axis=1)
        med = max(float(np.median(nrm)), 1e-9)
        cand = np.where(nrm >= INTV.vstar_norm_mult * med)[0]
        if len(cand) == 0:
            _per_scn.append({"sid": _sid, "n_high": 0})
            continue
        cand = cand[np.argsort(-nrm[cand])][:int(INTV.vstar_max_per_scenario)]
        _rows.append(X[cand])
        _used.append(_sid)
        _per_scn.append({"sid": _sid, "n_high": int(len(cand)),
                         "top_norm_over_median": round(float(nrm[cand[0]] / med), 2)})
        del X
    if len(_used) < int(INTV.vstar_min_scenarios):
        raise RuntimeError(
            f"v* fit needs at least {INTV.vstar_min_scenarios} scenarios with a "
            f"stored full tensor and at least one high-norm token; found "
            f"{len(_used)} (missing tensors: {len(_missing)}). The v4 main run "
            f"must have completed with save_full_tensor=True on this Drive.")
    U = np.vstack(_rows)                                     # [M, D]
    VSTAR, VSTAR_INFO = fit_vstar(U, ch1=CH1)
    # split-half stability, refit on two disjoint halves of the scenarios
    _h = len(_rows) // 2
    _va, _ = fit_vstar(np.vstack(_rows[:_h]), ch1=CH1)
    _vb, _ = fit_vstar(np.vstack(_rows[_h:]), ch1=CH1)
    VSTAR_INFO.update({
        "n_scenarios": len(_used), "n_missing": len(_missing),
        "l_star": L_STAR, "t_star": T_STAR, "ch1": CH1,
        "norm_mult": INTV.vstar_norm_mult, "max_per_scenario": INTV.vstar_max_per_scenario,
        "split_half_abs_cos": round(float(abs(_va @ _vb)), 6),
        "energy_on_ch2": float(VSTAR[CH2] ** 2),
        "top5_channels_by_energy": [int(c) for c in np.argsort(-VSTAR ** 2)[:5]],
        "top5_energy": [round(float(VSTAR[c] ** 2), 5) for c in np.argsort(-VSTAR ** 2)[:5]],
        "source": "v4 main full tensors at (l*, t*)",
        "config_hash": CONFIG_HASH, "dec_sha": INT_DEC_SHA,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds")})
    VSTAR_INFO["vstar_sha"] = hashlib.sha256(VSTAR.tobytes()).hexdigest()[:16]
    save_npz_atomic(VSTAR_PATH, vstar=VSTAR.astype(np.float32),
                    info_json=np.array(json.dumps(VSTAR_INFO)),
                    per_scenario_json=np.array(json.dumps(_per_scn)))
    print(f"[vstar] fitted from {VSTAR_INFO['n_rows']} register rows of {len(_used)} "
          f"scenarios -> {VSTAR_PATH} (sha {VSTAR_INFO['vstar_sha']})")

VSTAR_SHA = VSTAR_INFO["vstar_sha"]
assert abs(float(np.linalg.norm(VSTAR)) - 1.0) < 1e-4, "v* must be a unit vector"
print(f"[vstar] |cos(v*, e_{CH1})| = {VSTAR_INFO['abs_cos_with_e_ch1']:.4f} "
      f"(energy on ch1 {VSTAR_INFO['energy_on_ch1']:.4f}; on ch2 "
      f"{VSTAR_INFO.get('energy_on_ch2', float('nan')):.5f}) | top singular energy "
      f"{VSTAR_INFO['top_singular_energy']:.4f} | mean row cos {VSTAR_INFO['mean_cos_rows']:.4f} "
      f"| split-half |cos| {VSTAR_INFO.get('split_half_abs_cos', float('nan')):.5f}")
if VSTAR_INFO["abs_cos_with_e_ch1"] < 0.9:
    print(f"[vstar][WARN] v* is not close to the channel-{CH1} axis; the paper "
          f"reports about 99 percent of the energy on it in FLUX.1-Dev. Inspect "
          f"top5_channels_by_energy before running the direction-refresh condition.")

# float32 torch copy of v*. The recorder moves it to each device on first use
VSTAR_T = torch.tensor(VSTAR, dtype=torch.float32)

fig, ax = plt.subplots(figsize=(FIG_COL, 2.2))
_e = VSTAR ** 2
ax.plot(np.arange(CFG.d_model), _e, lw=0.5, color="0.4")
for _c in VSTAR_INFO["top5_channels_by_energy"][:2]:
    ax.annotate(f"ch. {_c}", (_c, _e[_c]), xytext=(4, 2), textcoords="offset points", fontsize=LEGEND_PT)
ax.set_xlabel("residual-stream channel index")
ax.set_ylabel("share of v* energy (squared coefficient)")
save_paper_fig(fig, int_fig_stem("I2b_vstar_energy"),
    title=(f"Register direction v* in {MODEL_LABEL}: energy per channel (fitted at block {L_STAR}, "
           f"step {T_STAR}, from {VSTAR_INFO['n_scenarios']} generations)"),
    caption=(
    f"Channel energy of the register direction v* in {MODEL_LABEL}, fitted as the top right-singular "
    f"vector of the unit-normalized high-norm register tokens (norm at least {INTV.vstar_norm_mult:g} times "
    f"the median, at most {INTV.vstar_max_per_scenario} per scenario) at block {L_STAR}, step {T_STAR}, "
    f"pooled over {VSTAR_INFO['n_scenarios']} scenarios of the v4 run ({VSTAR_INFO['n_rows']} rows). Channel "
    f"{CH1} carries {VSTAR_INFO['energy_on_ch1']:.3f} of the energy; split-half refits agree with absolute "
    f"cosine {VSTAR_INFO.get('split_half_abs_cos', float('nan')):.4f}."))


## I4. Intervention recorder and runner
Each block gets one forward pre-hook and one forward hook. The pre-hook
switches the attention probe on at probe points. The hook applies the
condition's operator inside its block range, to every token for channel
operators and to the register or the position-control token for token
operators. It re-checks the result and then measures the stream, modified
or not. The measurements are the tracked-channel statistics, the top 10
channels, every token's norm, the values and norms of the max-norm and
register tokens, the v\* geometry, the register vector in fp16, and the probe
fields at probe points.

In a baseline run the register is the fp32 argmax-norm token at the
reference block, found separately for each step and known from that block
on. Intervention runs reuse the baseline's register indices, its norm
reference and its position-control token. The norm reference is, per step,
the largest token norm over blocks l\* to the band end. Runs can be resumed,
and each run stores its condition definition, run hash, decision hash and
v\* hash so they can be checked again.


In [ ]:
# =====================================================================
# I4. Intervention recorder and runner
#
# This is the measurement and intervention core that both the smoke test
# and the capture loop use. Each block gets a forward pre-hook and a
# forward hook (both with kwargs).
#
# The pre-hook switches the attention probe on when (block, step) is a
# probe point, so the attention call inside that block gets captured.
#
# The hook first applies the condition's operator to the image-stream block
# output when the block is inside the intervention range. Channel operators
# act on every token, token operators on the register or on the
# position-control token. It then re-checks the operator's result (the
# violation count) and measures the stream after the edit. The measurements
# are five statistics of the tracked channels, the top-10 channels, every
# token's L2 norm, the max-norm token and the register token, the v*
# geometry of those two tokens and of the whole token population, and the
# register vector in fp16. At probe points it also stores the per-head
# query-key rank, cosine and sink share of the register key, plus the full
# per-key sink-share vector.
#
# In a baseline run the register is the fp32 argmax-norm token at the
# reference block, found per step and known from that block onward.
# Intervention runs reuse the baseline's indices. Interventions act on every
# batch row, measurements read the conditional row.
# =====================================================================
import gc, re, time, datetime
import numpy as np
import torch

INT_TRACKED = [int(INT_DEC["ch1"]), int(INT_DEC["ch2"]), int(INT_DEC["ch_ctrl"])]
TRACKED_NAMES = ["ch1", "ch2", "ctrl"]
if "ch3" in INT_DEC:
    INT_TRACKED.append(int(INT_DEC["ch3"]))
    TRACKED_NAMES.append("ch3")
K_TR = len(INT_TRACKED)
N_GEOM = len(GEOM_FIELDS)
N_PF = len(PROBE_HEAD_FIELDS)

_GROWTH_RE = re.compile(r"^growth_x([0-9]+(?:\.[0-9]+)?)(_early)?$")

def make_conditions(dec):
    """Build the condition definitions from the frozen decision. Each entry
    has a kind ("none", "channel" or "token"), a mode, the block range and
    the operator parameters. Every run stores its entry in the metadata."""
    lo, hi = int(dec["l_grow"]), int(dec["l_end"])
    l_ref = int(dec["l_refresh"])
    band_lo = int(dec["band"][0])
    ch1, ch2, ctrl = int(dec["ch1"]), int(dec["ch2"]), int(dec["ch_ctrl"])
    base = dict(spec="none", channels=[], exclude=[], gamma=None, target=None)

    def chan(name, mode, spec, channels=(), exclude=(), lo_=lo, gamma=None):
        d = dict(base); d.update(name=name, kind="channel", mode=mode, spec=spec,
                                 channels=[int(c) for c in channels],
                                 exclude=[int(c) for c in exclude], lo=lo_, hi=hi,
                                 gamma=gamma)
        return d

    def tok(name, mode, target):
        d = dict(base); d.update(name=name, kind="token", mode=mode, target=target,
                                 lo=l_ref, hi=hi)
        return d

    lib = {
        "baseline":      dict(base, name="baseline", kind="none", mode=None, lo=lo, hi=hi),
        "zero_ch2":      chan("zero_ch2", "zero", "list", [ch2]),
        "nogrow_ch2":    chan("nogrow_ch2", "nogrow", "list", [ch2]),
        "freeze_ch2":    chan("freeze_ch2", "freeze", "list", [ch2]),
        "nogrow_others": chan("nogrow_others", "nogrow", "all_except", exclude=[ch1, ch2]),
        "zero_ctrl":     chan("zero_ctrl", "zero", "list", [ctrl]),
        "dir_refresh":         tok("dir_refresh", "dir_refresh", "register"),
        "dir_refresh_random":  tok("dir_refresh_random", "dir_refresh", "random"),
        "norm_refresh":        tok("norm_refresh", "norm_refresh", "register"),
        "sham":                tok("sham", "sham", "register"),
    }
    if "ch3" in dec:
        ch3 = int(dec["ch3"])
        lib["zero_ch3"] = chan("zero_ch3", "zero", "list", [ch3])
        lib["zero_both"] = chan("zero_both", "zero", "list", [ch2, ch3])
    out = []
    for n in INTV.conditions:
        if n in lib:
            out.append(lib[n]); continue
        m = _GROWTH_RE.match(n)
        if m:
            g = float(m.group(1))
            out.append(chan(n, "growth", "list", [ch2],
                            lo_=(band_lo if m.group(2) else lo), gamma=g))
            continue
        raise RuntimeError(f"condition {n!r} is not defined (ch3 present: {'ch3' in dec})")
    return out

CONDITIONS = make_conditions(INT_DEC)
COND_BY_NAME = {c["name"]: c for c in CONDITIONS}
COND_NAMES = [c["name"] for c in CONDITIONS]
_pal = OKABE_ITO[:6] + [OKABE_ITO[7], OKABE_ITO[6]]
COND_COLORS = [_pal[i % len(_pal)] for i in range(len(CONDITIONS))]
COND_LS = ["-" if i < len(_pal) else "--" for i in range(len(CONDITIONS))]

# ------------------------------------------------- human-readable labels for figures
# Raw condition names stay in file names, tables and metadata. Figures use
# these labels so a reader can see what was done to which channel.
TRACKED_LABELS = {"ch1": f"dominant ch. {INT_TRACKED[0]}",
                  "ch2": f"catching-up ch. {INT_TRACKED[1]}",
                  "ctrl": f"control ch. {INT_TRACKED[2]}"}
if "ch3" in TRACKED_NAMES:
    TRACKED_LABELS["ch3"] = f"second grower ch. {INT_TRACKED[TRACKED_NAMES.index('ch3')]}"

def condition_label(cond):
    n, ch2 = cond["name"], int(INT_DEC["ch2"])
    fixed = {
        "baseline": "baseline (no intervention)",
        "zero_ch2": f"ch. {ch2} set to 0",
        "nogrow_ch2": f"ch. {ch2} cannot grow",
        "freeze_ch2": f"ch. {ch2} frozen",
        "nogrow_others": f"all other channels cannot grow",
        "zero_ctrl": f"control ch. {int(INT_DEC['ch_ctrl'])} set to 0",
        "dir_refresh": "register direction reset to v*",
        "dir_refresh_random": "random token direction reset to v*",
        "norm_refresh": "register norm kept at peak",
        "sham": "sham (register rewritten unchanged)",
    }
    if n in fixed:
        return fixed[n]
    if "ch3" in INT_DEC:
        ch3 = int(INT_DEC["ch3"])
        if n == "zero_ch3":
            return f"ch. {ch3} set to 0"
        if n == "zero_both":
            return f"ch. {ch2} and ch. {ch3} set to 0"
    if cond["mode"] == "growth":
        g = cond["gamma"]
        gs = f"{g:g}"
        return f"ch. {ch2} growth x{gs}" + (" from band onset" if n.endswith("_early") else "")
    return n

COND_LABELS = {c["name"]: condition_label(c) for c in CONDITIONS}
COND_LABELS_SHORT = {k: v.replace(" (no intervention)", "").replace(" (register rewritten unchanged)", "")
                     for k, v in COND_LABELS.items()}

def tracked_targets(cond):
    """Names of the tracked channels that a channel condition targets.
    Empty for token conditions and the baseline."""
    if cond["kind"] != "channel":
        return []
    if cond["spec"] == "list":
        sel = {int(c) for c in cond["channels"]}
        return [n for n, c in zip(TRACKED_NAMES, INT_TRACKED) if c in sel]
    if cond["spec"] == "all_except":
        exc = {int(c) for c in cond["exclude"]}
        return [n for n, c in zip(TRACKED_NAMES, INT_TRACKED) if c not in exc]
    raise ValueError(f"unknown channel spec {cond['spec']!r}")

class InterventionRecorder:
    def __init__(self, transformer, n_steps, n_img, n_txt, d_model,
                 tracked_channels, reg_idx=None, intervention=None,
                 deep=False, expected_batch=1, select_row=0,
                 vstar=None, reg_norm_ref=None, op_tok_idx=None,
                 reg_ref_block=None, amax_vec_step=None,
                 probe_blocks=(), probe_steps=(), store_reg_vec=True,
                 probe_chunk=256, geom_thr=0.9):
        if MODEL_FAMILY == "flux":
            self.all_blocks = enumerate_flux_blocks(transformer)
        else:
            self.all_blocks = enumerate_model_blocks(transformer)
        self.L = len(self.all_blocks)
        self.n_steps, self.n_img, self.n_txt = int(n_steps), int(n_img), int(n_txt)
        self.d_model = int(d_model)
        self.expected_batch, self.select_row = int(expected_batch), int(select_row)
        self.tracked = [int(c) for c in tracked_channels]
        self.reg_idx = None if reg_idx is None else np.asarray(reg_idx, np.int64)
        if self.reg_idx is not None:
            assert self.reg_idx.shape == (self.n_steps,), \
                f"reg_idx shape {self.reg_idx.shape} != ({self.n_steps},)"
        self.reg_ref_block = None if reg_ref_block is None else int(reg_ref_block)
        self.deep = bool(deep)
        self.vstar_np = None if vstar is None else np.asarray(vstar, np.float32)
        self.reg_norm_ref = None if reg_norm_ref is None else np.asarray(reg_norm_ref, np.float64)
        self.op_tok_idx = None if op_tok_idx is None else np.asarray(op_tok_idx, np.int64)
        self.amax_vec_step = None if amax_vec_step is None else int(amax_vec_step)
        self.probe_blocks = {int(b) for b in probe_blocks}
        self.probe_block_pos = {int(b): i for i, b in enumerate(probe_blocks)}
        self.probe_steps_list = sorted({int(s) for s in probe_steps})
        self.probe_steps = set(self.probe_steps_list)
        self.probe_step_pos = {s: i for i, s in enumerate(self.probe_steps_list)}
        self.store_reg_vec = bool(store_reg_vec)
        self.geom_thr = float(geom_thr)
        self.intv = None
        if intervention is not None and intervention.get("kind") in ("channel", "token"):
            self.intv = dict(intervention)
            if self.intv["kind"] == "channel":
                self.intv_idx_np = resolve_channel_indices(
                    self.intv["spec"], self.intv["channels"],
                    self.intv["exclude"], d_model)
            else:
                if self.op_tok_idx is None:
                    raise RuntimeError("token operator needs op_tok_idx [n_steps]")
                assert self.op_tok_idx.shape == (self.n_steps,)
                if self.intv["mode"] == "dir_refresh" and self.vstar_np is None:
                    raise RuntimeError("dir_refresh needs v*")
                if self.intv["mode"] == "norm_refresh" and self.reg_norm_ref is None:
                    raise RuntimeError("norm_refresh needs reg_norm_ref [n_steps]")
        self.probe = AttentionProbe(n_img=self.n_img, n_txt=self.n_txt,
                                    select_row=self.select_row, chunk=probe_chunk)
        self._handles = []
        self.reset()

    def reset(self):
        L, T, N, D, K = self.L, self.n_steps, self.n_img, self.d_model, len(self.tracked)
        P, Q = len(self.probe_blocks), len(self.probe_steps_list)
        H = N_HEADS_EXPECTED
        self.tn_all = np.zeros((L, T, N), np.float16)
        self.tracked_stats = np.zeros((5, K, L, T), np.float32)
        self.top10_ids = np.zeros((L, T, 10), np.int32)
        self.top10_vals = np.zeros((L, T, 10), np.float32)
        self.amax_idx = np.zeros((L, T), np.int32)
        self.amax_scalars = np.zeros((L, T, K + 1), np.float32)   # tracked values + token norm
        self.reg_scalars = np.full((L, T, K + 1), np.nan, np.float32)
        self.reg_tok_used = np.full((L, T), -1, np.int32)         # register index used per (block, step)
        self.viol = np.zeros((L, T), np.float32)
        self.geom = np.full((L, T, N_GEOM), np.nan, np.float32)
        self.reg_vec = (np.zeros((L, T, D), np.float16) if self.store_reg_vec else None)
        self.amax_vec = (np.zeros((L, D), np.float16) if self.amax_vec_step is not None else None)
        self.probe_head = np.full((P, Q, H, N_PF), np.nan, np.float32)
        self.probe_rel = np.zeros((P, Q, H, N), np.float16)
        self.probe_fires = np.zeros((P, Q), np.int32)
        if self.deep:
            self.deep_stats = np.zeros((L, T, 5, D), np.float32)
            self.deep_ch_maps = np.zeros((K, L, T, N), np.float16)
        self.counters = {b: 0 for b in range(self.L)}
        self.overflow = {b: 0 for b in range(self.L)}
        self.intv_fires = {b: 0 for b in range(self.L)}
        self.errors = []
        self._clamped = 0
        self._idx_cache = {}
        self.probe.fires = 0; self.probe.ignored = 0; self.probe.errors = []
        self.probe.stash = None; self.probe.active = None

    # ------------------------------------------- device-local tensors
    def _on(self, dev):
        key = str(dev)
        ent = self._idx_cache.get(key)
        if ent is None:
            tr = torch.tensor(self.tracked, dtype=torch.long, device=dev)
            iv = (torch.tensor(self.intv_idx_np, dtype=torch.long, device=dev)
                  if (self.intv is not None and self.intv["kind"] == "channel") else None)
            vs = (torch.tensor(self.vstar_np, dtype=torch.float32, device=dev)
                  if self.vstar_np is not None else None)
            ent = (tr, iv, vs)
            self._idx_cache[key] = ent
        return ent

    # --------------------------------------------- measurement-row select
    def _pick_row(self, t3):
        b = t3.shape[0]
        if b == self.expected_batch:
            return t3[self.select_row if b > 1 else 0]
        if self.expected_batch == 2 and b == 1:       # guidance disabled
            return t3[0]
        raise RuntimeError(f"unexpected batch size {b} "
                           f"(expected {self.expected_batch})")

    def _extract_measure_row(self, output):
        if isinstance(output, (tuple, list)):
            for o in output:
                if o is not None and torch.is_tensor(o) and o.dim() == 3 \
                        and o.shape[1] == self.n_img:
                    return self._pick_row(o)
            raise RuntimeError("no image stream in tuple output; shapes %s" % (
                [tuple(o.shape) for o in output
                 if o is not None and torch.is_tensor(o)],))
        if torch.is_tensor(output) and output.dim() == 3:
            Lseq = output.shape[1]
            if Lseq == self.n_img:
                return self._pick_row(output)
            if self.n_txt and Lseq == self.n_img + self.n_txt:
                return self._pick_row(output)[self.n_txt:, :]
        raise RuntimeError(f"unexpected hook output type/shape {type(output)}")

    def _reg_for(self, bidx, step, amax_now):
        """Register token index for this block and step. Uses the baseline's
        index if one was given. In a baseline run it is the argmax-norm token
        at the reference block, known from that block onward in the step."""
        if self.reg_idx is not None:
            return int(self.reg_idx[step])
        if self.reg_ref_block is None:
            return None
        if bidx == self.reg_ref_block:
            return int(amax_now)
        if bidx > self.reg_ref_block:
            return int(self.amax_idx[self.reg_ref_block, step])
        return None

    # ------------------------------------------------------------ hooks
    def _make_pre_hook(self, bidx):
        def pre_hook(module, args, kwargs):
            step = self.counters[bidx]
            if bidx in self.probe_blocks and step in self.probe_steps and step < self.n_steps:
                if self.probe.stash is not None:
                    self.errors.append(f"block {bidx} step {step} [probe]: stale stash")
                    self.probe.stash = None
                self.probe.active = bidx
            else:
                self.probe.active = None
            return None
        return pre_hook

    def _make_hook(self, bidx):
        def hook(module, args, kwargs, output):
            step = self.counters[bidx]
            self.counters[bidx] = step + 1
            self.probe.active = None
            if step >= self.n_steps:
                self.overflow[bidx] += 1
                self.probe.stash = None
                return None
            new_output = None
            viol_t = None
            # intervene on the block output
            if self.intv is not None and self.intv["lo"] <= bidx <= self.intv["hi"]:
                try:
                    holder = {}
                    if self.intv["kind"] == "channel":
                        in_region = None
                        if self.intv["mode"] in ("nogrow", "freeze", "growth"):
                            in_region = find_input_image_region(
                                args, kwargs, self.n_img, self.n_txt)
                            if in_region is None:
                                raise RuntimeError(
                                    f"block-input image stream not found for "
                                    f"{self.intv['mode']} at block {bidx}")
                        def _fn(img):
                            _, iv, _ = self._on(img.device)
                            y, v = apply_channel_op(img, in_region, iv,
                                                    self.intv["mode"],
                                                    gamma=self.intv.get("gamma"))
                            holder["viol"] = v
                            return y
                    else:
                        tok = int(self.op_tok_idx[step])
                        nref = (float(self.reg_norm_ref[step])
                                if self.reg_norm_ref is not None else None)
                        def _fn(img):
                            _, _, vs = self._on(img.device)
                            y, v = apply_token_op(img, tok, self.intv["mode"],
                                                  vstar=vs, norm_ref=nref)
                            holder["viol"] = v
                            return y
                    new_output, found = modify_block_output(
                        output, self.n_img, self.n_txt, _fn)
                    if not found:
                        new_output = None
                        raise RuntimeError(
                            f"image stream not found in output at block {bidx}")
                    self.intv_fires[bidx] += 1
                    viol_t = holder.get("viol")
                except Exception as e:
                    self.errors.append(f"block {bidx} step {step} [intervene]: "
                                       f"{type(e).__name__}: {e}")
            # measure the stream after any edit
            try:
                meas_src = new_output if new_output is not None else output
                X = self._extract_measure_row(meas_src).to(torch.float32)
                if X.shape[0] != self.n_img or X.shape[1] != self.d_model:
                    raise RuntimeError(f"measured stream shape {tuple(X.shape)}")
                tracked_dev, _, vstar_dev = self._on(X.device)
                st = compute_channel_stats(X)
                stats5 = torch.stack([st[s] for s in STAT_NAMES], dim=0)   # [5, D]
                norms = X.norm(dim=1)                                      # [N]
                amax = int(torch.argmax(norms).item())
                t10v, t10i = torch.topk(st["mean_abs"], 10)
                tr = stats5[:, tracked_dev]                                # [5, K]
                arow = X[amax]
                a_sc = torch.cat([arow[tracked_dev], norms[amax].reshape(1)])
                r = self._reg_for(bidx, step, amax)
                K = len(self.tracked)
                if r is not None:
                    rrow = X[r]
                    r_sc = torch.cat([rrow[tracked_dev], norms[r].reshape(1)])
                else:
                    rrow = None
                    r_sc = torch.full((K + 1,), float("nan"), device=X.device)
                v_sc = (viol_t.to(torch.float32).reshape(1) if viol_t is not None
                        else torch.zeros(1, device=X.device))
                if vstar_dev is not None:
                    gm = geometry_readout(X, vstar_dev, r, amax, thr=self.geom_thr)
                else:
                    gm = torch.full((N_GEOM,), float("nan"), device=X.device)
                pack = torch.cat([tr.flatten(), t10v, t10i.to(torch.float32),
                                  a_sc, r_sc, v_sc, gm]).to("cpu", torch.float32).numpy()
                o = 0
                self.tracked_stats[:, :, bidx, step] = pack[o:o + 5 * K].reshape(5, K); o += 5 * K
                self.top10_vals[bidx, step] = pack[o:o + 10]; o += 10
                self.top10_ids[bidx, step] = pack[o:o + 10].astype(np.int32); o += 10
                self.amax_idx[bidx, step] = amax
                self.amax_scalars[bidx, step] = pack[o:o + K + 1]; o += K + 1
                self.reg_scalars[bidx, step] = pack[o:o + K + 1]; o += K + 1
                self.viol[bidx, step] = pack[o]; o += 1
                self.geom[bidx, step] = pack[o:o + N_GEOM]; o += N_GEOM
                self.reg_tok_used[bidx, step] = -1 if r is None else int(r)
                self._clamped += int((norms > 65504.0).sum().item())
                self.tn_all[bidx, step] = norms.clamp(max=65504.0) \
                                               .to("cpu", torch.float16).numpy()
                if self.reg_vec is not None and rrow is not None:
                    self.reg_vec[bidx, step] = rrow.clamp(-65504.0, 65504.0) \
                        .to("cpu", torch.float16).numpy()
                if self.amax_vec is not None and step == self.amax_vec_step:
                    self.amax_vec[bidx] = arow.clamp(-65504.0, 65504.0) \
                        .to("cpu", torch.float16).numpy()
                if bidx in self.probe_blocks and step in self.probe_steps:
                    pi, qi = self.probe_block_pos[bidx], self.probe_step_pos[step]
                    fields, rel = self.probe.consume(r)
                    if fields is None:
                        self.errors.append(f"block {bidx} step {step} [probe]: "
                                           f"no attention call captured")
                    else:
                        H = fields.shape[0]
                        if H != N_HEADS_EXPECTED:
                            raise RuntimeError(f"probe saw {H} heads, expected {N_HEADS_EXPECTED}")
                        self.probe_head[pi, qi] = fields.to("cpu").numpy()
                        self.probe_rel[pi, qi] = rel.clamp(max=65504.0).to("cpu", torch.float16).numpy()
                        self.probe_fires[pi, qi] += 1
                if self.deep:
                    self.deep_stats[bidx, step] = stats5.to("cpu", torch.float32).numpy()
                    self.deep_ch_maps[:, bidx, step] = (
                        X[:, tracked_dev].T.clamp(-65504.0, 65504.0)
                        .to("cpu", torch.float16).numpy())
            except Exception as e:
                self.errors.append(f"block {bidx} step {step} [measure]: "
                                   f"{type(e).__name__}: {e}")
            return new_output
        return hook

    def attach(self):
        assert not self._handles, "recorder already attached"
        self.probe.patch()
        for i, mod, _kind in self.all_blocks:
            self._handles.append(
                mod.register_forward_pre_hook(self._make_pre_hook(i), with_kwargs=True))
            self._handles.append(
                mod.register_forward_hook(self._make_hook(i), with_kwargs=True))
        return self

    def detach(self):
        for h in self._handles:
            h.remove()
        self._handles = []
        self.probe.unpatch()

    def finalize(self, expected_steps=None):
        exp = int(self.n_steps if expected_steps is None else expected_steps)
        fires_ok = (all(self.counters[b] == exp for b in range(self.L))
                    and all(v == 0 for v in self.overflow.values()))
        intv_ok = True
        if self.intv is not None:
            intv_ok = all(self.intv_fires[b] == exp
                          for b in range(self.intv["lo"], self.intv["hi"] + 1))
        n_probe_exp = sum(1 for s in self.probe_steps if s < exp) * len(self.probe_blocks)
        probe_ok = (int(self.probe_fires.sum()) == n_probe_exp
                    and not self.probe.errors)
        nan_flag = not (np.isfinite(self.tracked_stats).all()
                        and np.isfinite(self.tn_all.astype(np.float32)).all())
        return {"fires_ok": bool(fires_ok), "intv_fires_ok": bool(intv_ok),
                "probe_ok": bool(probe_ok), "probe_fires": int(self.probe_fires.sum()),
                "probe_expected": int(n_probe_exp), "probe_ignored": int(self.probe.ignored),
                "nan_flag": bool(nan_flag),
                "hook_errors": len(self.errors) + len(self.probe.errors),
                "errors": list(self.errors) + list(self.probe.errors),
                "counters": {int(b): int(c) for b, c in self.counters.items()},
                "viol_total": float(self.viol.sum()),
                "fp16_clamped": int(self._clamped)}

    def arrays(self):
        out = {"tn_all": self.tn_all, "tracked_stats": self.tracked_stats,
               "top10_ids": self.top10_ids, "top10_vals": self.top10_vals,
               "amax_idx": self.amax_idx, "amax_scalars": self.amax_scalars,
               "reg_scalars": self.reg_scalars, "reg_tok_used": self.reg_tok_used,
               "viol": self.viol, "geom": self.geom,
               "probe_head": self.probe_head, "probe_rel": self.probe_rel,
               "probe_fires": self.probe_fires}
        if self.reg_vec is not None:
            out["reg_vec"] = self.reg_vec
        if self.amax_vec is not None:
            out["amax_vec_tstar"] = self.amax_vec
        if self.deep:
            for b in range(self.L):
                for si, s in enumerate(STAT_NAMES):
                    out[f"b{b:02d}__{s}"] = self.deep_stats[b, :, si, :]
            out["deep_ch_maps"] = self.deep_ch_maps
        return out

def stored_run_healthy(paths):
    """True if a finished run file also has clean diagnostics."""
    try:
        with np.load(paths["npz"], allow_pickle=False) as z:
            m = meta_from_npz(z)
            return bool(m["fires_ok"] and m["intv_fires_ok"] and m["probe_ok"]
                        and m["hook_errors"] == 0 and not m["nan_flag"]
                        and m["viol_total"] == 0.0)
    except Exception:
        return False

def baseline_aux_from_arrays(arr, seed):
    """Build three per-step arrays [T] from a finished baseline run.

    reg_idx is the register index. reg_norm_ref is the norm target for
    norm_refresh, the largest register norm from the reference block to the
    band end in each step (from the fp32 max-norm-token scalars).
    rand_tok_idx is the position-control token, one ordinary token per
    scenario drawn with a fixed seed from tokens whose norm at the reference
    block and t* is below twice the median.
    """
    T = arr["amax_idx"].shape[1]
    reg = arr["amax_idx"][REG_REF_BLOCK].astype(np.int32)
    hi = int(CFG.band[1])
    ref = arr["amax_scalars"][REG_REF_BLOCK:hi + 1, :, -1].max(axis=0).astype(np.float32)   # [T]
    tn = arr["tn_all"][REG_REF_BLOCK, min(T_STAR, T - 1)].astype(np.float32)
    med = max(float(np.median(tn)), 1e-9)
    ordinary = np.where(tn < 2.0 * med)[0]
    ordinary = ordinary[ordinary != reg[min(T_STAR, T - 1)]]
    rng = np.random.default_rng(int(seed) + int(INTV.random_token_seed))
    rand_tok = int(rng.choice(ordinary)) if len(ordinary) else int((reg[0] + 1) % N_IMG)
    return reg, ref, np.full(T, rand_tok, np.int32)

def run_intervention_run(pipe, cond, sid_base, pid, prompt_text, seed,
                         baseline_aux=None, deep=False, out_dir=None, n_steps=None,
                         tag="intv", force=False, probe_steps=None):
    """Run one generation under one condition. Finished healthy runs are
    skipped and files are saved atomically. baseline_aux holds reg_idx,
    reg_norm_ref and rand_tok_idx from the scenario's baseline run and is
    required for every other condition."""
    out_dir = out_dir or INT_PATHS["runs"]
    n_steps = int(n_steps or CFG.num_steps)
    probe_steps = tuple(PROBE_STEPS_RESOLVED if probe_steps is None else probe_steps)
    rid = f"{cond['name']}__{sid_base}"
    paths = scenario_paths(out_dir, rid, False)
    expect = {"condition": cond["name"], "intv_hash": INTV_HASH,
              "dec_sha": INT_DEC_SHA, "vstar_sha": VSTAR_SHA, "num_steps": n_steps}
    if (not force and is_scenario_complete(paths, need_tensor=False, expect=expect)
            and stored_run_healthy(paths)):
        return {"tag": tag, "sid": rid, "prompt_id": pid, "seed": seed,
                "status": "skipped", "precision_mode": PRECISION_MODE}
    if cond["kind"] != "none" and baseline_aux is None:
        raise RuntimeError(f"{rid}: non-baseline run needs the baseline auxiliaries")
    reg_idx = None if baseline_aux is None else baseline_aux["reg_idx"]
    op_tok = None
    if cond["kind"] == "token":
        op_tok = (baseline_aux["rand_tok_idx"] if cond["target"] == "random"
                  else baseline_aux["reg_idx"])

    rec = InterventionRecorder(
        pipe.transformer, n_steps=n_steps, n_img=N_IMG, n_txt=N_TXT_IN_SEQ,
        d_model=CFG.d_model, tracked_channels=INT_TRACKED, reg_idx=reg_idx,
        intervention=(None if cond["kind"] == "none" else cond), deep=deep,
        expected_batch=EXP_BATCH, select_row=SELECT_ROW,
        vstar=VSTAR, reg_norm_ref=(None if baseline_aux is None else baseline_aux["reg_norm_ref"]),
        op_tok_idx=op_tok, reg_ref_block=REG_REF_BLOCK,
        amax_vec_step=min(T_STAR, n_steps - 1),
        probe_blocks=PROBE_BLOCKS, probe_steps=probe_steps,
        store_reg_vec=INTV.store_reg_vec, probe_chunk=INTV.probe_chunk,
        geom_thr=INTV.geom_cos_thr)

    torch.manual_seed(seed)
    gen = torch.Generator("cpu").manual_seed(seed)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    rec.attach()
    try:
        with torch.inference_mode():
            if MODEL_FAMILY == "flux":
                pe, pp = EMBEDS[pid]
                out = pipe(prompt_embeds=pe.to(DEVICE, DTYPE),
                           pooled_prompt_embeds=pp.to(DEVICE, DTYPE),
                           height=CFG.height, width=CFG.width,
                           num_inference_steps=n_steps,
                           guidance_scale=CFG.guidance_scale,
                           generator=gen, output_type="pil")
            else:
                E = EMBEDS[pid]
                out = pipe(prompt_embeds=E["pe"].to(DEVICE, DTYPE),
                           prompt_attention_mask=E["pm"].to(DEVICE),
                           negative_prompt_embeds=E["ne"].to(DEVICE, DTYPE),
                           negative_prompt_attention_mask=E["nm"].to(DEVICE),
                           negative_prompt=None,
                           height=CFG.height, width=CFG.width,
                           num_inference_steps=n_steps,
                           guidance_scale=CFG.guidance_scale,
                           generator=gen, output_type="pil",
                           use_resolution_binning=False)
    finally:
        rec.detach()
    wall = time.time() - t0
    peak = (torch.cuda.max_memory_allocated() / 2**30
            if torch.cuda.is_available() else 0.0)
    image = out.images[0]
    fin = rec.finalize(expected_steps=n_steps)
    if fin["errors"]:
        append_error_log(ERROR_LOG_PATH, rid,
                         "hook errors:\n  " + "\n  ".join(fin["errors"][:20]))

    arrays = rec.arrays()
    if cond["kind"] == "none":
        reg_out, ref_out, rand_out = baseline_aux_from_arrays(arrays, seed)
    else:
        reg_out = np.asarray(baseline_aux["reg_idx"], np.int32)
        ref_out = np.asarray(baseline_aux["reg_norm_ref"], np.float32)
        rand_out = np.asarray(baseline_aux["rand_tok_idx"], np.int32)
    arrays["reg_idx"] = reg_out
    arrays["reg_norm_ref"] = ref_out
    arrays["rand_tok_idx"] = rand_out
    arrays["op_tok_idx"] = (np.asarray(op_tok, np.int32) if op_tok is not None
                            else np.full(n_steps, -1, np.int32))
    arrays["sigmas"] = (pipe.scheduler.sigmas.detach().float().cpu().numpy()
                        if hasattr(pipe.scheduler, "sigmas")
                        else np.zeros(0, np.float32))
    arrays["timesteps"] = (pipe.scheduler.timesteps.detach().float().cpu().numpy()
                           if hasattr(pipe.scheduler, "timesteps")
                           else np.zeros(0, np.float32))

    meta = {"sid": rid, "sid_base": sid_base, "prompt_id": pid, "seed": int(seed),
            "prompt": prompt_text, "tag": tag, "model_id": CFG.model_id,
            "condition": cond["name"], "kind": cond["kind"], "mode": cond["mode"],
            "channels_spec": cond["spec"], "channels": cond["channels"],
            "exclude": cond["exclude"], "gamma": cond.get("gamma"),
            "target": cond.get("target"),
            "n_intv_channels": (int(len(resolve_channel_indices(
                cond["spec"], cond["channels"], cond["exclude"], CFG.d_model)))
                if cond["kind"] == "channel" else 0),
            "block_lo": cond["lo"], "block_hi": cond["hi"],
            "ch1": INT_TRACKED[0], "ch2": INT_TRACKED[1], "ch_ctrl": INT_TRACKED[2],
            "tracked_names": list(TRACKED_NAMES),
            "tracked_channels": [int(c) for c in INT_TRACKED],
            "reg_ref_block": int(REG_REF_BLOCK),
            "intv_hash": INTV_HASH, "dec_sha": INT_DEC_SHA, "vstar_sha": VSTAR_SHA,
            "geom_fields": list(GEOM_FIELDS), "probe_fields": list(PROBE_HEAD_FIELDS),
            "probe_blocks": [int(b) for b in PROBE_BLOCKS],
            "probe_steps": [int(s) for s in probe_steps],
            "n_heads": int(N_HEADS_EXPECTED),
            "height": CFG.height, "width": CFG.width, "num_steps": n_steps,
            "guidance_scale": CFG.guidance_scale,
            "n_img": int(N_IMG), "n_txt": int(N_TXT_IN_SEQ),
            "d_model": CFG.d_model, "deep": bool(deep),
            "expected_batch": EXP_BATCH, "select_row": SELECT_ROW,
            "applied_rows": "all",
            "counters": fin["counters"], "fires_ok": fin["fires_ok"],
            "intv_fires_ok": fin["intv_fires_ok"], "probe_ok": fin["probe_ok"],
            "probe_fires": fin["probe_fires"], "probe_expected": fin["probe_expected"],
            "probe_ignored": fin["probe_ignored"], "nan_flag": fin["nan_flag"],
            "hook_errors": fin["hook_errors"], "hook_error_msgs": fin["errors"][:10],
            "viol_total": fin["viol_total"], "fp16_clamped": fin["fp16_clamped"],
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "precision_mode": PRECISION_MODE, "config_hash": CONFIG_HASH,
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds")}
    if "ch3" in INT_DEC:
        meta["ch3"] = int(INT_DEC["ch3"])
    arrays["meta_json"] = meta_to_npz_field(meta)

    save_png_atomic(image, paths["png"])
    save_npz_atomic(paths["npz"], **arrays)      # written last, marks the run complete

    ok = (fin["fires_ok"] and fin["intv_fires_ok"] and fin["probe_ok"]
          and not fin["nan_flag"] and fin["hook_errors"] == 0 and fin["viol_total"] == 0.0)
    _tprobe = min(T_STAR, n_steps - 1)
    return {"tag": f"{tag}:{cond['name']}", "sid": rid, "prompt_id": pid,
            "seed": seed, "status": "ok" if ok else "warn",
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "fires_ok": fin["fires_ok"], "nan_flag": fin["nan_flag"],
            "hook_errors": fin["hook_errors"],
            "top1_channel": int(rec.top10_ids[L_STAR, _tprobe, 0]),
            "top1_score": round(float(rec.top10_vals[L_STAR, _tprobe, 0]), 4),
            "precision_mode": PRECISION_MODE}

def load_baseline_aux(sid_base, out_dir=None):
    bp = scenario_paths(out_dir or INT_PATHS["runs"], f"baseline__{sid_base}", False)
    with np.load(bp["npz"], allow_pickle=False) as z:
        return {"reg_idx": z["reg_idx"].copy(), "reg_norm_ref": z["reg_norm_ref"].copy(),
                "rand_tok_idx": z["rand_tok_idx"].copy()}

print(f"[intv-runner] ready | tracked channels {dict(zip(TRACKED_NAMES, INT_TRACKED))} "
      f"| conditions {COND_NAMES} | channel ops blocks {INT_DEC['l_grow']}..{INT_DEC['l_end']} "
      f"| token ops blocks {INT_DEC['l_refresh']}..{INT_DEC['l_end']} | v* sha {VSTAR_SHA}")
for _c in CONDITIONS:
    print(f"    {_c['name']:20s} kind={_c['kind']:7s} mode={str(_c['mode']):13s} "
          f"blocks {_c['lo']}..{_c['hi']}"
          + (f" channels {_c['channels']}" if _c['kind'] == 'channel' and _c['spec'] == 'list' else "")
          + (f" all except {_c['exclude']}" if _c['spec'] == 'all_except' else "")
          + (f" gamma={_c['gamma']}" if _c.get('gamma') is not None else "")
          + (f" target={_c['target']}" if _c.get('target') else ""))


## I5. Intervention smoke test

This cell has to pass before the capture starts. It runs short generations
of one scenario with the full recorder and checks the main operators. The
baseline must give a register index, finite geometry, and one probe call per
probe block with sink shares that sum to N. `zero_ch2` must give exact zeros
inside the range. For the first growth condition, the register's ch2 value
at the onset block must equal in + gamma (out - in) from the baseline.
`dir_refresh` must put the register at cosine one with v\*, `norm_refresh`
must keep the register norm at or above the reference, and `sham` must match
the baseline token norms within tolerance. `nogrow_others` and `zero_both`
are checked too if they are present. Any violation, hook error or failed
assert stops everything here, before the expensive runs.


In [ ]:
# =====================================================================
# I5. Intervention smoke test
#
# Short generations (CFG.smoke_steps) on one scenario with the full
# recorder. The baseline is a deep capture and must give register indices,
# finite v* geometry, and one probe call per probe block with per-head sink
# shares that sum to N. zero_ch2 must leave ch2 exactly zero inside the
# range and untouched before it. For the first growth_x<g> condition, the
# register's ch2 value at the onset block must equal in + g (out - in) from
# the baseline within bf16 tolerance. dir_refresh must give cosine one with v* at every intervened
# block (the operator itself checks the norm), norm_refresh must keep the
# register norm at or above the reference, and sham must match the baseline
# token norms. nogrow_others and zero_both are checked if present. Any
# violation, hook error or failed assert stops here, before the real runs.
# =====================================================================
print("=" * 70)
print("INTERVENTION SMOKE TEST (v3)")
print("=" * 70)
_LO, _HI = int(INT_DEC["l_grow"]), int(INT_DEC["l_end"])
_LR = int(INT_DEC["l_refresh"])
_SS = int(CFG.smoke_steps)
_I_CH2, _I_CTRL = TRACKED_NAMES.index("ch2"), TRACKED_NAMES.index("ctrl")
_G = {f: i for i, f in enumerate(GEOM_FIELDS)}
_PF = {f: i for i, f in enumerate(PROBE_HEAD_FIELDS)}
_SMOKE_PROBE_STEPS = (_SS - 1,)
for _f in glob.glob(os.path.join(INT_PATHS["smoke"], "*")):
    os.remove(_f)                                   # always a fresh smoke run

_smk = dict(sid_base="p00_s0", pid="p00", prompt_text=PROMPTS["p00"], seed=0,
            out_dir=INT_PATHS["smoke"], n_steps=_SS, tag="intv_smoke", force=True,
            probe_steps=_SMOKE_PROBE_STEPS)

def _smoke_load(name):
    p = scenario_paths(INT_PATHS["smoke"], f"{name}__p00_s0", False)
    z = np.load(p["npz"], allow_pickle=False)
    return z, meta_from_npz(z)

# ---- baseline (deep capture, register indices, geometry, probe)
_d0 = run_intervention_run(pipe, COND_BY_NAME["baseline"], deep=True, **_smk)
print(f"[intv-smoke] baseline: {_d0['status']} ({_d0['wall_s']}s)")
assert _d0["status"] == "ok", f"baseline smoke not clean: {_d0}"
_z, _m = _smoke_load("baseline")
_L = CFG.n_blocks_expected
assert all(int(c) == _SS for c in _m["counters"].values()), _m["counters"]
assert _z["tn_all"].shape == (_L, _SS, N_IMG) and _z["tn_all"].dtype == np.float16
assert _z["tracked_stats"].shape == (5, K_TR, _L, _SS)
assert _z["reg_idx"].shape == (_SS,) and _z["reg_norm_ref"].shape == (_SS,)
assert np.isfinite(_z["tracked_stats"]).all()
assert f"b{REG_REF_BLOCK:02d}__mean_abs" in _z.files, "deep arrays missing"
_REG = _z["reg_idx"].copy()
assert (_z["amax_idx"][REG_REF_BLOCK] == _REG).all(), \
    "register indices disagree with the recorded argmax-norm token"
_tn_ref = _z["tn_all"][REG_REF_BLOCK].astype(np.float32)
_rn = _tn_ref[np.arange(_SS), _REG]
assert (_rn >= _tn_ref.max(axis=1) * (1.0 - 1e-3)).all(), \
    "register token norm not maximal in the stored token-norm map"
# geometry is NaN before the reference block and finite from it onward
_gm = _z["geom"]
assert _gm.shape == (_L, _SS, len(GEOM_FIELDS))
assert np.isnan(_gm[:REG_REF_BLOCK, :, _G["cos_reg"]]).all(), "baseline register geometry before the reference block must be NaN"
assert np.isfinite(_gm[REG_REF_BLOCK:, :, _G["cos_reg"]]).all(), "register geometry not finite"
assert np.isfinite(_gm[:, :, _G["med_cos_all"]]).all()
_cos_reg_ref = float(_gm[REG_REF_BLOCK, _SS - 1, _G["cos_reg"]])
_cos_med_ref = float(_gm[REG_REF_BLOCK, _SS - 1, _G["med_cos_all"]])
print(f"[intv-smoke] baseline geometry at block {REG_REF_BLOCK}, step {_SS - 1}: "
      f"register cos(v*) = {_cos_reg_ref:.4f}, alpha = {_gm[REG_REF_BLOCK, _SS - 1, _G['alpha_reg']]:.1f}, "
      f"||x_perp|| = {_gm[REG_REF_BLOCK, _SS - 1, _G['xperp_reg']]:.1f}, "
      f"median token cos = {_cos_med_ref:.4f}")
if _cos_reg_ref < 0.9:
    print("[intv-smoke][WARN] register cosine with v* below 0.9 at the reference block "
          "(short smoke run; verify on the pilot before trusting dir_refresh)")
# the stored register vector should agree with the geometry
if "reg_vec" in _z.files:
    _rv = _z["reg_vec"][REG_REF_BLOCK, _SS - 1].astype(np.float32)
    _cos_chk = float(_rv @ VSTAR / max(np.linalg.norm(_rv), 1e-9))
    assert abs(_cos_chk - _cos_reg_ref) < 5e-3, (_cos_chk, _cos_reg_ref)
# the probe fires once per probe block at the smoke probe step, shares sum to N
assert _m["probe_ok"] and _m["probe_fires"] == len(PROBE_BLOCKS), \
    (f"attention probe did not fire as expected: fires={_m['probe_fires']}, "
     f"expected={_m['probe_expected']}, ignored={_m['probe_ignored']}, "
     f"errors={_m['hook_error_msgs']}. If fires == 0 the model's attention does not "
     f"reach torch.nn.functional.scaled_dot_product_attention (non-native backend).")
_ph = _z["probe_head"]                            # [P, Q, H, F]
_pr = _z["probe_rel"].astype(np.float32)          # [P, Q, H, N]
_qi = list(_SMOKE_PROBE_STEPS).index(_SS - 1) if _SS - 1 in _SMOKE_PROBE_STEPS else 0
assert _ph.shape == (len(PROBE_BLOCKS), len(_SMOKE_PROBE_STEPS), N_HEADS_EXPECTED, len(PROBE_HEAD_FIELDS))
assert np.isfinite(_ph[:, _qi]).all(), "probe fields not finite"
_sums = _pr[:, _qi].sum(axis=-1)                  # [P, H]
assert np.allclose(_sums, N_IMG, rtol=2e-2), f"sink shares do not sum to N: {_sums.min()}..{_sums.max()}"
for _pi, _pb in enumerate(PROBE_BLOCKS):
    _rk = _ph[_pi, _qi, :, _PF["reg_key_rank"]]
    _tp = _ph[_pi, _qi, :, _PF["reg_is_top_sink"]]
    print(f"[intv-smoke] probe block {_pb:2d}: register key rank median {np.median(_rk):.0f}, "
          f"top-10 in {np.mean(_rk <= 10):.0%} of heads, register is top sink in "
          f"{np.mean(_tp):.0%} of heads, register sink share {np.median(_ph[_pi, _qi, :, _PF['reg_sink_rel']]):.1f}x uniform")
_ch2_lo = float(_z["tracked_stats"][1, _I_CH2, max(_LO - 1, 0), _SS - 1])
_ch2_hi = float(_z["tracked_stats"][1, _I_CH2, min(int(CFG.band[1]), _L - 1), _SS - 1])
print(f"[intv-smoke] baseline ch2 mean|x| at block {max(_LO - 1, 0)}: {_ch2_lo:.3f} "
      f"-> band end: {_ch2_hi:.3f}")
_deep_ch2 = _z["deep_ch_maps"][_I_CH2].astype(np.float32)          # [L, T, N]
_z.close()
if REG_EXPECT_TOKEN is not None:
    _frac = float((_REG == REG_EXPECT_TOKEN).mean())
    print(f"[intv-smoke] register at the expected token {REG_EXPECT_TOKEN} in {_frac:.0%} of smoke steps")
_AUX = load_baseline_aux("p00_s0", out_dir=INT_PATHS["smoke"])
assert (_AUX["reg_idx"] == _REG).all()
assert (_AUX["rand_tok_idx"] != _AUX["reg_idx"]).all(), "position-control token equals the register"

# ---- zero_ch2 (exact zeros inside the range, untouched before it)
if "zero_ch2" in COND_BY_NAME:
    _d1 = run_intervention_run(pipe, COND_BY_NAME["zero_ch2"], baseline_aux=_AUX, **_smk)
    print(f"[intv-smoke] zero_ch2: {_d1['status']} ({_d1['wall_s']}s)")
    assert _d1["status"] == "ok", f"zero_ch2 smoke not clean: {_d1}"
    _z, _m = _smoke_load("zero_ch2")
    assert _m["viol_total"] == 0.0 and _m["intv_fires_ok"], _m["sid"]
    _ma_ch2 = _z["tracked_stats"][1, _I_CH2]
    assert float(np.abs(_ma_ch2[_LO:_HI + 1]).max()) < 1e-7, "ch2 not exactly zero inside the intervened range"
    if _LO > 0:
        assert float(_ma_ch2[:_LO].max()) > 1e-3, "ch2 unexpectedly zero before the intervention onset"
    assert float(np.nanmax(np.abs(_z["reg_scalars"][_LO:_HI + 1, :, _I_CH2]))) < 1e-7, "register-token ch2 value not zeroed"
    assert (_z["reg_idx"] == _REG).all()
    _z.close()
    print(f"[intv-smoke] zero_ch2 verified: ch2 mean|x| == 0 at blocks {_LO}..{_HI}, nonzero before block {_LO}")

# ---- growth_x<g> (the ch2 update at the onset block scaled by gamma)
# The check is exact only at the first denoising step. From step 1 on, the
# latents already carry the effect of the intervention at earlier steps, so
# the input to the onset block is no longer the baseline's. That is true of
# every intervention and is intended. At step 0 the input matches up to GPU
# nondeterminism, so in + gamma (out - in) has to hold within tolerance.
_growth_names = [c["name"] for c in CONDITIONS if c["mode"] == "growth"]
for _gn in _growth_names[:1]:
    _gc = COND_BY_NAME[_gn]
    _dg = run_intervention_run(pipe, _gc, baseline_aux=_AUX, **_smk)
    print(f"[intv-smoke] {_gn}: {_dg['status']} ({_dg['wall_s']}s)")
    assert _dg["status"] == "ok", f"{_gn} smoke not clean: {_dg}"
    _z, _m = _smoke_load(_gn)
    assert _m["viol_total"] == 0.0 and _m["intv_fires_ok"]
    _g = float(_gc["gamma"]); _b0 = int(_gc["lo"])
    assert _b0 >= 1, "growth condition cannot start at block 0 (no block input to compare)"
    _t = 0                                             # step 0 input matches the baseline
    _r = int(_REG[_t])
    _in = float(_deep_ch2[_b0 - 1, _t, _r])           # previous block's baseline output is this block's input
    _out = float(_deep_ch2[_b0, _t, _r])              # baseline raw output of the onset block
    _exp = _in + _g * (_out - _in)
    _got = float(_z["reg_scalars"][_b0, _t, _I_CH2])
    _got_in = float(_z["reg_scalars"][_b0 - 1, _t, _I_CH2])
    _tol_in = 0.02 * abs(_in) + 1e-2
    assert abs(_got_in - _in) <= _tol_in, \
        (f"{_gn}: at step 0 the block-{_b0 - 1} output differs from the baseline ({_got_in:.3f} vs "
         f"{_in:.3f}) before any intervention; the model is not run-to-run deterministic on this "
         f"GPU, or the hook order changed")
    _tol = 0.05 * max(abs(_exp), abs(_out), abs(_in)) + 1e-2
    assert abs(_got - _exp) <= _tol, \
        f"{_gn}: register ch2 at block {_b0}, step 0 is {_got:.3f}, expected in + gamma (out - in) = {_exp:.3f} (in {_in:.3f}, out {_out:.3f})"
    # at later steps the input has drifted from baseline, so only report it
    _tl = _SS - 1
    _div = abs(float(_z["reg_scalars"][_b0 - 1, _tl, _I_CH2]) - float(_deep_ch2[_b0 - 1, _tl, int(_REG[_tl])]))
    _z.close()
    print(f"[intv-smoke] {_gn} verified at block {_b0}, step 0: register ch2 {_got:.2f} vs "
          f"in + {_g:g} (out - in) = {_exp:.2f} (baseline in {_in:.2f}, out {_out:.2f}); "
          f"pre-onset input at step {_tl} differs from baseline by {_div:.3f} "
          f"(cross-step propagation of the intervention; expected to be nonzero)")

# ---- dir_refresh (register cosine with v* is one inside the range)
if "dir_refresh" in COND_BY_NAME:
    _dd = run_intervention_run(pipe, COND_BY_NAME["dir_refresh"], baseline_aux=_AUX, **_smk)
    print(f"[intv-smoke] dir_refresh: {_dd['status']} ({_dd['wall_s']}s)")
    assert _dd["status"] == "ok", f"dir_refresh smoke not clean: {_dd}"
    _z, _m = _smoke_load("dir_refresh")
    assert _m["viol_total"] == 0.0 and _m["intv_fires_ok"]
    _c = _z["geom"][_LR:_HI + 1, :, _G["cos_reg"]]
    assert np.isfinite(_c).all() and float(_c.min()) >= 1.0 - 1e-3, \
        f"register cosine with v* not one inside the refresh range (min {float(_c.min()):.5f})"
    _xp = _z["geom"][_LR:_HI + 1, :, _G["xperp_reg"]]
    _nr = _z["geom"][_LR:_HI + 1, :, _G["norm_reg"]]
    assert float((_xp / np.maximum(_nr, 1e-9)).max()) < 5e-2, "||x_perp|| not small after refresh"
    assert (_z["op_tok_idx"] == _REG).all()
    _z.close()
    print(f"[intv-smoke] dir_refresh verified: cos(register, v*) >= {1 - 1e-3} at blocks {_LR}..{_HI}")

# ---- norm_refresh (register norm at or above the reference inside the range)
if "norm_refresh" in COND_BY_NAME:
    _dn = run_intervention_run(pipe, COND_BY_NAME["norm_refresh"], baseline_aux=_AUX, **_smk)
    print(f"[intv-smoke] norm_refresh: {_dn['status']} ({_dn['wall_s']}s)")
    assert _dn["status"] == "ok", f"norm_refresh smoke not clean: {_dn}"
    _z, _m = _smoke_load("norm_refresh")
    assert _m["viol_total"] == 0.0 and _m["intv_fires_ok"]
    _nr = _z["geom"][_LR:_HI + 1, :, _G["norm_reg"]]                  # [range, T]
    _ref = _z["reg_norm_ref"][None, :]
    assert (_nr >= _ref * (1.0 - 1e-2)).all(), "register norm below the reference inside the refresh range"
    _z.close()
    print(f"[intv-smoke] norm_refresh verified: register norm >= reference "
          f"({float(_AUX['reg_norm_ref'][_SS - 1]):.0f} at step {_SS - 1}) at blocks {_LR}..{_HI}")

# ---- sham (token norms match the baseline within tolerance)
if "sham" in COND_BY_NAME:
    _ds = run_intervention_run(pipe, COND_BY_NAME["sham"], baseline_aux=_AUX, **_smk)
    print(f"[intv-smoke] sham: {_ds['status']} ({_ds['wall_s']}s)")
    assert _ds["status"] == "ok", f"sham smoke not clean: {_ds}"
    _z, _m = _smoke_load("sham")
    _zb, _ = _smoke_load("baseline")
    _a = _z["tn_all"].astype(np.float32); _b = _zb["tn_all"].astype(np.float32)
    _rel = np.abs(_a - _b) / np.maximum(np.abs(_b), 1.0)
    _z.close(); _zb.close()
    print(f"[intv-smoke] sham vs baseline token norms: max relative difference "
          f"{float(_rel.max()):.2e}, median {float(np.median(_rel)):.2e}")
    assert float(np.median(_rel)) < 1e-2, "sham run differs from baseline beyond tolerance"

# ---- optional conditions
if "nogrow_others" in COND_BY_NAME:
    _d2 = run_intervention_run(pipe, COND_BY_NAME["nogrow_others"], baseline_aux=_AUX, **_smk)
    assert _d2["status"] == "ok", f"nogrow_others smoke not clean: {_d2}"
    _z, _m = _smoke_load("nogrow_others")
    assert _m["viol_total"] == 0.0 and _m["intv_fires_ok"]
    for _tn_name in tracked_targets(COND_BY_NAME["nogrow_others"]):
        _ma_tgt = _z["tracked_stats"][1, TRACKED_NAMES.index(_tn_name)]
        for _t in range(_SS):
            _seg = _ma_tgt[max(_LO - 1, 0):_HI + 1, _t]
            assert (np.diff(_seg) <= _seg[:-1] * 0.02 + 1e-6).all(), \
                f"clamped channel {_tn_name} grew inside the range at step {_t}"
    _z.close()
    print("[intv-smoke] nogrow_others verified (clamped tracked channels non-increasing)")
if "zero_both" in COND_BY_NAME:
    _d3 = run_intervention_run(pipe, COND_BY_NAME["zero_both"], baseline_aux=_AUX, **_smk)
    assert _d3["status"] == "ok", f"zero_both smoke not clean: {_d3}"
    _z, _m = _smoke_load("zero_both")
    for _tn_name in tracked_targets(COND_BY_NAME["zero_both"]):
        _ma_tgt = _z["tracked_stats"][1, TRACKED_NAMES.index(_tn_name)]
        assert float(np.abs(_ma_tgt[_LO:_HI + 1]).max()) < 1e-7, f"{_tn_name} not zero in range"
    _z.close()
    print("[intv-smoke] zero_both verified")

print("INTERVENTION SMOKE TEST PASSED -- safe to start the capture loop.")


## I6. Capture of all conditions

For each scenario the baseline runs first and saves its register indices,
norm reference and position-control token. Every other condition then runs
with those. Finished healthy runs are skipped, so a disconnect only loses the
generation in progress. The five pilot scenarios also get deep captures
(full channel statistics and token maps of the tracked channels).


In [ ]:
# =====================================================================
# I6. Capture of all scenarios and conditions
#
# Per scenario the baseline runs first and stores its register indices,
# norm reference and position-control token in its npz. Every other
# condition then runs with those. Finished healthy runs are skipped, so the
# loop can be resumed. If the decision, v* or run hash changes, the metadata check
# treats old files as stale and they are redone.
# =====================================================================
from tqdm.auto import tqdm

_n_runs = len(INTV_SCENARIOS) * len(CONDITIONS)
_eta_s = _n_runs * (22 if MODEL_FAMILY == "flux" else 5)
print("=" * 70)
print(f"INTERVENTION CAPTURE (v3): {len(INTV_SCENARIOS)} scenarios x "
      f"{len(CONDITIONS)} conditions = {_n_runs} generations "
      f"(~{_eta_s / 3600:.1f} h estimated on an A100)")
print(f"conditions: {COND_NAMES}")
print(f"channel ops blocks {INT_DEC['l_grow']}..{INT_DEC['l_end']} | token ops blocks "
      f"{INT_DEC['l_refresh']}..{INT_DEC['l_end']} | all steps | probes at blocks "
      f"{list(PROBE_BLOCKS)} steps {list(PROBE_STEPS_RESOLVED)}")
print("=" * 70)

_n_ok = _n_skip = _n_warn = _n_err = 0
_pbar = tqdm(INTV_SCENARIOS, desc="scenarios")
for _sid_base, _pid, _seed, _text in _pbar:
    _deep = (_pid, _seed) in DEEP_PAIRS
    _aux = None
    for _cond in CONDITIONS:
        try:
            _diag = run_intervention_run(
                pipe, _cond, _sid_base, _pid, _text, _seed,
                baseline_aux=(None if _cond["kind"] == "none" else _aux),
                deep=_deep)
        except Exception as _e:
            _n_err += 1
            append_error_log(ERROR_LOG_PATH, f"{_cond['name']}__{_sid_base}",
                             f"{type(_e).__name__}: {_e}")
            print(f"[capture][ERROR] {_cond['name']}__{_sid_base}: "
                  f"{type(_e).__name__}: {_e} -- continuing")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if _cond["name"] == "baseline":
                break                    # the other conditions need the baseline auxiliaries
            continue
        if _diag["status"] == "skipped":
            _n_skip += 1
        else:
            append_run_log(RUN_LOG_PATH, _diag)
            if _diag["status"] == "ok":
                _n_ok += 1
            else:
                _n_warn += 1
                append_error_log(ERROR_LOG_PATH, _diag["sid"], f"warn diag: {_diag}")
        if _cond["name"] == "baseline":
            if _diag["status"] == "warn":
                print(f"[capture][WARN] baseline for {_sid_base} not clean -- "
                      f"skipping its conditions this pass")
                break
            try:
                _aux = load_baseline_aux(_sid_base)
            except Exception as _e:
                print(f"[capture][ERROR] cannot read baseline auxiliaries for "
                      f"{_sid_base}: {_e} -- skipping its conditions")
                break
        _pbar.set_postfix_str(f"{_cond['name']} {_diag.get('wall_s', 0)}s "
                              f"ok={_n_ok} skip={_n_skip}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f"[capture] done: {_n_ok} ok | {_n_skip} skipped(resume) | "
      f"{_n_warn} warn | {_n_err} error")
if _n_warn or _n_err:
    print(f"[capture] see {ERROR_LOG_PATH}; incomplete or unhealthy runs are "
          f"redone automatically on the next run of this cell.")


## I7. Integrity audit

Every expected run file is opened again and checked. The metadata has to
match the current condition definition, decision and v\*. Hook and probe
counts have to be right, with no hook errors, no operator violations and
finite arrays. The baseline auxiliaries must be the same across all
conditions of a scenario, and each condition gets one check of its own
operator on its own targets. Scenarios where every condition passes
make up the paired analysis set.


In [ ]:
# =====================================================================
# I7. Audit of all runs
#
# Opens every expected run file again and builds CLEAN_INTV, the paired
# analysis set. Each file must be complete, its metadata must match the
# current condition, run hash, decision and v*, the hook and probe counts
# must be exact, and there must be no hook errors, no operator violations
# and no non-finite values. Register indices and auxiliaries must be the
# same in every condition of a scenario. On top of that, each condition
# gets one check of its own operator on its own targets.
#
#   zero          tracked target channels exactly zero in range
#   nogrow        tracked target channels non-increasing in range (2 pct slack)
#   freeze        same check as nogrow
#   growth        gamma stored, and ch2 not exactly zero when gamma > 0
#   dir_refresh   register cosine with v* >= 1 - 1e-3 in the refresh range
#   norm_refresh  register norm >= reference (1 pct slack) in the refresh range
#   sham          only the generic checks
#
# Scenarios where every condition passes enter the analysis. Failed runs
# are redone by running I6 again.
# =====================================================================
_LO, _HI = int(INT_DEC["l_grow"]), int(INT_DEC["l_end"])
_LR = int(INT_DEC["l_refresh"])
_G = {f: i for i, f in enumerate(GEOM_FIELDS)}
_clean, _problems = [], []
_reg_pos_hits, _reg_pos_total = 0, 0

def _audit_one(rid, cond, base_aux):
    paths = scenario_paths(INT_PATHS["runs"], rid, False)
    if not (os.path.exists(paths["npz"]) and os.path.exists(paths["png"])):
        return f"{rid}: missing files"
    try:
        with np.load(paths["npz"], allow_pickle=False) as z:
            m = meta_from_npz(z)
            if (m["condition"] != cond["name"] or m["intv_hash"] != INTV_HASH
                    or m["dec_sha"] != INT_DEC_SHA or m.get("vstar_sha") != VSTAR_SHA):
                return f"{rid}: metadata mismatch (stale decision, v*, or config)"
            if m["mode"] != cond["mode"] or m["block_lo"] != cond["lo"] or m["block_hi"] != cond["hi"] \
                    or m.get("gamma") != cond.get("gamma") or m.get("target") != cond.get("target"):
                return f"{rid}: stored condition definition differs from the current one"
            if not (m["fires_ok"] and m["intv_fires_ok"] and m["probe_ok"]
                    and m["hook_errors"] == 0 and not m["nan_flag"]):
                return f"{rid}: unhealthy run diagnostics"
            if m["viol_total"] != 0.0:
                return f"{rid}: operator violations ({m['viol_total']})"
            ts = z["tracked_stats"]
            if not np.isfinite(ts).all():
                return f"{rid}: non-finite tracked stats"
            if not np.isfinite(z["geom"][REG_REF_BLOCK:, :, _G["cos_reg"]]).all():
                return f"{rid}: register geometry not finite from the reference block"
            if int(z["probe_fires"].sum()) != len(PROBE_BLOCKS) * len(PROBE_STEPS_RESOLVED):
                return f"{rid}: probe fire count {int(z['probe_fires'].sum())}"
            aux = {"reg_idx": z["reg_idx"], "reg_norm_ref": z["reg_norm_ref"],
                   "rand_tok_idx": z["rand_tok_idx"]}
            if base_aux is not None:
                for k in aux:
                    if not np.array_equal(aux[k], base_aux[k]):
                        return f"{rid}: {k} differs from baseline"
            ma = {n: ts[1, i] for i, n in enumerate(TRACKED_NAMES)}     # mean|x| [L, T]
            lo, hi = int(cond["lo"]), int(cond["hi"])
            if cond["kind"] == "channel":
                tgts = tracked_targets(cond)
                if cond["mode"] == "zero":
                    for t in tgts:
                        if float(np.abs(ma[t][lo:hi + 1]).max()) >= 1e-7:
                            return f"{rid}: zeroed channel {t} not exactly zero in range"
                elif cond["mode"] in ("nogrow", "freeze"):
                    for t in tgts:
                        seg = ma[t][max(lo - 1, 0):hi + 1]
                        if not (np.diff(seg, axis=0) <= seg[:-1] * 0.02 + 1e-6).all():
                            return f"{rid}: clamped channel {t} grew in range"
                elif cond["mode"] == "growth":
                    if m.get("gamma") is None:
                        return f"{rid}: growth run without gamma"
                    if float(m["gamma"]) > 0 and float(np.abs(ma["ch2"][lo:hi + 1]).max()) < 1e-7:
                        return f"{rid}: growth run has ch2 exactly zero (operator misapplied)"
            elif cond["kind"] == "token":
                if not np.array_equal(z["op_tok_idx"],
                                      aux["rand_tok_idx"] if cond["target"] == "random" else aux["reg_idx"]):
                    return f"{rid}: token operator index differs from its target"
                if cond["mode"] == "dir_refresh" and cond["target"] == "register":
                    c = z["geom"][lo:hi + 1, :, _G["cos_reg"]]
                    if not (np.isfinite(c).all() and float(c.min()) >= 1.0 - 1e-3):
                        return f"{rid}: register cosine with v* below 1 - 1e-3 in the refresh range"
                if cond["mode"] == "norm_refresh":
                    nr = z["geom"][lo:hi + 1, :, _G["norm_reg"]]
                    if not (nr >= aux["reg_norm_ref"][None, :] * (1.0 - 1e-2)).all():
                        return f"{rid}: register norm below the reference in the refresh range"
            return ("OK", aux, m)
    except Exception as e:
        return f"{rid}: unreadable ({type(e).__name__}: {e})"

for _sid_base, _pid, _seed, _text in INTV_SCENARIOS:
    _ok_all, _aux = True, None
    for _cond in CONDITIONS:
        _res = _audit_one(f"{_cond['name']}__{_sid_base}", _cond, _aux)
        if isinstance(_res, tuple):
            if _cond["name"] == "baseline":
                _aux = _res[1]
                if REG_EXPECT_TOKEN is not None:
                    _reg_pos_hits += int((_aux["reg_idx"] == REG_EXPECT_TOKEN).sum())
                    _reg_pos_total += len(_aux["reg_idx"])
        else:
            _ok_all = False
            _problems.append(_res)
            if _cond["name"] == "baseline":
                break
    if _ok_all:
        _clean.append((_sid_base, _pid, _seed, _text))

CLEAN_INTV = _clean
print(f"[audit] {len(CLEAN_INTV)}/{len(INTV_SCENARIOS)} scenarios fully clean "
      f"across all {len(CONDITIONS)} conditions")
if REG_EXPECT_TOKEN is not None and _reg_pos_total:
    _fr = _reg_pos_hits / _reg_pos_total
    print(f"[audit] baseline register at the expected token {REG_EXPECT_TOKEN}: "
          f"{_fr:.1%} of (scenario, step) pairs"
          + ("" if _fr >= 0.8 else "  [WARN: below 80 percent, inspect reg_idx]"))
if _problems:
    print(f"[audit] {len(_problems)} problem(s), first 10:")
    for _p in _problems[:10]:
        print(f"    {_p}")
    print("[audit] incomplete or stale runs are redone by re-running I6.")
save_json_atomic({"clean": [s for s, _, _, _ in CLEAN_INTV],
                  "problems": _problems[:200],
                  "intv_hash": INTV_HASH, "dec_sha": INT_DEC_SHA, "vstar_sha": VSTAR_SHA,
                  "timestamp": datetime.datetime.now().isoformat(timespec="seconds")},
                 os.path.join(INT_PATHS["analysis"], "intervention_audit.json"))
if not CLEAN_INTV:
    raise RuntimeError("no clean scenarios; run I6 to completion first")
print("[audit] AUDIT PASSED for the clean set -- proceed to the analysis.")


## I8. Register persistence analysis

The register token is the baseline's maximum-norm token at the reference
block, found per step and kept fixed across conditions. Register strength is
its norm divided by the median token norm. The persistence depth at
threshold tau is the deepest block, counting from the band onset, where the
strength is at least tau.

The primary endpoint is the depth at tau = 4 and step t\*. We also check tau
in {2, 4, 8} and other steps, and an any-token version of the depth that
still works if the register moves to another token. Each condition is
compared with the baseline of the same scenario. The tests are an exact sign
test, Wilcoxon, a prompt-level sign test with seeds collapsed, bootstrap
CIs, and Holm correction over conditions. Controls count as no effect when
the CI of the median shift stays within one block, a margin fixed in
advance. If at least three growth levels are present, a dose-response trend
over gamma is tested too.


In [ ]:
# =====================================================================
# I8. Persistence analysis
#
# Reads every run once into a cache, then builds the endpoint tables and
# figures. The cache also holds the v* geometry and the attention probe
# arrays that I8b uses.
# =====================================================================
import math
from tqdm.auto import tqdm
import pandas as pd

def fig_legend_top(fig, ax, ncol=4):
    """One legend row above the whole figure, for multi-panel figures."""
    h, l = ax.get_legend_handles_labels()
    if h:
        fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=ncol,
                   frameon=False, fontsize=_BASE_PT - 1.5, columnspacing=1.0, handletextpad=0.4)

FORCE_REBUILD_CACHE = False
_LO_B, _HI_B = int(CFG.band[0]), int(CFG.band[1])
_L_GROW, _L_END, _L_REF = int(INT_DEC["l_grow"]), int(INT_DEC["l_end"]), int(INT_DEC["l_refresh"])
_TAUS = tuple(float(t) for t in INTV.reg_tau_list)
_TAU_I = _TAUS.index(float(INTV.reg_tau_primary))
_COSL = tuple(float(c) for c in INTV.geom_cos_list)
_COS_I = _COSL.index(float(INTV.geom_cos_thr))
_G = {f: i for i, f in enumerate(GEOM_FIELDS)}
PROBE_BLOCKS_L = [int(b) for b in PROBE_BLOCKS]
_S, _C, _L, _T = len(CLEAN_INTV), len(CONDITIONS), CFG.n_blocks_expected, CFG.num_steps
_P, _Q, _H, _F = len(PROBE_BLOCKS), len(PROBE_STEPS_RESOLVED), N_HEADS_EXPECTED, len(PROBE_HEAD_FIELDS)
_CACHE = os.path.join(INT_PATHS["analysis"], "persistence_cache.npz")
_ckey = sha256_of_text(json.dumps({
    "dec": INT_DEC_SHA, "intv": INTV_HASH, "vstar": VSTAR_SHA, "taus": _TAUS,
    "cos": _COSL, "t": T_STAR, "conds": COND_NAMES,
    "sids": sorted(s for s, _, _, _ in CLEAN_INTV)}, sort_keys=True))

def _depths(mask_LT, lo):
    """Deepest block >= lo where the mask is true, per step. lo - 1 if never."""
    out = np.full(mask_LT.shape[1], lo - 1, np.int16)
    for t in range(mask_LT.shape[1]):
        nz = np.nonzero(mask_LT[lo:, t])[0]
        if len(nz):
            out[t] = lo + nz[-1]
    return out

def _build_cache():
    A = {
        "Rreg_t":   np.zeros((_S, _C, _L), np.float32),
        "Rany_t":   np.zeros((_S, _C, _L), np.float32),
        "med_t":    np.zeros((_S, _C, _L), np.float32),
        "regn_t":   np.zeros((_S, _C, _L), np.float32),
        "p99_t":    np.zeros((_S, _C, _L), np.float32),
        "ident_t":  np.zeros((_S, _C, _L), np.uint8),
        "tracked_ma_t": np.zeros((_S, _C, K_TR, _L), np.float32),
        "regsc_t":  np.full((_S, _C, _L, K_TR + 1), np.nan, np.float32),
        "amaxsc_t": np.zeros((_S, _C, _L, K_TR + 1), np.float32),
        "top10_probe": np.zeros((_S, _C, len(PROBE_BLOCKS_L), 10), np.int32),
        "depth_reg": np.zeros((_S, _C, len(_TAUS), _T), np.int16),
        "depth_any": np.zeros((_S, _C, len(_TAUS), _T), np.int16),
        "depth_cos": np.zeros((_S, _C, len(_COSL), _T), np.int16),
        "heat_reg":  np.zeros((_C, _L, _T), np.float64),
        "geom_t":   np.full((_S, _C, _L, len(GEOM_FIELDS)), np.nan, np.float32),
        "cos_reg_all": np.full((_S, _C, _L, _T), np.nan, np.float32),
        "norm_reg_all": np.full((_S, _C, _L, _T), np.nan, np.float32),
        "probe_head": np.full((_S, _C, _P, _Q, _H, _F), np.nan, np.float32),
        "psnr": np.full((_S, _C), np.nan, np.float32),
        "ident_ref_all": np.zeros((_S, _C, _T), np.uint8),   # per step, live argmax at the reference block equals the baseline register
    }
    from PIL import Image
    for si, (sid_base, pid, seed, _text) in enumerate(tqdm(CLEAN_INTV, desc="cache")):
        base_img = None
        for ci, cond in enumerate(CONDITIONS):
            rid = f"{cond['name']}__{sid_base}"
            pth = scenario_paths(INT_PATHS["runs"], rid, False)
            img = np.asarray(Image.open(pth["png"]).convert("RGB")).astype(np.float64)
            if ci == 0:
                base_img = img
                A["psnr"][si, ci] = 99.0
            else:
                mse = float(np.mean((img - base_img) ** 2))
                A["psnr"][si, ci] = 99.0 if mse == 0 else 10.0 * math.log10(255.0 ** 2 / mse)
            with np.load(pth["npz"], allow_pickle=False) as z:
                tn = z["tn_all"].astype(np.float32)               # [L, T, N]
                reg = z["reg_idx"].astype(np.int64)               # [T]
                med = np.median(tn, axis=2)                       # [L, T]
                mx = tn.max(axis=2)
                p99 = np.percentile(tn, 99, axis=2)
                regn = np.take_along_axis(tn, reg[None, :, None], axis=2)[:, :, 0]
                den = np.maximum(med, 1e-9)
                Rreg, Rany = regn / den, mx / den
                A["Rreg_t"][si, ci] = Rreg[:, T_STAR]
                A["Rany_t"][si, ci] = Rany[:, T_STAR]
                A["med_t"][si, ci] = med[:, T_STAR]
                A["regn_t"][si, ci] = regn[:, T_STAR]
                A["p99_t"][si, ci] = p99[:, T_STAR]
                A["ident_t"][si, ci] = (z["amax_idx"][:, T_STAR] == reg[T_STAR]).astype(np.uint8)
                A["ident_ref_all"][si, ci] = (z["amax_idx"][REG_REF_BLOCK, :] == reg).astype(np.uint8)
                A["tracked_ma_t"][si, ci] = z["tracked_stats"][1, :, :, T_STAR]
                A["regsc_t"][si, ci] = z["reg_scalars"][:, T_STAR, :]
                A["amaxsc_t"][si, ci] = z["amax_scalars"][:, T_STAR, :]
                A["top10_probe"][si, ci] = z["top10_ids"][PROBE_BLOCKS_L, T_STAR, :]
                A["heat_reg"][ci] += Rreg
                gm = z["geom"]                                    # [L, T, G]
                A["geom_t"][si, ci] = gm[:, T_STAR, :]
                A["cos_reg_all"][si, ci] = gm[:, :, _G["cos_reg"]]
                A["norm_reg_all"][si, ci] = gm[:, :, _G["norm_reg"]]
                A["probe_head"][si, ci] = z["probe_head"]
                for ti, tau in enumerate(_TAUS):
                    A["depth_reg"][si, ci, ti] = _depths(Rreg >= tau, _LO_B)
                    A["depth_any"][si, ci, ti] = _depths(Rany >= tau, _LO_B)
                cosreg = gm[:, :, _G["cos_reg"]]
                for ki, thr in enumerate(_COSL):
                    A["depth_cos"][si, ci, ki] = _depths(np.nan_to_num(cosreg, nan=-1.0) >= thr, _LO_B)
    A["heat_reg"] /= max(_S, 1)
    np.savez_compressed(_CACHE, cache_key=np.array(_ckey), **A)
    return A

if os.path.exists(_CACHE) and not FORCE_REBUILD_CACHE:
    with np.load(_CACHE, allow_pickle=False) as _z:
        if str(_z["cache_key"]) == _ckey:
            CACHE = {k: _z[k].copy() for k in _z.files if k != "cache_key"}
            print(f"[persist] cache loaded ({_S} scenarios x {_C} conditions)")
        else:
            print("[persist] cache stale -> rebuilding")
            CACHE = _build_cache()
else:
    CACHE = _build_cache()

# ------------------------------------------------- paired statistics
def _sign_test_p(diffs):
    d = np.asarray(diffs, np.float64)
    d = d[d != 0]
    n = len(d)
    if n == 0:
        return 1.0, 0
    k = int((d > 0).sum())
    lo = sum(math.comb(n, i) for i in range(0, min(k, n - k) + 1)) / 2 ** n
    return min(1.0, 2.0 * lo), n

def _boot_ci_median(diffs, n_boot=10000, seed=0):
    d = np.asarray(diffs, np.float64)
    if len(d) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    meds = np.median(rng.choice(d, size=(int(n_boot), len(d)), replace=True), axis=1)
    return (float(np.percentile(meds, 2.5)), float(np.percentile(meds, 97.5)))

def _boot_ci_mean(diffs, n_boot=10000, seed=0):
    d = np.asarray(diffs, np.float64)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = rng.choice(d, size=(int(n_boot), len(d)), replace=True).mean(axis=1)
    return (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))

try:
    from scipy.stats import wilcoxon as _wilcoxon
except Exception:
    _wilcoxon = None
    print("[persist][note] scipy unavailable: Wilcoxon skipped, sign test only")

PROMPT_IDS = [pid for _, pid, _, _ in CLEAN_INTV]
_uniq_prompts = sorted(set(PROMPT_IDS))

REL_SCALE = float(CFG.n_blocks_expected - 1)   # blocks per unit of relative depth, to compare models

def paired_table(values_SC, names, equiv_margin=None, censor_at=None, rel_scale=None):
    """Paired shift statistics against the baseline.

    values_SC is the endpoint per scenario and condition, shape [S, C], with
    the baseline in column 0. Returns one row dict per other condition with
    the median and mean shift, bootstrap CIs, exact sign test, Wilcoxon, a
    prompt-level sign test with seeds collapsed, and the Holm-corrected sign
    p over conditions. With equiv_margin set, the row also says whether the
    95 percent CI of the median shift lies inside [-margin, +margin], which
    is the equivalence rule for the controls.
    """
    rows, pvals = [], []
    for ci in range(1, values_SC.shape[1]):
        dlt = values_SC[:, ci] - values_SC[:, 0]
        ok = np.isfinite(dlt)
        dlt = dlt[ok]
        p_sign, n_nz = _sign_test_p(dlt)
        p_wil = np.nan
        if _wilcoxon is not None and (dlt != 0).any():
            try:
                p_wil = float(_wilcoxon(dlt, alternative="two-sided").pvalue)
            except Exception:
                pass
        pids = [p for p, k in zip(PROMPT_IDS, ok) if k]
        pm = [float(np.median(dlt[[i for i, p in enumerate(pids) if p == q]]))
              for q in sorted(set(pids))]
        p_sign_prompt, _ = _sign_test_p(pm)
        ci_lo, ci_hi = _boot_ci_median(dlt)
        row = {"condition": names[ci], "n": int(len(dlt)),
               "median_delta": float(np.median(dlt)) if len(dlt) else np.nan,
               "mean_delta": float(np.mean(dlt)) if len(dlt) else np.nan,
               "iqr_lo": float(np.percentile(dlt, 25)) if len(dlt) else np.nan,
               "iqr_hi": float(np.percentile(dlt, 75)) if len(dlt) else np.nan,
               "boot_ci_lo": ci_lo, "boot_ci_hi": ci_hi,
               "n_pos": int((dlt > 0).sum()), "n_neg": int((dlt < 0).sum()),
               "n_zero": int((dlt == 0).sum()),
               "p_sign": p_sign, "p_wilcoxon": p_wil,
               "p_sign_prompt_level": p_sign_prompt, "n_prompts": len(set(pids))}
        if rel_scale is not None:
            row["median_delta_rel"] = row["median_delta"] / rel_scale
            row["boot_ci_lo_rel"] = ci_lo / rel_scale
            row["boot_ci_hi_rel"] = ci_hi / rel_scale
        if censor_at is not None:
            row["censored_at_last"] = int((values_SC[:, ci] == censor_at).sum())
        if equiv_margin is not None:
            row["equiv_margin"] = float(equiv_margin)
            row["equivalent"] = bool(np.isfinite(ci_lo) and np.isfinite(ci_hi)
                                     and ci_lo >= -equiv_margin and ci_hi <= equiv_margin)
        rows.append(row)
        pvals.append(p_sign)
    order = np.argsort(pvals)
    run_max, m = 0.0, len(pvals)
    holm = {}
    for rank, oi in enumerate(order):
        run_max = max(run_max, (m - rank) * pvals[oi])
        holm[oi] = min(1.0, run_max)
    for i, r in enumerate(rows):
        r["p_sign_holm"] = holm[i]
    return rows

def print_table(rows, title, unit="blocks"):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)
    for r in rows:
        print(f"  {r['condition']:20s} median d = {r['median_delta']:+7.2f} {unit:6s} "
              f"CI95 [{r['boot_ci_lo']:+.2f}, {r['boot_ci_hi']:+.2f}]  mean {r['mean_delta']:+7.2f}  "
              f"+/0/- = {r['n_pos']}/{r['n_zero']}/{r['n_neg']}  "
              f"sign p = {r['p_sign']:.2e} (Holm {r['p_sign_holm']:.2e})  "
              f"Wilcoxon p = {r['p_wilcoxon']:.2e}  prompt-level p = {r['p_sign_prompt_level']:.2e}"
              + (f"  rel {r['median_delta_rel']:+.3f}" if "median_delta_rel" in r else "")
              + (f"  censored {r['censored_at_last']}" if "censored_at_last" in r else "")
              + (f"  equivalent(+/-{r['equiv_margin']:g}) = {r['equivalent']}" if "equivalent" in r else ""))

EQUIV_MARGIN_BLOCKS = 1.0     # fixed in advance, a control has no effect if the CI of its median shift is within +/- 1 block
DEPTH = CACHE["depth_reg"][:, :, _TAU_I, T_STAR].astype(np.float32)   # [S, C]
_rows = paired_table(DEPTH, COND_NAMES, equiv_margin=EQUIV_MARGIN_BLOCKS, censor_at=LAST_BLOCK, rel_scale=REL_SCALE)
print_table(_rows, f"PRIMARY ENDPOINT: register persistence depth (tau = {INTV.reg_tau_primary:g} x "
                   f"median token norm, step t* = {T_STAR}, n = {_S} paired scenarios; baseline median "
                   f"depth {np.median(DEPTH[:, 0]):.0f}, band {_LO_B}..{_HI_B}, last block {LAST_BLOCK})")
print("  (positive d = the register state persists into deeper blocks than baseline; Holm over the "
      f"{_C - 1} condition comparisons; the prompt-level sign test collapses seeds within each of the "
      f"{len(_uniq_prompts)} prompts; equivalence = bootstrap CI inside +/- {EQUIV_MARGIN_BLOCKS:g} block; "
      f"rel = shift / {REL_SCALE:g} blocks, for comparison across architectures)")
_bl_cens = int((DEPTH[:, 0] == LAST_BLOCK).sum())
if _bl_cens:
    print(f"  [note] {_bl_cens} baseline scenario(s) already censored at the last block")

# robustness over other thresholds and steps
_rob = []
for ti, tau in enumerate(_TAUS):
    for t in sorted({T_STAR, max(0, _T // 4), _T // 2, _T - 1}):
        D_ = CACHE["depth_reg"][:, :, ti, t].astype(np.float32)
        for r in paired_table(D_, COND_NAMES):
            _rob.append({"tau": tau, "step": t, **r})
_rob = pd.DataFrame(_rob)
_rob.to_csv(os.path.join(INT_PATHS["analysis"], "persistence_endpoint_robustness.csv"), index=False)
print("\n  robustness (median shift by tau and step):")
print(_rob.pivot_table(index="condition", columns=["tau", "step"], values="median_delta").round(1).to_string())

# any-token variant, which tolerates the register moving
_rows_any = paired_table(CACHE["depth_any"][:, :, _TAU_I, T_STAR].astype(np.float32), COND_NAMES,
                         equiv_margin=EQUIV_MARGIN_BLOCKS, censor_at=LAST_BLOCK, rel_scale=REL_SCALE)
print_table(_rows_any, f"SECONDARY: any-token persistence depth (max-norm token, tau = {INTV.reg_tau_primary:g}, step {T_STAR})")

# Register identity agreement is the fraction of (scenario, step) pairs where
# the live argmax-norm token at the reference block is still the baseline
# register that the token operators act on. Low agreement for a token
# condition means the operator hit a token that is no longer the register.
# The any-token depth above is the endpoint that still works in that case.
print("\n  register identity agreement at block %d (live argmax == baseline identity, over scenarios x steps):" % REG_REF_BLOCK)
IDENT_AGREE = {}
for ci, cn in enumerate(COND_NAMES):
    fr = float(CACHE["ident_ref_all"][:, ci, :].mean())
    IDENT_AGREE[cn] = fr
    flag = ""
    if CONDITIONS[ci]["kind"] == "token" and fr < 0.8:
        flag = "  [WARN: token operator target disagrees with the live register in >20% of steps]"
    print(f"    {cn:20s} {fr:.3f}{flag}")

# dose-response over the growth family (gamma is 0 for freeze, 1 for baseline, g for growth_x<g>)
_dose = [(1.0, 0)] + [(float(c["gamma"]), i) for i, c in enumerate(CONDITIONS) if c["mode"] == "growth" and not c["name"].endswith("_early")]
_dose += [(0.0, i) for i, c in enumerate(CONDITIONS) if c["name"] == "freeze_ch2"]
_dose = sorted(set(_dose))
if len(_dose) >= 3:
    from scipy.stats import spearmanr
    _gam = np.array([g for g, _ in _dose]); _cols = [i for _, i in _dose]
    _rhos = np.array([spearmanr(_gam, DEPTH[s, _cols]).correlation for s in range(_S)])
    _rhos = _rhos[np.isfinite(_rhos)]
    _p_rho, _ = _sign_test_p(_rhos)
    print(f"\n  DOSE-RESPONSE over gamma {list(_gam)}: per-scenario Spearman rho of depth vs gamma, "
          f"median {np.median(_rhos):+.2f}, {np.mean(_rhos < 0):.0%} negative, sign test p = {_p_rho:.2e} "
          f"(channel-competition hypothesis predicts negative rho: more growth, earlier dissolution)")
else:
    print(f"\n  dose-response trend test skipped ({len(_dose)} gamma levels; needs >= 3, e.g. add growth_x0.5 or freeze_ch2)")

_tab = pd.DataFrame(_rows)
_tab.to_csv(os.path.join(INT_PATHS["analysis"], "persistence_endpoint.csv"), index=False)
pd.DataFrame(_rows_any).to_csv(os.path.join(INT_PATHS["analysis"], "persistence_endpoint_anytoken.csv"), index=False)
_per = []
for si, (sid_base, pid, seed, _t) in enumerate(CLEAN_INTV):
    for ci, cn in enumerate(COND_NAMES):
        row = {"sid": sid_base, "prompt_id": pid, "seed": seed, "condition": cn,
               "psnr_db": float(CACHE["psnr"][si, ci])}
        for ti, tau in enumerate(_TAUS):
            row[f"depth_reg_tau{tau:g}"] = int(CACHE["depth_reg"][si, ci, ti, T_STAR])
            row[f"depth_any_tau{tau:g}"] = int(CACHE["depth_any"][si, ci, ti, T_STAR])
        for ki, thr in enumerate(_COSL):
            row[f"depth_cos{thr:g}"] = int(CACHE["depth_cos"][si, ci, ki, T_STAR])
        _per.append(row)
pd.DataFrame(_per).to_csv(os.path.join(INT_PATHS["analysis"], "persistence_depths.csv"), index=False)
print(f"[persist] endpoint and per-scenario tables -> {INT_PATHS['analysis']}")

# ------------------------------------------------- F1 persistence curves
_x = np.arange(_L)
_XLAB = "transformer block"
fig, ax = plt.subplots(figsize=(FIG_PAGE, 2.9))
ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band")
ax.axvline(_L_GROW, color="0.3", ls=":", lw=0.8, label=f"channel-operator onset (block {_L_GROW})")
ax.axvline(_L_REF, color="0.3", ls="-.", lw=0.6, label=f"token-operator onset (block {_L_REF})")
for ci, cn in enumerate(COND_NAMES):
    med = np.median(CACHE["Rreg_t"][:, ci, :], axis=0)
    lo = np.percentile(CACHE["Rreg_t"][:, ci, :], 25, axis=0)
    hi = np.percentile(CACHE["Rreg_t"][:, ci, :], 75, axis=0)
    ax.semilogy(_x, med, color=COND_COLORS[ci], ls=COND_LS[ci], lw=1.2, label=COND_LABELS_SHORT[cn])
    ax.fill_between(_x, lo, hi, color=COND_COLORS[ci], alpha=0.12, lw=0)
for tau in _TAUS:
    ax.axhline(tau, color="0.55", ls="--", lw=0.6)
    ax.annotate(f"threshold {tau:g}" + (" (primary)" if tau == INTV.reg_tau_primary else ""), (1.0, tau),
                xycoords=("axes fraction", "data"), xytext=(-2, 1), textcoords="offset points",
                ha="right", va="bottom", fontsize=LEGEND_PT - 1, color="0.45")
ax.set_xlabel(_XLAB)
ax.set_ylabel("register strength\n(register norm / median token norm)")
save_paper_fig(fig, int_fig_stem("I8_persistence_curves"), legend_ax=ax, ncol=3,
    title=(f"Register persistence under interventions in {MODEL_LABEL}: register-token norm relative "
           f"to the median token, per block (step {T_STAR}, median and interquartile range over {_S} generations)"),
    caption=(
    f"Register persistence in {MODEL_LABEL} under the interventions: the register token's L2 norm "
    f"divided by the median token norm, per block at step {T_STAR}, median with interquartile band "
    f"over {_S} scenarios. The register token is fixed per scenario and step from the baseline run "
    f"(maximum-norm token at block {REG_REF_BLOCK}). The shaded region is the high-norm band; the "
    f"dotted line marks the channel-intervention onset block {_L_GROW}, the dash-dotted line the "
    f"token-intervention onset block {_L_REF} (all interventions act through block {_L_END}); dashed "
    f"lines mark the persistence thresholds {', '.join(f'{t:g}' for t in _TAUS)}."))

# ------------------------------------------------- F2 paired depth shifts
fig, ax = plt.subplots(figsize=(FIG_PAGE * 0.75, 3.0))
_rng = np.random.default_rng(0)
for ci in range(1, _C):
    dlt = (DEPTH[:, ci] - DEPTH[:, 0]).astype(float)
    xj = ci + _rng.uniform(-0.16, 0.16, len(dlt))
    ax.scatter(xj, dlt, s=4, color=COND_COLORS[ci], alpha=0.45, linewidths=0)
    ax.hlines(np.median(dlt), ci - 0.28, ci + 0.28, color="0.1", lw=1.4, zorder=5)
    r = _rows[ci - 1]
    ax.vlines(ci, r["boot_ci_lo"], r["boot_ci_hi"], color="0.1", lw=0.8, zorder=5)
ax.axhline(0, color="0.6", lw=0.7)
ax.axhspan(-EQUIV_MARGIN_BLOCKS, EQUIV_MARGIN_BLOCKS, color="0.85", alpha=0.5, lw=0, zorder=0,
           label=f"equivalence margin (+/- {EQUIV_MARGIN_BLOCKS:g} block)")
ax.scatter([], [], s=4, color="0.3", label="one generation (prompt, seed)")
ax.plot([], [], color="0.1", lw=1.4, label="median with bootstrap 95% CI")
ax.legend(loc="best", fontsize=LEGEND_PT, frameon=True, framealpha=0.9, edgecolor="none")
ax.set_xticks(range(1, _C))
ax.set_xticklabels([wrap_label(COND_LABELS_SHORT[c], 18) for c in COND_NAMES[1:]], rotation=35, ha="right", fontsize=LEGEND_PT)
ax.set_ylabel("shift of persistence depth vs. baseline (blocks)\npositive = register survives deeper")
save_paper_fig(fig, int_fig_stem("I8_depth_shifts"),
    title=(f"Does the register survive deeper under each intervention? Paired shift of the register "
           f"persistence depth vs. baseline ({MODEL_LABEL}, {_S} generations, step {T_STAR})"),
    caption=(
    f"Paired shifts of the register persistence depth (threshold {INTV.reg_tau_primary:g} times the "
    f"median token norm, step {T_STAR}) for each condition relative to the baseline of the same "
    f"scenario, over {_S} scenarios in {MODEL_LABEL}; black bars mark medians with bootstrap 95 "
    f"percent confidence intervals; the grey band is the pre-declared equivalence margin of "
    f"+/- {EQUIV_MARGIN_BLOCKS:g} block. Positive values mean the register state survives into "
    f"deeper blocks than in the baseline run."))

# ------------------------------------------------- F3 block x step heatmaps
fig, _axs = panel_grid(_C, max_cols=4, panel_h=1.9)
_hm = CACHE["heat_reg"]
_vmax = max(float(np.percentile(_hm, 99.5)), 10.0)
for ci, (ax, cn) in enumerate(zip(_axs, COND_NAMES)):
    im = ax.imshow(_hm[ci], aspect="auto", origin="lower", cmap="magma",
                   norm=LogNorm(vmin=1.0, vmax=_vmax), interpolation="nearest", rasterized=True)
    ax.axhline(_HI_B, color="w", ls="--", lw=0.5)
    ax.set_title(wrap_label(COND_LABELS_SHORT[cn], 24), fontsize=LEGEND_PT, pad=2)
    ax.tick_params(labelsize=LEGEND_PT)
grid_labels(_axs, 4, "denoising step", "transformer block")
cb = fig.colorbar(im, ax=_axs, pad=0.01, shrink=0.9)
cb.set_label("mean register strength (register norm / median norm)", fontsize=LEGEND_PT)
cb.ax.tick_params(labelsize=LEGEND_PT)
save_paper_fig(fig, int_fig_stem("I8_persistence_heatmaps"),
    title=(f"Register strength across depth and denoising time, per condition ({MODEL_LABEL}, mean over "
           f"{_S} generations; the dashed line marks the band end, block {_HI_B})"),
    caption=(
    f"Register strength (register norm over median norm, mean across {_S} scenarios) per block and "
    f"denoising step in {MODEL_LABEL}, one panel per condition, shared log color scale. The dashed "
    f"line marks the band end (block {_HI_B})."))

# ------------------------------------------------- F4 identity persistence
fig, ax = plt.subplots(figsize=(FIG_PAGE * 0.75, 2.5))
ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band")
for ci, cn in enumerate(COND_NAMES):
    ax.plot(_x, CACHE["ident_t"][:, ci, :].mean(axis=0), color=COND_COLORS[ci], ls=COND_LS[ci],
            lw=1.1, label=COND_LABELS_SHORT[cn])
ax.set_xlabel(_XLAB)
ax.set_ylabel("fraction of generations in which\nthe max-norm token is the register")
ax.set_ylim(0, 1.02)
save_paper_fig(fig, int_fig_stem("I8_identity_persistence"), legend_ax=ax, ncol=3,
    title=(f"Is the largest-norm token still the register? Fraction of generations per block "
           f"({MODEL_LABEL}, step {T_STAR}; register identity fixed from the baseline at block {REG_REF_BLOCK})"),
    caption=(
    f"Fraction of scenarios in which the maximum-norm token at a block equals the scenario's register "
    f"token (baseline identity at block {REG_REF_BLOCK}), at step {T_STAR}, per condition in "
    f"{MODEL_LABEL}. The shaded region is the high-norm band."))


## I8b. Geometry and attention endpoints

These secondary endpoints look at the register's direction and its role in
attention instead of its norm. The alignment depth is the deepest block
where cos(register, v\*) is at or above the threshold. We also plot alpha,
||x_perp||, the cosine and the angular velocity per block, and how much of
the register's squared norm sits on the tracked channels and along v\*.

At each probe block we measure the fraction of heads whose top sink is the
register, the fraction with the register key in the top 10, the median rank
of that key and the attention it receives. All of these are paired against
the baseline. The cell ends by refitting v\* per block from the baseline
register vectors, to see whether the natural direction changes with depth.


In [ ]:
# =====================================================================
# I8b. Geometry and attention endpoints
#
# Secondary endpoints, fixed in advance, that look at the register's
# direction and its attention role rather than its norm.
#
# The alignment depth is the deepest block from the band onset where the
# register's cosine with v* is at least geom_cos_thr. It gets the same
# paired test against baseline as the primary endpoint.
#
# The sink endpoints are measured at each probe block. They are the fraction
# of heads whose top sink is the register, the fraction of heads with the
# register key in the top 10 by alignment with the mean query, and the
# register's incoming attention relative to uniform.
#
# The cell also plots alpha, ||x_perp||, the cosine and the angular velocity
# per condition. Finally it refits v* per block from the baseline register
# vectors, to check whether the natural register direction itself rotates
# with depth.
# =====================================================================
_PF = {f: i for i, f in enumerate(PROBE_HEAD_FIELDS)}
_QI = list(PROBE_STEPS_RESOLVED).index(T_STAR)

# ------------------------------------------------- alignment depth endpoint
ADEPTH = CACHE["depth_cos"][:, :, _COS_I, T_STAR].astype(np.float32)   # [S, C]
_rows_cos = paired_table(ADEPTH, COND_NAMES, equiv_margin=EQUIV_MARGIN_BLOCKS, censor_at=LAST_BLOCK, rel_scale=REL_SCALE)
print_table(_rows_cos, f"SECONDARY ENDPOINT: alignment depth (deepest block with cos(register, v*) >= "
                       f"{INTV.geom_cos_thr:g}, step {T_STAR}, n = {_S}; baseline median {np.median(ADEPTH[:, 0]):.0f})")
pd.DataFrame(_rows_cos).to_csv(os.path.join(INT_PATHS["analysis"], "alignment_endpoint.csv"), index=False)
print("  robustness (median shift by cosine threshold):")
for ki, thr in enumerate(_COSL):
    D_ = CACHE["depth_cos"][:, :, ki, T_STAR].astype(np.float32)
    print(f"    thr {thr:g}: " + ", ".join(f"{r['condition']} {r['median_delta']:+.1f}" for r in paired_table(D_, COND_NAMES)))

# ------------------------------------------------- geometry trajectories
_x = np.arange(_L)
_gt = CACHE["geom_t"]                                    # [S, C, L, G]
fig, axes = plt.subplots(1, 4, figsize=(FIG_PAGE, 2.5))
_panels = [("alpha_reg", "(a) projection on v*", r"$\alpha = \langle x_r, v^*\rangle$", True),
           ("xperp_reg", "(b) off-axis norm", r"$\|x_\perp\| = \|x_r - \alpha v^*\|$", True),
           ("cos_reg", "(c) alignment with v*", r"$\cos(x_r, v^*)$", False),
           (None, "(d) rotation rate", "angle change to v* (deg / block)", False)]
for pi, (fld, ptitle, lab, logy) in enumerate(_panels):
    ax = axes[pi]
    ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band" if pi == 0 else None)
    ax.axvline(_L_GROW, color="0.3", ls=":", lw=0.6, label=f"channel-operator onset" if pi == 0 else None)
    ax.axvline(_L_REF, color="0.3", ls="-.", lw=0.6, label=f"token-operator onset" if pi == 0 else None)
    _fl = (max(1e-2, 1e-4 * float(np.nanmedian(_gt[:, :, :, _G[fld]]))) if (logy and fld is not None) else None)
    for ci, cn in enumerate(COND_NAMES):
        if fld is not None:
            v = _gt[:, ci, :, _G[fld]]
        else:
            ang = np.degrees(np.arccos(np.clip(_gt[:, ci, :, _G["cos_reg"]], -1, 1)))   # [S, L]
            v = np.concatenate([np.full((_S, 1), np.nan), np.abs(np.diff(ang, axis=1))], axis=1)
        med = np.nanmedian(v, axis=0)
        (ax.semilogy if logy else ax.plot)(_x, np.maximum(med, _fl) if logy else med,
                                           color=COND_COLORS[ci], ls=COND_LS[ci], lw=1.0,
                                           label=COND_LABELS_SHORT[cn] if pi == 0 else None)
    if logy:
        ax.set_ylim(bottom=_fl)
    if fld == "cos_reg":
        bg = np.nanmedian(_gt[:, 0, :, _G["med_cos_all"]], axis=0)
        ax.plot(_x, bg, color="0.5", lw=0.8, ls="--", label="median over all image tokens (baseline)")
        ax.axhline(INTV.geom_cos_thr, color="0.55", ls=":", lw=0.6, label=f"alignment threshold {INTV.geom_cos_thr:g}")
        ax.set_ylim(-0.05, 1.05)
        ax.legend(loc="best", fontsize=LEGEND_PT - 1, frameon=True, framealpha=0.9, edgecolor="none")
    ax.set_title(ptitle, fontsize=_BASE_PT, pad=3)
    ax.set_xlabel("transformer block", fontsize=LEGEND_PT); ax.set_ylabel(lab, fontsize=LEGEND_PT + 0.5)
    ax.tick_params(labelsize=LEGEND_PT)
save_paper_fig(fig, int_fig_stem("I8b_geometry_trajectories"), legend_ax=axes[0], ncol=4,
    title=(f"Register geometry relative to the register direction v* under interventions "
           f"({MODEL_LABEL}, median over {_S} generations, step {T_STAR})"),
    caption=(
    f"Register geometry under the interventions in {MODEL_LABEL} (median over {_S} scenarios, step "
    f"{T_STAR}). The register token x_r is decomposed as alpha v* + x_perp with v* the register "
    f"direction fitted from the v4 run: (a) alpha, (b) the norm of x_perp, (c) the cosine with v* (grey "
    f"dashed: the median cosine of all image tokens in the baseline; dotted: the alignment threshold "
    f"{INTV.geom_cos_thr:g}), and (d) the absolute change of the angle to v* per block. Baseline "
    f"geometry is defined from the reference block {REG_REF_BLOCK} onward. Shaded: the high-norm band; "
    f"dotted and dash-dotted vertical lines: the channel and token intervention onsets."))

# ------------------------------------------------- register norm split over the tracked channels
_gr_names = [n for n in TRACKED_NAMES if n != "ctrl"]
_gr_idx = [TRACKED_NAMES.index(n) for n in _gr_names]
fig, _axs = panel_grid(_C, max_cols=4, panel_h=1.9)
for ci, (ax, cn) in enumerate(zip(_axs, COND_NAMES)):
    sc = CACHE["regsc_t"][:, ci, :, :]                       # [S, L, K+1]
    tot2 = np.maximum(sc[:, :, -1] ** 2, 1e-12)
    ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band" if ci == 0 else None)
    for gname, gi in zip(_gr_names, _gr_idx):
        frac = np.nanmedian(sc[:, :, gi] ** 2 / tot2, axis=0)
        ax.plot(_x, frac, lw=0.9, label=TRACKED_LABELS[gname] if ci == 0 else None,
                color={"ch1": OKABE_ITO[0], "ch2": OKABE_ITO[3], "ch3": OKABE_ITO[4]}[gname])
    a2 = np.nanmedian(_gt[:, ci, :, _G["alpha_reg"]] ** 2 / np.maximum(_gt[:, ci, :, _G["norm_reg"]] ** 2, 1e-12), axis=0)
    ax.plot(_x, a2, lw=0.9, color="0.1", ls="--", label="along v*" if ci == 0 else None)
    ax.set_title(wrap_label(COND_LABELS_SHORT[cn], 24), fontsize=LEGEND_PT, pad=2); ax.set_ylim(0, 1.02)
    ax.tick_params(labelsize=LEGEND_PT)
grid_labels(_axs, 4, "transformer block", "fraction of squared\nregister norm")
save_paper_fig(fig, int_fig_stem("I8b_register_channel_fractions"), legend_ax=_axs[0], ncol=5,
    title=(f"Where the register's squared norm sits under each intervention: dominant channel, "
           f"catching-up channel and the direction v* ({MODEL_LABEL}, median over {_S} generations, step {T_STAR})"),
    caption=(
    f"Where the register token's squared norm sits, per block and condition in {MODEL_LABEL} "
    f"(median over {_S} scenarios, step {T_STAR}): the fraction on the dominant channel, on the "
    f"catching-up channel" + (", on the second grower" if "ch3" in TRACKED_NAMES else "")
    + ", and along v* (dashed). A register that dissolves by rotation keeps its norm while the v* "
    f"fraction falls; a register that dissolves by homogenization keeps its v* fraction while its "
    f"strength relative to the median token falls."))

# ------------------------------------------------- attention endpoints at the probe blocks
_ph = CACHE["probe_head"]                                    # [S, C, P, Q, H, F]
def _agg(field, fn):
    return fn(_ph[..., _PF[field]], axis=-1)                 # [S, C, P, Q]
FRAC_TOP = _agg("reg_is_top_sink", np.nanmean)
FRAC_T10 = np.nanmean(_ph[..., _PF["reg_key_rank"]] <= 10, axis=-1)
MED_RANK = np.nanmedian(_ph[..., _PF["reg_key_rank"]], axis=-1)
MED_REL = np.nanmedian(_ph[..., _PF["reg_sink_rel"]], axis=-1)
MED_COSADV = np.nanmedian(_ph[..., _PF["reg_key_cos"]] - _ph[..., _PF["med_key_cos"]], axis=-1)

_att_rows = []
for pi, pb in enumerate(PROBE_BLOCKS_L):
    for qi, st in enumerate(PROBE_STEPS_RESOLVED):
        for nm, arr, marg in (("frac_heads_reg_top_sink", FRAC_TOP, 0.05),
                              ("frac_heads_reg_key_top10", FRAC_T10, 0.05),
                              ("median_reg_sink_rel", MED_REL, None),
                              ("median_reg_key_rank", MED_RANK, None),
                              ("median_key_cos_advantage", MED_COSADV, None)):
            for r in paired_table(arr[:, :, pi, qi], COND_NAMES, equiv_margin=marg):
                _att_rows.append({"probe_block": pb, "probe_role": PROBE_ROLES[pi],
                                  "probe_block_rel": round(pb / REL_SCALE, 3), "step": st, "metric": nm,
                                  "baseline_median": float(np.nanmedian(arr[:, 0, pi, qi])), **r})
_att = pd.DataFrame(_att_rows)
_att.to_csv(os.path.join(INT_PATHS["analysis"], "attention_endpoints.csv"), index=False)
print()
print("=" * 100)
print(f"ATTENTION ENDPOINTS at step {T_STAR} (paired median shift vs baseline; * = Holm-corrected sign p < 0.05)")
print("=" * 100)
for nm in ("frac_heads_reg_top_sink", "frac_heads_reg_key_top10", "median_reg_sink_rel"):
    sub = _att[(_att.metric == nm) & (_att.step == T_STAR)]
    print(f"  {nm}:")
    print("    baseline median by probe block: " + ", ".join(
        f"b{pb} ({PROBE_ROLES[i]}) {float(sub[sub.probe_block == pb].baseline_median.iloc[0]):.2f}"
        for i, pb in enumerate(PROBE_BLOCKS_L)))
    for cn in COND_NAMES[1:]:
        s2 = sub[sub.condition == cn].sort_values("probe_block")
        print(f"    {cn:20s} " + "  ".join(
            f"b{int(r.probe_block)} {r.median_delta:+.2f}{'*' if r.p_sign_holm < 0.05 else ' '}"
            for r in s2.itertuples()))

# sink persistence is the deepest probe block where the register is the top sink in >= 50% of heads
SINK_DEPTH = np.full((_S, _C), float(PROBE_BLOCKS_L[0]) - 1, np.float32)
for si in range(_S):
    for ci in range(_C):
        ok = np.where(FRAC_TOP[si, ci, :, _QI] >= 0.5)[0]
        if len(ok):
            SINK_DEPTH[si, ci] = PROBE_BLOCKS_L[ok[-1]]
_rows_sink = paired_table(SINK_DEPTH, COND_NAMES, censor_at=PROBE_BLOCKS_L[-1], rel_scale=REL_SCALE)
print_table(_rows_sink, f"SECONDARY: sink persistence (deepest probe block where the register is the top sink "
                        f"in >= 50 percent of heads, step {T_STAR}; probe granularity {PROBE_BLOCKS_L})")
pd.DataFrame(_rows_sink).to_csv(os.path.join(INT_PATHS["analysis"], "sink_persistence_endpoint.csv"), index=False)

fig, axes = plt.subplots(1, 3, figsize=(FIG_PAGE, 2.5))
_att_panels = ((axes[0], FRAC_TOP, "(a) register is the top sink", "fraction of heads", False),
               (axes[1], MED_RANK, "(b) register-key rank", "median rank among image keys\n(1 = most aligned with queries)", True),
               (axes[2], MED_REL, "(c) attention received by the register", "incoming attention (x uniform share)", True))
for ax, arr, ptitle, lab, logy in _att_panels:
    ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band" if ax is axes[0] else None)
    for ci, cn in enumerate(COND_NAMES):
        v = arr[:, ci, :, _QI]                                   # [S, P]
        med = np.nanmedian(v, axis=0); lo = np.nanpercentile(v, 25, axis=0); hi = np.nanpercentile(v, 75, axis=0)
        (ax.semilogy if logy else ax.plot)(PROBE_BLOCKS_L, np.maximum(med, 1e-3) if logy else med,
                                           marker="o", ms=2.5, color=COND_COLORS[ci], ls=COND_LS[ci],
                                           lw=1.0, label=COND_LABELS_SHORT[cn] if ax is axes[0] else None)
        ax.fill_between(PROBE_BLOCKS_L, np.maximum(lo, 1e-3) if logy else lo,
                        np.maximum(hi, 1e-3) if logy else hi, color=COND_COLORS[ci], alpha=0.10, lw=0)
    ax.set_title(ptitle, fontsize=_BASE_PT, pad=3)
    ax.set_xlabel("probe block (transformer block)", fontsize=LEGEND_PT); ax.set_ylabel(lab, fontsize=LEGEND_PT)
    ax.tick_params(labelsize=LEGEND_PT)
axes[0].set_ylim(-0.02, 1.02)
save_paper_fig(fig, int_fig_stem("I8b_attention_probes"), legend_ax=axes[0], ncol=4,
    title=(f"Attention role of the register token at the probe blocks under interventions "
           f"({MODEL_LABEL}, step {T_STAR}, median and interquartile range over {_S} generations)"),
    caption=(
    f"Attention role of the register token at the probe blocks {PROBE_BLOCKS_L} in {MODEL_LABEL} "
    f"(step {T_STAR}; median with interquartile band over {_S} scenarios). (a) Fraction of heads "
    f"whose strongest image-key sink is the register token (incoming attention renormalized over "
    f"image keys and averaged over image queries). (b) Median rank of the register key among "
    f"image keys by dot product with the mean image query. (c) The register's incoming "
    f"attention relative to the uniform share. The register token is the baseline identity fixed "
    f"at block {REG_REF_BLOCK}."))

# ------------------------------------------------- v* stability across depth, from baseline register vectors
_blocks_fit = list(range(REG_REF_BLOCK, LAST_BLOCK + 1))
_vb_cos, _vb_rows, _vb_energy = [], [], []
_pool_rows = []
for si, (sid_base, pid, seed, _t) in enumerate(tqdm(CLEAN_INTV, desc="v* refit")):
    with np.load(scenario_paths(INT_PATHS["runs"], f"baseline__{sid_base}", False)["npz"], allow_pickle=False) as z:
        if "reg_vec" not in z.files:
            break
        rv = z["reg_vec"][_blocks_fit, T_STAR, :].astype(np.float32)           # [Lf, D]
        R = z["tn_all"][_blocks_fit, T_STAR, :].astype(np.float32)               # [Lf, N]
        _reg_t = int(z["reg_idx"][T_STAR])
        strength = R[:, _reg_t] / np.maximum(np.median(R, axis=1), 1e-9)         # [Lf]
    _pool_rows.append((rv, strength))
if _pool_rows:
    V_BLOCK = np.full((len(_blocks_fit), CFG.d_model), np.nan, np.float32)
    for bi, b in enumerate(_blocks_fit):
        _sel = [rv[bi] for rv, s in _pool_rows if s[bi] >= INTV.reg_tau_primary]
        rows = np.stack(_sel) if _sel else np.zeros((0, CFG.d_model), np.float32)
        if len(rows) >= 5:
            vb, info = fit_vstar(rows, ch1=INT_TRACKED[0])
            V_BLOCK[bi] = vb
            _vb_cos.append(abs(float(vb @ VSTAR))); _vb_rows.append(len(rows)); _vb_energy.append(info["energy_on_ch1"])
        else:
            _vb_cos.append(np.nan); _vb_rows.append(len(rows)); _vb_energy.append(np.nan)
    zone_rows = np.concatenate([rv[:(_HI_B - REG_REF_BLOCK + 1)] for rv, s in _pool_rows], axis=0)
    zone_mask = np.concatenate([s[:(_HI_B - REG_REF_BLOCK + 1)] >= INTV.reg_tau_primary for rv, s in _pool_rows])
    if int(zone_mask.sum()) >= 2:
        V_ZONE, _zinfo = fit_vstar(zone_rows[zone_mask], ch1=INT_TRACKED[0])
        _zone_cos = abs(float(V_ZONE @ VSTAR))
    else:
        V_ZONE, _zinfo, _zone_cos = None, {"energy_on_ch1": float("nan")}, float("nan")
        print("[vstar-check][WARN] fewer than 2 strong register vectors in the zone; zone refit skipped")
    print(f"\n[vstar-check] zone-pooled refit from {int(zone_mask.sum())} baseline register vectors "
          f"(blocks {REG_REF_BLOCK}..{_HI_B}, step {T_STAR}): |cos| with the frozen v* = {_zone_cos:.5f}; "
          f"energy on ch1 {_zinfo['energy_on_ch1']:.4f}")
    print("[vstar-check] per-block |cos(v*_b, v*)|: " + ", ".join(
        f"b{b} {c:.3f}(n={n})" for b, c, n in zip(_blocks_fit, _vb_cos, _vb_rows) if np.isfinite(c)))
    save_json_atomic({"zone_cos": _zone_cos, "zone_info": _zinfo, "blocks": _blocks_fit,
                      "per_block_abs_cos": [None if not np.isfinite(c) else float(c) for c in _vb_cos],
                      "per_block_rows": _vb_rows,
                      "per_block_energy_ch1": [None if not np.isfinite(e) else float(e) for e in _vb_energy]},
                     os.path.join(INT_PATHS["analysis"], "vstar_stability.json"))
    fig, ax = plt.subplots(figsize=(FIG_COL, 2.3))
    ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band")
    ax.plot(_blocks_fit, _vb_cos, marker="o", ms=2.5, lw=1.0, color=OKABE_ITO[0], label="|cos| of per-block refit with frozen v*")
    ax.plot(_blocks_fit, _vb_energy, marker="s", ms=2.0, lw=0.8, color=OKABE_ITO[3], label=f"energy of refit on dominant ch. {INT_TRACKED[0]}")
    ax.set_ylim(0, 1.02); ax.set_xlabel("transformer block"); ax.set_ylabel("cosine / energy share")
    save_paper_fig(fig, int_fig_stem("I8b_vstar_stability"), legend_ax=ax, ncol=1,
        title=(f"Does the natural register direction rotate with depth? Per-block refit of v* from the "
               f"{MODEL_LABEL} baseline vs. the frozen v* (step {T_STAR})"),
        caption=(
        f"Stability of the register direction across depth in {MODEL_LABEL}: for each block, v*_b is "
        f"refitted from the baseline register vectors of scenarios whose register strength at that "
        f"block is at least {INTV.reg_tau_primary:g} (step {T_STAR}); plotted are its absolute cosine "
        f"with the frozen v* (fitted at block {L_STAR} from the v4 run) and the energy of v*_b on the "
        f"dominant channel. A cosine that stays near one through the band means the direction-refresh "
        f"operator restores the register's natural direction; a systematic decline would mean the "
        f"natural direction itself rotates with depth, which the refresh would counteract."))
else:
    print("[vstar-check] reg_vec not stored; per-block v* stability skipped")
print("[geometry/attention] complete")


## I9. Mechanism views

The tracked-channel profiles confirm that each intervention did what it
should. The register norm decomposition and the homogenization ratios tell
apart two cases, a register that loses its own magnitude and a background
that grows until it catches up. The takeover table checks whether
suppressing a channel brings a replacement channel into the top 10.
Token-norm maps and image PSNR show the effect qualitatively and how
invasive each intervention is.


In [ ]:
# =====================================================================
# I9. Mechanism views
#
# Tracked channels, register norm decomposition, homogenization ratios,
# channel takeover, token-norm maps and distance of the image to baseline.
# =====================================================================
_x = np.arange(_L)
_XLAB = "transformer block"

# ---- M1 tracked-channel profiles under every condition, to verify the interventions
_panels = [n for n in ("ch2", "ch3", "ch1", "ctrl") if n in TRACKED_NAMES]
fig, axes = plt.subplots(1, len(_panels), figsize=(FIG_PAGE, 2.4), sharex=True)
for pi, tname in enumerate(_panels):
    ax = np.atleast_1d(axes)[pi]
    ki = TRACKED_NAMES.index(tname)
    ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band" if pi == 0 else None)
    ax.axvline(_L_GROW, color="0.3", ls=":", lw=0.7, label="channel-operator onset" if pi == 0 else None)
    _fl = log_floor(CACHE["tracked_ma_t"][:, :, ki, :].mean(axis=0))
    for ci, cn in enumerate(COND_NAMES):
        v = CACHE["tracked_ma_t"][:, ci, ki, :].mean(axis=0)
        ax.semilogy(_x, np.maximum(v, _fl), color=COND_COLORS[ci], ls=COND_LS[ci], lw=1.0,
                    label=COND_LABELS_SHORT[cn] if pi == 0 else None)
    ax.set_ylim(bottom=_fl)
    ax.set_title(f"({'abcd'[pi]}) {TRACKED_LABELS[tname]}", fontsize=_BASE_PT, pad=3)
    ax.set_xlabel(_XLAB, fontsize=LEGEND_PT); ax.tick_params(labelsize=LEGEND_PT)
np.atleast_1d(axes)[0].set_ylabel("mean |activation| over image tokens")
save_paper_fig(fig, int_fig_stem("I9_tracked_channels"), legend_ax=np.atleast_1d(axes)[0], ncol=4,
    title=(f"Intervention check: magnitude of the tracked channels across depth under each condition "
           f"({MODEL_LABEL}, mean over {_S} generations, step {T_STAR})"),
    caption=(
    f"Tracked-channel verification in {MODEL_LABEL}: per-block mean absolute activation (mean over "
    f"{_S} scenarios, step {T_STAR}) of each tracked channel under every condition. Zeroed channels "
    f"sit on the floor of the log axis (exact zeros) inside the intervened range (blocks {_L_GROW} to {_L_END}); "
    f"clamped channels stop growing; growth-scaled channels grow faster than baseline; token "
    f"operators leave the channel population unchanged except through downstream effects."))

# ---- M2 register norm decomposition, per scenario and then averaged
_gr_names = [n for n in TRACKED_NAMES if n != "ctrl"]
_gr_idx = [TRACKED_NAMES.index(n) for n in _gr_names]
_gr_colors = {"ch1": OKABE_ITO[0], "ch2": OKABE_ITO[3], "ch3": OKABE_ITO[4]}
_fl = log_floor(np.concatenate([CACHE["med_t"].ravel(), np.abs(CACHE["regsc_t"]).ravel()]))
fig, _axs = panel_grid(_C, max_cols=4, panel_h=1.9)
for ci, (ax, cn) in enumerate(zip(_axs, COND_NAMES)):
    scs = CACHE["regsc_t"][:, ci, :, :]                      # [S, L, K+1]
    tot_s = scs[:, :, -1]
    sq = np.zeros_like(tot_s)
    for gi in _gr_idx:
        sq = sq + scs[:, :, gi] ** 2
    resid_s = np.sqrt(np.maximum(tot_s ** 2 - sq, 0.0))
    tot = np.nanmean(np.abs(tot_s), axis=0)
    resid = np.nanmean(resid_s, axis=0)
    med = CACHE["med_t"][:, ci, :].mean(axis=0)
    first = ci == 0
    ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band" if first else None)
    ax.semilogy(_x, np.maximum(tot, _fl), color="0.1", lw=1.1, label="register-token norm" if first else None)
    for gname, gi in zip(_gr_names, _gr_idx):
        v = np.nanmean(np.abs(scs[:, :, gi]), axis=0)
        ax.semilogy(_x, np.maximum(v, _fl), color=_gr_colors[gname], lw=0.9,
                    label=f"|value| on {TRACKED_LABELS[gname]}" if first else None)
    ax.semilogy(_x, np.maximum(resid, _fl), color=OKABE_ITO[2], lw=0.9, label="norm on all other channels" if first else None)
    ax.semilogy(_x, np.maximum(med, _fl), color="0.6", lw=0.9, ls="--", label="median token norm" if first else None)
    ax.set_ylim(bottom=_fl)
    ax.set_title(wrap_label(COND_LABELS_SHORT[cn], 24), fontsize=LEGEND_PT, pad=2)
    ax.tick_params(labelsize=LEGEND_PT)
grid_labels(_axs, 4, _XLAB, "L2 norm or |value|")
save_paper_fig(fig, int_fig_stem("I9_register_decomposition"), legend_ax=_axs[0], ncol=3,
    title=(f"Register-token norm decomposition per condition: how much of the register lives on the tracked "
           f"channels vs. the rest, next to the median token ({MODEL_LABEL}, mean over {_S} generations, step {T_STAR})"),
    caption=(
    f"Register-token decomposition in {MODEL_LABEL} (mean over {_S} scenarios, step {T_STAR}): the "
    f"register token's L2 norm, its absolute value on each tracked growing channel, the residual norm "
    f"over all remaining channels (per scenario, then averaged), and the median token norm, per block "
    f"and condition, from the reference block {REG_REF_BLOCK} onward. If baseline dissolution is norm "
    f"homogenization, the median curve rises to meet the register curve where the growing channels "
    f"catch up, and their suppression keeps the median curve low instead of raising the register curve."))

# ---- M3 background homogenization ratios
fig, axes = plt.subplots(1, 2, figsize=(FIG_PAGE, 2.5), sharex=True)
for pi, (ax, key, ptitle, lab) in enumerate(((axes[0], "med_t", "(a) background: median token norm", "median token norm, condition / baseline"),
                                              (axes[1], "regn_t", "(b) register-token norm", "register norm, condition / baseline"))):
    ax.axvspan(_LO_B, _HI_B, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band" if pi == 0 else None)
    base = np.maximum(CACHE[key][:, 0, :], 1e-9)
    for ci in range(1, _C):
        ratio = (CACHE[key][:, ci, :] / base).mean(axis=0)
        ax.plot(_x, ratio, color=COND_COLORS[ci], ls=COND_LS[ci], lw=1.0, label=COND_LABELS_SHORT[COND_NAMES[ci]] if pi == 0 else None)
    ax.axhline(1.0, color="0.6", lw=0.7)
    ax.set_title(ptitle, fontsize=_BASE_PT, pad=3)
    ax.set_xlabel(_XLAB, fontsize=LEGEND_PT); ax.set_ylabel(lab, fontsize=LEGEND_PT)
    ax.tick_params(labelsize=LEGEND_PT)
save_paper_fig(fig, int_fig_stem("I9_homogenization"), legend_ax=axes[0], ncol=4,
    title=(f"Background vs. register: does an intervention keep the register large or keep the other tokens "
           f"small? Per-block ratio to the baseline ({MODEL_LABEL}, mean over {_S} generations, step {T_STAR})"),
    caption=(
    f"Background versus register effects in {MODEL_LABEL} (mean over {_S} scenarios, step {T_STAR}): "
    f"per-block ratio of each condition to its baseline for the median token norm (a) and the "
    f"register token norm (b). A condition that extends register persistence by keeping the "
    f"background low shows a ratio below one in (a) and near one in (b)."))

# ---- M4 takeover table, which channels enter the top 10 at the probe blocks
_chan_conds = [c["name"] for c in CONDITIONS if c["kind"] == "channel"]
_take_rows = []
for cn in _chan_conds:
    ci = COND_NAMES.index(cn)
    for pi, pb in enumerate(PROBE_BLOCKS_L):
        cnt = {}
        for si in range(_S):
            base10 = set(CACHE["top10_probe"][si, 0, pi].tolist())
            for ch in CACHE["top10_probe"][si, ci, pi].tolist():
                if ch not in base10:
                    cnt[ch] = cnt.get(ch, 0) + 1
        for ch, c in sorted(cnt.items(), key=lambda kv: -kv[1]):
            _take_rows.append({"condition": cn, "block": int(pb), "channel": int(ch),
                               "n_scenarios": int(c), "frac": round(c / _S, 3)})
_take = pd.DataFrame(_take_rows)
_take.to_csv(os.path.join(INT_PATHS["analysis"], "takeover_channels.csv"), index=False)
print("[mechanism] top-10 entrants vs baseline (takeover check), leading rows:")
if len(_take):
    print(_take.sort_values("n_scenarios", ascending=False).head(12).to_string(index=False))
else:
    print("    none: intervention top-10 sets match baseline everywhere")

# ---- M5 token-norm maps, a qualitative view of persistence
_map_sid = next((s for s, p, sd, _t in CLEAN_INTV if (p, sd) in DEEP_PAIRS), CLEAN_INTV[0][0])
_map_blocks = sorted({_L_GROW, _HI_B - 1, _HI_B, min(_HI_B + 2, LAST_BLOCK), min(_HI_B + 5, LAST_BLOCK), LAST_BLOCK})
_side = int(round(math.sqrt(N_IMG)))
fig, axes = plt.subplots(_C, len(_map_blocks), figsize=(FIG_PAGE, 1.05 * _C), sharex=True, sharey=True)
_v0 = None
for ci, cn in enumerate(COND_NAMES):
    with np.load(scenario_paths(INT_PATHS["runs"], f"{cn}__{_map_sid}", False)["npz"], allow_pickle=False) as z:
        tn = z["tn_all"][:, T_STAR, :].astype(np.float32)
    if _v0 is None:
        _v0 = (max(float(np.percentile(tn, 5)), 1e-3), max(float(tn.max()), 1.0))
    for bi, b in enumerate(_map_blocks):
        ax = axes[ci, bi]
        im = ax.imshow(tn[b].reshape(_side, _side), cmap="magma", norm=LogNorm(vmin=_v0[0], vmax=_v0[1]),
                       interpolation="nearest", rasterized=True)
        if ci == 0:
            ax.set_title(f"block {b}" + (" (after band)" if b > _HI_B else ""), fontsize=LEGEND_PT, pad=2)
        if bi == 0:
            ax.set_ylabel("\n".join(_textwrap.wrap(COND_LABELS_SHORT[cn], 16)), fontsize=LEGEND_PT - 1)
        ax.set_xticks([]); ax.set_yticks([])
cb = fig.colorbar(im, ax=axes.ravel().tolist(), pad=0.01, shrink=0.6)
cb.set_label("token L2 norm (log scale)", fontsize=LEGEND_PT); cb.ax.tick_params(labelsize=LEGEND_PT)
save_paper_fig(fig, int_fig_stem("I9_token_norm_maps"),
    title=(f"Token-norm maps across depth for one generation ({_map_sid}), one row per condition "
           f"({MODEL_LABEL}, step {T_STAR}); the register is the isolated bright token"),
    caption=(
    f"Token-norm maps for one scenario ({_map_sid}) in {MODEL_LABEL} at step {T_STAR}: the "
    f"{_side} by {_side} grid of image-token L2 norms at selected blocks (blocks beyond the band end "
    f"are marked), one row per condition, shared log color scale. The register appears as an "
    f"isolated bright token."))

# ---- M6 image distance to baseline (from the cache) and contact sheet
from PIL import Image
fig, ax = plt.subplots(figsize=(FIG_PAGE * 0.75, 2.8))
_rng = np.random.default_rng(1)
for ci, cn in enumerate(COND_NAMES[1:], start=1):
    v = CACHE["psnr"][:, ci]
    xj = ci + _rng.uniform(-0.16, 0.16, len(v))
    ax.scatter(xj, v, s=4, color=COND_COLORS[ci], alpha=0.45, linewidths=0)
    ax.hlines(np.nanmedian(v), ci - 0.28, ci + 0.28, color="0.1", lw=1.4, zorder=5)
ax.set_xticks(range(1, _C)); ax.set_xticklabels([wrap_label(COND_LABELS_SHORT[c], 18) for c in COND_NAMES[1:]], rotation=35, ha="right", fontsize=LEGEND_PT)
ax.set_ylabel("PSNR vs. baseline image (dB)\nhigher = less change")
save_paper_fig(fig, int_fig_stem("I9_image_psnr"),
    title=(f"How much each intervention changes the generated image: PSNR against the baseline image "
           f"of the same prompt and seed ({MODEL_LABEL}, {_S} generations)"),
    caption=(
    f"Distance of each condition's generated image from its baseline image (PSNR, higher = more "
    f"similar) over {_S} scenarios in {MODEL_LABEL}; black bars mark medians. This quantifies how "
    f"invasive each intervention is at the image level; the sham condition bounds the run-to-run "
    f"numerical noise."))
_sheet = [(s, p, sd) for s, p, sd, _t in CLEAN_INTV if (p, sd) in DEEP_PAIRS][:5]
if _sheet:
    fig, axes = plt.subplots(len(_sheet), _C, figsize=(FIG_PAGE, 1.5 * len(_sheet)))
    axes = np.atleast_2d(axes)
    for ri, (sid_base, pid, seed) in enumerate(_sheet):
        for ci, cn in enumerate(COND_NAMES):
            png = scenario_paths(INT_PATHS["runs"], f"{cn}__{sid_base}", False)["png"]
            axes[ri, ci].imshow(Image.open(png)); axes[ri, ci].set_xticks([]); axes[ri, ci].set_yticks([])
            if ri == 0:
                axes[ri, ci].set_title("\n".join(_textwrap.wrap(COND_LABELS_SHORT[cn], 18)), fontsize=LEGEND_PT - 1, pad=2)
            if ci == 0:
                axes[ri, ci].set_ylabel(sid_base, fontsize=LEGEND_PT - 1)
    save_paper_fig(fig, int_fig_stem("I9_image_contact_sheet"),
        title=(f"Generated images under every condition (columns) for the five pilot generations (rows), "
               f"{MODEL_LABEL}"),
        caption=(
        f"Generated images for the pilot scenario pairs of {MODEL_LABEL} under every condition "
        f"(columns), one scenario per row."))
print("[mechanism] complete")


## I10. Manifest

Writes the decision, a v\* summary, the condition definitions, the headline
numbers for the three depths (norm, alignment and sink), storage use and a
short interpretation guide to `analysis_intervention_v3/`.


In [ ]:
# =====================================================================
# I10. Run manifest
# =====================================================================
def _du_gib(path):
    total = 0
    for r, _d, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(r, f))
            except OSError:
                pass
    return total / 2**30

_manifest = {
    "experiment": "Q6 register dissolution: channel competition, growth dose, direction refresh (v3)",
    "model_id": CFG.model_id, "model_family": MODEL_FAMILY,
    "decision": {k: INT_DEC[k] for k in
                 ("ch1", "ch2", "ch3", "ch_ctrl", "l_grow", "l_catch", "l_end", "l_refresh",
                  "band", "l_star", "t_star", "reg_ref_block", "probe_blocks", "probe_steps",
                  "dec_sha") if k in INT_DEC},
    "decision_rules": {k: INT_DEC.get(k) for k in ("ch2_rule", "ch3_rule", "onset_rule", "refresh_rule")},
    "vstar": {k: VSTAR_INFO.get(k) for k in
              ("vstar_sha", "n_rows", "n_scenarios", "abs_cos_with_e_ch1", "energy_on_ch1",
               "energy_on_ch2", "top_singular_energy", "split_half_abs_cos", "source")},
    "tracked_channels": dict(zip(TRACKED_NAMES, [int(c) for c in INT_TRACKED])),
    "conditions": [{k: c.get(k) for k in ("name", "kind", "mode", "lo", "hi", "channels", "exclude", "gamma", "target")}
                   for c in CONDITIONS],
    "scenario_scope": INTV.scenario_scope,
    "n_scenarios_clean": len(CLEAN_INTV),
    "primary_endpoint": {
        "definition": (f"register persistence depth, tau = {INTV.reg_tau_primary:g} x median token "
                       f"norm, step {T_STAR}, register identity fixed from baseline at block {REG_REF_BLOCK}"),
        "equivalence_margin_blocks": EQUIV_MARGIN_BLOCKS,
        "baseline_median_depth": float(np.median(DEPTH[:, 0])),
        "per_condition": _rows},
    "secondary_endpoints": {
        "alignment_depth": {"definition": f"deepest block with cos(register, v*) >= {INTV.geom_cos_thr:g}, step {T_STAR}",
                            "baseline_median": float(np.median(ADEPTH[:, 0])), "per_condition": _rows_cos},
        "sink_persistence": {"definition": "deepest probe block where the register is the top sink in >= 50 percent of heads",
                             "per_condition": _rows_sink},
        "any_token_depth": {"per_condition": _rows_any},
        "attention_probe_csv": "attention_endpoints.csv"},
    "comparability": COMPARABILITY,
    "register_identity_agreement": IDENT_AGREE,
    "intv_hash": INTV_HASH, "config_hash": CONFIG_HASH,
    "precision_mode": PRECISION_MODE, "lib_versions": LIB_VERSIONS,
    "storage_gib": {"runs": round(_du_gib(INT_PATHS["runs"]), 2),
                    "figures": round(_du_gib(INT_PATHS["figs"]), 2),
                    "analysis": round(_du_gib(INT_PATHS["analysis"]), 2)},
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds")}
save_json_atomic(_manifest, os.path.join(INT_PATHS["analysis"], "intervention_manifest.json"))
print("=" * 70)
print("INTERVENTION MANIFEST / HANDOFF (v3)")
print("=" * 70)
print(f"  model {MODEL_LABEL} | ch1 {INT_DEC['ch1']} ch2 {INT_DEC['ch2']} "
      + (f"ch3 {INT_DEC['ch3']} " if "ch3" in INT_DEC else "")
      + f"ctrl {INT_DEC['ch_ctrl']} | channel ops {INT_DEC['l_grow']}..{INT_DEC['l_end']} | "
      f"token ops {INT_DEC['l_refresh']}..{INT_DEC['l_end']} | v* {VSTAR_SHA}")
print(f"  clean scenarios: {len(CLEAN_INTV)} | conditions: {COND_NAMES}")
print(f"  baseline median persistence depth: {np.median(DEPTH[:, 0]):.0f} | alignment depth "
      f"{np.median(ADEPTH[:, 0]):.0f} (band end {int(CFG.band[1])}, last block {LAST_BLOCK})")
print(f"  {'condition':20s} {'norm depth':>12s} {'align depth':>12s} {'sink depth':>12s}   (median shift in blocks, Holm sign p; rel = / {REL_SCALE:g})")
for r1, r2, r3 in zip(_rows, _rows_cos, _rows_sink):
    print(f"  {r1['condition']:20s} {r1['median_delta']:+6.1f} ({r1['p_sign_holm']:.0e}) "
          f"{r2['median_delta']:+6.1f} ({r2['p_sign_holm']:.0e}) "
          f"{r3['median_delta']:+6.1f} ({r3['p_sign_holm']:.0e})   rel {r1['median_delta_rel']:+.3f} / "
          f"{r2['median_delta_rel']:+.3f} / {r3['median_delta_rel']:+.3f}   identity agreement {IDENT_AGREE[r1['condition']]:.2f}")
print(f"  storage: " + ", ".join(f"{k} {v} GiB" for k, v in _manifest["storage_gib"].items()))
print(f"  -> {os.path.join(INT_PATHS['analysis'], 'intervention_manifest.json')}")
print("=" * 70)
print("Interpretation guide (pre-registered): channel suppression prolongs all three depths and "
      "growth shortens them, controls equivalent -> channel competition; dir_refresh prolongs but "
      "channel suppression does not -> geometric rotation not tied to one channel; neither -> "
      "broader re-encoding; norm_refresh but not dir_refresh -> magnitude maintenance.")
